## Batch 0: Inventory bootstrap and source contract

This batch loads the project file inventory from `outputs/inventory/project_file_inventory.csv`.

The inventory is the source of truth for:
- source CSV paths
- source CSV columns
- source CSV row counts
- image/restoration asset paths
- basic file diagnostics

No OpenCV/LaMa source table is loaded in this batch. Batch 0 only validates that the required source artifacts are present and schema-compatible.

In [1]:
from __future__ import annotations

import ast
import json
import math
import re
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 160)
pd.set_option("display.max_colwidth", 180)
pd.set_option("display.width", 240)

In [2]:
def resolve_project_root() -> Path:
    """
    Resolve the thesis project root.

    This notebook may be opened from:
    - .../painting_restoration_eval
    - .../painting_restoration_eval/notebooks

    The project root is the directory containing:
    outputs/inventory/project_file_inventory.csv
    """
    cwd = Path.cwd().resolve()
    inventory_rel = Path("outputs/inventory/project_file_inventory.csv")

    for candidate in [cwd, *cwd.parents]:
        if (candidate / inventory_rel).is_file():
            return candidate

    if cwd.name == "notebooks":
        return cwd.parent

    return cwd


PROJECT_ROOT = resolve_project_root()
NOTEBOOK_ID = "20_opencv_lama_comparison"
NOTEBOOK_SLUG = "20_opencv_lama_comparison_inventory_rebuild_v2"

INVENTORY_CSV_PATH = PROJECT_ROOT / "outputs" / "inventory" / "project_file_inventory.csv"

OUTPUT_ROOT = PROJECT_ROOT / "outputs" / NOTEBOOK_SLUG
BATCH0_OUTPUT_DIR = OUTPUT_ROOT / "batch0_inventory"
BATCH1_OUTPUT_DIR = OUTPUT_ROOT / "batch1_sources"

for directory in [OUTPUT_ROOT, BATCH0_OUTPUT_DIR, BATCH1_OUTPUT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

STAGE_MANIFEST_JSON_PATH = OUTPUT_ROOT / "stage_manifest.json"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("INVENTORY_CSV_PATH:", INVENTORY_CSV_PATH)
print("OUTPUT_ROOT:", OUTPUT_ROOT)

PROJECT_ROOT: D:\Masters\FH\Thesis\painting-restoration-eval
INVENTORY_CSV_PATH: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\inventory\project_file_inventory.csv
OUTPUT_ROOT: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\20_opencv_lama_comparison_inventory_rebuild_v2


In [3]:
def project_relative_path(path: str | Path) -> str:
    path = Path(path)
    try:
        return path.resolve().relative_to(PROJECT_ROOT.resolve()).as_posix()
    except ValueError:
        return path.as_posix()


def path_size(path: str | Path) -> int:
    path = Path(path)
    return path.stat().st_size if path.is_file() else 0


def json_for_csv(value):
    if isinstance(value, (dict, list, tuple, set)):
        return json.dumps(value, ensure_ascii=False, sort_keys=True)
    if pd.isna(value) if not isinstance(value, (list, dict, tuple, set)) else False:
        return ""
    return value


def save_json(path: str | Path, payload: dict) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False, sort_keys=True), encoding="utf-8")


def read_json(path: str | Path, default: dict | None = None) -> dict:
    path = Path(path)
    if not path.is_file():
        return {} if default is None else default
    return json.loads(path.read_text(encoding="utf-8"))


def build_check(check_name, observed, expected, passed, failure_message):
    return {
        "check_name": check_name,
        "observed": observed,
        "expected": expected,
        "passed": bool(passed),
        "failure_message": "" if passed else failure_message,
    }


def batch0_is_blank(value) -> bool:
    if value is None:
        return True
    if isinstance(value, float) and math.isnan(value):
        return True
    text = str(value).strip()
    return text == "" or text.lower() in {"nan", "none", "null", "<na>"}

In [4]:
def normalize_relative_path(value) -> str:
    text = "" if batch0_is_blank(value) else str(value).strip()
    text = text.replace("\\", "/")
    text = re.sub(r"^\./+", "", text)
    text = text.lstrip("/")
    return text


def parse_inventory_columns(value) -> list[str]:
    if isinstance(value, list):
        return [str(item).strip() for item in value if str(item).strip()]

    if batch0_is_blank(value):
        return []

    text = str(value).strip()

    for parser in [json.loads, ast.literal_eval]:
        try:
            parsed = parser(text)
            if isinstance(parsed, list):
                return [str(item).strip() for item in parsed if str(item).strip()]
        except Exception:
            pass

    if "\n" in text:
        parts = text.splitlines()
    elif "|" in text and "," not in text:
        parts = text.split("|")
    else:
        parts = text.split(",")

    return [part.strip().strip("'\"") for part in parts if part.strip().strip("'\"")]


def inventory_csv_error_is_clean(value) -> bool:
    return batch0_is_blank(value)


def inventory_file_kind_is_csv(value) -> bool:
    text = "" if batch0_is_blank(value) else str(value).strip().lower()
    return "csv" in text

In [5]:
BATCH0_INVENTORY_SNAPSHOT_OUTPUT_PATH = BATCH0_OUTPUT_DIR / "batch0_inventory_snapshot.csv"
BATCH0_SOURCE_PLAN_OUTPUT_PATH = BATCH0_OUTPUT_DIR / "batch0_inventory_source_plan.csv"
BATCH0_VALIDATION_OUTPUT_PATH = BATCH0_OUTPUT_DIR / "batch0_validation.csv"

REQUIRED_INVENTORY_COLUMNS = [
    "relative_path",
    "file_kind",
    "csv_row_count",
    "csv_column_count",
    "csv_columns",
    "csv_error",
]

In [6]:
if not INVENTORY_CSV_PATH.is_file():
    raise FileNotFoundError(
        f"Inventory CSV not found at {project_relative_path(INVENTORY_CSV_PATH)}. "
        "Expected project-local inventory at outputs/inventory/project_file_inventory.csv."
    )

inventory_df = pd.read_csv(INVENTORY_CSV_PATH, low_memory=False)

missing_inventory_columns = [
    column for column in REQUIRED_INVENTORY_COLUMNS
    if column not in inventory_df.columns
]

if missing_inventory_columns:
    raise RuntimeError(
        "Inventory CSV is missing required columns: "
        + ", ".join(missing_inventory_columns)
    )

inventory_df = inventory_df.copy()
inventory_df["relative_path"] = inventory_df["relative_path"].map(normalize_relative_path)
inventory_df["absolute_path"] = inventory_df["relative_path"].map(lambda value: str(PROJECT_ROOT / value))
inventory_df["exists_on_disk_now"] = inventory_df["relative_path"].map(lambda value: (PROJECT_ROOT / value).is_file())
inventory_df["csv_columns_list"] = inventory_df["csv_columns"].map(parse_inventory_columns)
inventory_df["csv_columns_parsed_count"] = inventory_df["csv_columns_list"].map(len)

print("Loaded inventory:", project_relative_path(INVENTORY_CSV_PATH))
print("Inventory rows:", len(inventory_df))
print("Inventory columns:", len(inventory_df.columns))

display(inventory_df.head())
display(inventory_df["file_kind"].value_counts(dropna=False).reset_index(name="rows").head(30))

Loaded inventory: outputs/inventory/project_file_inventory.csv
Inventory rows: 3047
Inventory columns: 21


,relative_path,file_name,parent_dir,extension,file_kind,size_bytes,last_modified_iso,depth,csv_row_count,csv_column_count,csv_columns,csv_error,image_width,image_height,image_mode,image_error,sha256_first_1mb,absolute_path,exists_on_disk_now,csv_columns_list,csv_columns_parsed_count
0,.gitattributes,.gitattributes,NaN,NaN,other,60,2026-07-06T17:52:52.879628+00:00,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,439ad27a2f48aacaaf48efa19bc2f8b8fcb5b402d23454dbf8904f2b06c6d3c1,D:\Masters\FH\Thesis\painting-restoration-eval\.gitattributes,True,[],0
1,.gitignore,.gitignore,NaN,NaN,other,643,2026-07-06T17:53:38.894066+00:00,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,d4ef631ec54b12485b0f98e910cc14d435433c33f264896b7549f3417a009c71,D:\Masters\FH\Thesis\painting-restoration-eval\.gitignore,True,[],0
2,config/experiment_50_config.yaml,experiment_50_config.yaml,config,.yaml,other,7111,2026-07-23T12:34:35.693189+00:00,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7af22deb75434907302e3e460b4c28a73f25533f3274e46b6a14cb7668fce65e,D:\Masters\FH\Thesis\painting-restoration-eval\config\experiment_50_config.yaml,True,[],0
3,config/pilot_config.yaml,pilot_config.yaml,config,.yaml,other,677,2026-06-29T14:26:59.322671+00:00,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9c81e4ee3aec9dc1b86aa6bc28f5ba442b694810d83eedef513bd207bcb5e4d4,D:\Masters\FH\Thesis\painting-restoration-eval\config\pilot_config.yaml,True,[],0
4,data/model_audit/model_candidates.csv,model_candidates.csv,data/model_audit,.csv,csv,5383,2026-07-03T09:14:19.349960+00:00,2,5.0,17.0,model_name | model_family | open_or_closed | deterministic_or_stochastic | training_data_summary | painting_or_art_representation | known_domain_gap | bias_risk | input_format ...,NaN,NaN,NaN,NaN,NaN,4ec7ee8d89e5a9d122ea5e9dd37efa6346a6f686263d6636261243a6b047e8b8,D:\Masters\FH\Thesis\painting-restoration-eval\data\model_audit\model_candidates.csv,True,"[model_name, model_family, open_or_closed, deterministic_or_stochastic, training_data_summary, painting_or_art_representation, known_domain_gap, bias_risk, input_format, mask_f...",17


,file_kind,rows
0,mask_image,1174
1,restored_image,1152
2,damaged_image,376
3,image,56
4,clean_image,50
5,metric_csv,50
6,notebook,40
7,csv,32
8,other,25
9,validation_csv,25


In [7]:
SOURCE_SPECS = [
    {
        "source_id": "opencv_restoration_manifest",
        "method": "opencv_telea",
        "source_family": "restoration_manifest",
        "relative_path": "data/processed/metadata/metadata_restored_opencv_telea.csv",
        "required_columns": [
            "case_id", "dataset_name", "painting_id", "mask_id", "mask_type",
            "restored_path", "runtime_seconds", "output_written", "status", "issue",
        ],
    },
    {
        "source_id": "lama_restoration_manifest",
        "method": "lama",
        "source_family": "restoration_manifest",
        "relative_path": "data/processed/metadata/metadata_restored_lama.csv",
        "required_columns": [
            "case_id", "dataset_name", "painting_id", "mask_id", "mask_type",
            "restored_path", "runtime_seconds", "output_written",
            "iopaint_returncode", "status", "issue",
        ],
    },
    {
        "source_id": "opencv_classical_metrics",
        "method": "opencv_telea",
        "source_family": "classical_metrics",
        "relative_path": "data/processed/metrics/metrics_opencv_telea_classical.csv",
        "required_columns": [
            "case_id", "dataset_name", "painting_id", "mask_id", "mask_type",
            "evaluation_region", "restored_mse", "mse_improvement",
            "restored_psnr", "psnr_improvement", "restored_ssim", "ssim_improvement",
            "status", "issue",
        ],
    },
    {
        "source_id": "lama_classical_metrics",
        "method": "lama",
        "source_family": "classical_metrics",
        "relative_path": "outputs/metrics/classical_metrics_lama.csv",
        "required_columns": [
            "case_id", "dataset_name", "painting_id", "mask_id", "mask_type",
            "evaluation_region", "restored_mse", "mse_improvement",
            "restored_psnr", "psnr_improvement", "restored_ssim", "ssim_improvement",
            "status", "issue",
        ],
    },
    {
        "source_id": "opencv_lpips_metrics",
        "method": "opencv_telea",
        "source_family": "lpips_metrics",
        "relative_path": "data/processed/metrics/metrics_opencv_telea_lpips.csv",
        "required_columns": [
            "case_id", "dataset_name", "painting_id", "mask_id", "mask_type",
            "evaluation_region", "restored_lpips", "lpips_improvement",
            "status", "issue",
        ],
    },
    {
        "source_id": "lama_lpips_metrics",
        "method": "lama",
        "source_family": "lpips_metrics",
        "relative_path": "outputs/metrics/lpips_metrics_lama.csv",
        "required_columns": [
            "case_id", "dataset_name", "painting_id", "mask_id", "mask_type",
            "evaluation_region", "restored_lpips", "lpips_improvement",
            "status", "issue",
        ],
    },
    {
        "source_id": "opencv_feature_metrics",
        "method": "opencv_telea",
        "source_family": "feature_metrics",
        "relative_path": "data/processed/metrics/metrics_opencv_telea_feature_similarity.csv",
        "required_columns": [
            "case_id", "dataset_name", "painting_id", "mask_id", "mask_type",
            "evaluation_region", "clip_restored_similarity", "clip_similarity_improvement",
            "dinov2_restored_similarity", "dinov2_similarity_improvement",
            "status", "issue",
        ],
    },
    {
        "source_id": "lama_feature_metrics",
        "method": "lama",
        "source_family": "feature_metrics",
        "relative_path": "outputs/metrics/feature_similarity_metrics_lama.csv",
        "required_columns": [
            "case_id", "dataset_name", "painting_id", "mask_id", "mask_type",
            "evaluation_region", "clip_restored_similarity", "clip_similarity_improvement",
            "dinov2_restored_similarity", "dinov2_similarity_improvement",
            "status", "issue",
        ],
    },
]

In [8]:
source_plan_rows = []

for spec in SOURCE_SPECS:
    expected_relative_path = normalize_relative_path(spec["relative_path"])
    matching_inventory_df = inventory_df[inventory_df["relative_path"].eq(expected_relative_path)].copy()

    inventory_row_found = len(matching_inventory_df) == 1

    if inventory_row_found:
        row = matching_inventory_df.iloc[0]
        inventory_columns = row["csv_columns_list"]
        missing_required_columns = [
            column for column in spec["required_columns"]
            if column not in inventory_columns
        ]
        source_path = PROJECT_ROOT / expected_relative_path
        file_kind = row.get("file_kind", "")
        csv_error = row.get("csv_error", "")
        csv_row_count = row.get("csv_row_count", np.nan)
        csv_column_count = row.get("csv_column_count", np.nan)
        sha256_first_1mb = row.get("sha256_first_1mb", "")
    else:
        inventory_columns = []
        missing_required_columns = list(spec["required_columns"])
        source_path = PROJECT_ROOT / expected_relative_path
        file_kind = ""
        csv_error = ""
        csv_row_count = np.nan
        csv_column_count = np.nan
        sha256_first_1mb = ""

    source_plan_rows.append(
        {
            "source_id": spec["source_id"],
            "method": spec["method"],
            "source_family": spec["source_family"],
            "relative_path": expected_relative_path,
            "absolute_path": str(source_path),
            "inventory_row_found": inventory_row_found,
            "exists_on_disk_now": source_path.is_file(),
            "file_kind": file_kind,
            "file_kind_is_csv": inventory_file_kind_is_csv(file_kind),
            "csv_error": csv_error,
            "csv_error_clean": inventory_csv_error_is_clean(csv_error),
            "csv_row_count_inventory": csv_row_count,
            "csv_column_count_inventory": csv_column_count,
            "csv_columns_parsed_count": len(inventory_columns),
            "required_columns": spec["required_columns"],
            "missing_required_columns": missing_required_columns,
            "required_columns_present": len(missing_required_columns) == 0,
            "sha256_first_1mb": sha256_first_1mb,
        }
    )

batch0_source_plan_df = pd.DataFrame(source_plan_rows)

display(batch0_source_plan_df)

,source_id,method,source_family,relative_path,absolute_path,inventory_row_found,exists_on_disk_now,file_kind,file_kind_is_csv,csv_error,csv_error_clean,csv_row_count_inventory,csv_column_count_inventory,csv_columns_parsed_count,required_columns,missing_required_columns,required_columns_present,sha256_first_1mb
0,opencv_restoration_manifest,opencv_telea,restoration_manifest,data/processed/metadata/metadata_restored_opencv_telea.csv,D:\Masters\FH\Thesis\painting-restoration-eval\data\processed\metadata\metadata_restored_opencv_telea.csv,True,True,csv,True,NaN,True,410.0,194.0,194,"[case_id, dataset_name, painting_id, mask_id, mask_type, restored_path, runtime_seconds, output_written, status, issue]",[],True,c76e783cb0b1b6deb7de72b05de070c8ac8ea72d1d2e30522d774096d8beb714
1,lama_restoration_manifest,lama,restoration_manifest,data/processed/metadata/metadata_restored_lama.csv,D:\Masters\FH\Thesis\painting-restoration-eval\data\processed\metadata\metadata_restored_lama.csv,True,True,csv,True,NaN,True,410.0,218.0,218,"[case_id, dataset_name, painting_id, mask_id, mask_type, restored_path, runtime_seconds, output_written, iopaint_returncode, status, issue]",[],True,58b52d61924fa2a2c9d2f9c8d875c43da6a58d44fd6ca28ada2d7b73a403da1a
2,opencv_classical_metrics,opencv_telea,classical_metrics,data/processed/metrics/metrics_opencv_telea_classical.csv,D:\Masters\FH\Thesis\painting-restoration-eval\data\processed\metrics\metrics_opencv_telea_classical.csv,True,True,metric_csv,True,NaN,True,2295.0,48.0,48,"[case_id, dataset_name, painting_id, mask_id, mask_type, evaluation_region, restored_mse, mse_improvement, restored_psnr, psnr_improvement, restored_ssim, ssim_improvement, sta...",[],True,5676c2d3624798a9da0b4b712e0f7366bf3bbc4f7089eb775edf45bb2ecf5ab3
3,lama_classical_metrics,lama,classical_metrics,outputs/metrics/classical_metrics_lama.csv,D:\Masters\FH\Thesis\painting-restoration-eval\outputs\metrics\classical_metrics_lama.csv,True,True,metric_csv,True,NaN,True,2260.0,51.0,51,"[case_id, dataset_name, painting_id, mask_id, mask_type, evaluation_region, restored_mse, mse_improvement, restored_psnr, psnr_improvement, restored_ssim, ssim_improvement, sta...",[],True,79112d73767d3682d2e273844efde67004bfb3e1ddec3be621ad87a53a1cdbad
4,opencv_lpips_metrics,opencv_telea,lpips_metrics,data/processed/metrics/metrics_opencv_telea_lpips.csv,D:\Masters\FH\Thesis\painting-restoration-eval\data\processed\metrics\metrics_opencv_telea_lpips.csv,True,True,metric_csv,True,NaN,True,1175.0,47.0,47,"[case_id, dataset_name, painting_id, mask_id, mask_type, evaluation_region, restored_lpips, lpips_improvement, status, issue]",[],True,99f0d27dc96d950f193242e7d01229f236a50eb0d51fd46cc24542735b4f46df
5,lama_lpips_metrics,lama,lpips_metrics,outputs/metrics/lpips_metrics_lama.csv,D:\Masters\FH\Thesis\painting-restoration-eval\outputs\metrics\lpips_metrics_lama.csv,True,True,metric_csv,True,NaN,True,1175.0,47.0,47,"[case_id, dataset_name, painting_id, mask_id, mask_type, evaluation_region, restored_lpips, lpips_improvement, status, issue]",[],True,6fc713e7406806d9e3f3feeb9ceffd2091c83660848a5212c3e0da17a5efcf4c
6,opencv_feature_metrics,opencv_telea,feature_metrics,data/processed/metrics/metrics_opencv_telea_feature_similarity.csv,D:\Masters\FH\Thesis\painting-restoration-eval\data\processed\metrics\metrics_opencv_telea_feature_similarity.csv,True,True,metric_csv,True,NaN,True,1175.0,59.0,59,"[case_id, dataset_name, painting_id, mask_id, mask_type, evaluation_region, clip_restored_similarity, clip_similarity_improvement, dinov2_restored_similarity, dinov2_similarity...",[],True,c0b52ae8716747527308ced65ed3d01480a9b43397e509258253065b7b290aa3
7,lama_feature_metrics,lama,feature_metrics,outputs/metrics/feature_similarity_metrics_lama.csv,D:\Masters\FH\Thesis\painting-restoration-eval\outputs\metrics\feature_similarity_metrics_lama.csv,True,True,metric_csv,True,NaN,True,1175.0,59.0,59,"[case_id, dataset_name, painting_id, mask_id, mask_type, evaluation_region, clip_restore

In [9]:
batch0_check_rows = [
    build_check(
        "inventory_file_exists",
        project_relative_path(INVENTORY_CSV_PATH),
        "existing inventory CSV",
        INVENTORY_CSV_PATH.is_file(),
        "Project inventory CSV is missing.",
    ),
    build_check(
        "inventory_has_rows",
        len(inventory_df),
        "> 0",
        len(inventory_df) > 0,
        "Inventory CSV has no rows.",
    ),
    build_check(
        "required_inventory_columns_present",
        missing_inventory_columns,
        "no missing inventory columns",
        len(missing_inventory_columns) == 0,
        "Inventory CSV is missing required columns.",
    ),
    build_check(
        "all_required_source_rows_found_in_inventory",
        batch0_source_plan_df.loc[
            ~batch0_source_plan_df["inventory_row_found"], "relative_path"
        ].tolist(),
        "all required sources present in inventory",
        bool(batch0_source_plan_df["inventory_row_found"].all()),
        "One or more required source CSVs are missing from the inventory.",
    ),
    build_check(
        "all_required_sources_exist_on_disk",
        batch0_source_plan_df.loc[
            ~batch0_source_plan_df["exists_on_disk_now"], "relative_path"
        ].tolist(),
        "all required sources exist on disk",
        bool(batch0_source_plan_df["exists_on_disk_now"].all()),
        "One or more inventory-resolved source CSVs are not present on disk.",
    ),
    build_check(
        "all_required_sources_are_csv_like",
        batch0_source_plan_df.loc[
            ~batch0_source_plan_df["file_kind_is_csv"], ["source_id", "file_kind"]
        ].to_dict("records"),
        "all required sources have CSV-like file_kind",
        bool(batch0_source_plan_df["file_kind_is_csv"].all()),
        "One or more required sources are not marked as CSV-like in the inventory.",
    ),
    build_check(
        "all_required_sources_have_clean_csv_scan",
        batch0_source_plan_df.loc[
            ~batch0_source_plan_df["csv_error_clean"], ["source_id", "csv_error"]
        ].to_dict("records"),
        "all required sources have blank csv_error",
        bool(batch0_source_plan_df["csv_error_clean"].all()),
        "One or more required source CSVs had an inventory csv_error.",
    ),
    build_check(
        "all_required_source_columns_present_in_inventory",
        batch0_source_plan_df.loc[
            ~batch0_source_plan_df["required_columns_present"],
            ["source_id", "missing_required_columns"],
        ].to_dict("records"),
        "all required columns present in inventory csv_columns",
        bool(batch0_source_plan_df["required_columns_present"].all()),
        "One or more required source CSVs are missing required columns according to the inventory.",
    ),
]

batch0_validation_df = pd.DataFrame(batch0_check_rows)

display(batch0_validation_df)

if not batch0_validation_df["passed"].astype(bool).all():
    failed_batch0_checks_df = batch0_validation_df[~batch0_validation_df["passed"].astype(bool)].copy()
    display(failed_batch0_checks_df)
    raise RuntimeError("Notebook 20 Batch 0 validation failed. Inspect inventory source plan.")

,check_name,observed,expected,passed,failure_message
0,inventory_file_exists,outputs/inventory/project_file_inventory.csv,existing inventory CSV,True,
1,inventory_has_rows,3047,> 0,True,
2,required_inventory_columns_present,[],no missing inventory columns,True,
3,all_required_source_rows_found_in_inventory,[],all required sources present in inventory,True,
4,all_required_sources_exist_on_disk,[],all required sources exist on disk,True,
5,all_required_sources_are_csv_like,[],all required sources have CSV-like file_kind,True,
6,all_required_sources_have_clean_csv_scan,[],all required sources have blank csv_error,True,
7,all_required_source_columns_present_in_inventory,[],all required columns present in inventory csv_columns,True,


In [10]:
batch0_inventory_export_df = inventory_df.copy()
batch0_inventory_export_df["csv_columns_list"] = batch0_inventory_export_df["csv_columns_list"].map(
    lambda value: json.dumps(value, ensure_ascii=False)
)

batch0_source_plan_export_df = batch0_source_plan_df.copy()
for column in ["required_columns", "missing_required_columns"]:
    batch0_source_plan_export_df[column] = batch0_source_plan_export_df[column].map(
        lambda value: json.dumps(value, ensure_ascii=False)
    )

batch0_validation_export_df = batch0_validation_df.copy()
for column in ["observed", "expected"]:
    batch0_validation_export_df[column] = batch0_validation_export_df[column].map(json_for_csv)

batch0_inventory_export_df.to_csv(BATCH0_INVENTORY_SNAPSHOT_OUTPUT_PATH, index=False)
batch0_source_plan_export_df.to_csv(BATCH0_SOURCE_PLAN_OUTPUT_PATH, index=False)
batch0_validation_export_df.to_csv(BATCH0_VALIDATION_OUTPUT_PATH, index=False)

stage_manifest = {
    "notebook_id": NOTEBOOK_ID,
    "notebook_slug": NOTEBOOK_SLUG,
    "status": "batch_0_inventory_bootstrap_complete",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "updated_at_utc": datetime.now(timezone.utc).isoformat(),
    "project_root": str(PROJECT_ROOT),
    "inventory_source": project_relative_path(INVENTORY_CSV_PATH),
    "output_root": project_relative_path(OUTPUT_ROOT),
    "outputs": {
        "batch0_inventory_snapshot": project_relative_path(BATCH0_INVENTORY_SNAPSHOT_OUTPUT_PATH),
        "batch0_source_plan": project_relative_path(BATCH0_SOURCE_PLAN_OUTPUT_PATH),
        "batch0_validation": project_relative_path(BATCH0_VALIDATION_OUTPUT_PATH),
    },
    "row_counts": {
        "inventory_rows": int(len(inventory_df)),
        "batch0_required_sources": int(len(batch0_source_plan_df)),
        "batch0_validation_checks": int(len(batch0_validation_df)),
    },
    "source_ids": batch0_source_plan_df["source_id"].tolist(),
    "next_batch": "Batch 1: load inventory-resolved source CSV tables and validate actual schemas.",
}

save_json(STAGE_MANIFEST_JSON_PATH, stage_manifest)

print("Batch 0 complete.")
print("Saved:", project_relative_path(BATCH0_INVENTORY_SNAPSHOT_OUTPUT_PATH))
print("Saved:", project_relative_path(BATCH0_SOURCE_PLAN_OUTPUT_PATH))
print("Saved:", project_relative_path(BATCH0_VALIDATION_OUTPUT_PATH))
print("Updated manifest:", project_relative_path(STAGE_MANIFEST_JSON_PATH))

Batch 0 complete.
Saved: outputs/20_opencv_lama_comparison_inventory_rebuild_v2/batch0_inventory/batch0_inventory_snapshot.csv
Saved: outputs/20_opencv_lama_comparison_inventory_rebuild_v2/batch0_inventory/batch0_inventory_source_plan.csv
Saved: outputs/20_opencv_lama_comparison_inventory_rebuild_v2/batch0_inventory/batch0_validation.csv
Updated manifest: outputs/20_opencv_lama_comparison_inventory_rebuild_v2/stage_manifest.json


In [11]:
BATCH1_SOURCE_TABLE_SHAPES_OUTPUT_PATH = BATCH1_OUTPUT_DIR / "batch1_source_table_shapes.csv"
BATCH1_SCHEMA_VALIDATION_OUTPUT_PATH = BATCH1_OUTPUT_DIR / "batch1_source_schema_validation.csv"

In [12]:
SOURCE_TABLES: dict[str, pd.DataFrame] = {}

for row in batch0_source_plan_df.itertuples(index=False):
    source_id = row.source_id
    source_path = Path(row.absolute_path)

    print(f"Loading {source_id}: {project_relative_path(source_path)}")

    if not source_path.is_file():
        raise FileNotFoundError(
            f"Batch 1 source file missing for {source_id}: {project_relative_path(source_path)}"
        )

    SOURCE_TABLES[source_id] = pd.read_csv(source_path, low_memory=False)

print("Loaded source tables:", len(SOURCE_TABLES))

Loading opencv_restoration_manifest: data/processed/metadata/metadata_restored_opencv_telea.csv
Loading lama_restoration_manifest: data/processed/metadata/metadata_restored_lama.csv
Loading opencv_classical_metrics: data/processed/metrics/metrics_opencv_telea_classical.csv
Loading lama_classical_metrics: outputs/metrics/classical_metrics_lama.csv
Loading opencv_lpips_metrics: data/processed/metrics/metrics_opencv_telea_lpips.csv
Loading lama_lpips_metrics: outputs/metrics/lpips_metrics_lama.csv
Loading opencv_feature_metrics: data/processed/metrics/metrics_opencv_telea_feature_similarity.csv
Loading lama_feature_metrics: outputs/metrics/feature_similarity_metrics_lama.csv
Loaded source tables: 8


In [13]:
batch1_shape_rows = []

for _, source_row in batch0_source_plan_df.iterrows():
    source_id = source_row["source_id"]
    source_df = SOURCE_TABLES[source_id]

    actual_columns = list(source_df.columns)
    inventory_columns = parse_inventory_columns(
        inventory_df.loc[
            inventory_df["relative_path"].eq(source_row["relative_path"]),
            "csv_columns",
        ].iloc[0]
    )

    required_columns = source_row["required_columns"]
    if isinstance(required_columns, str):
        required_columns = parse_inventory_columns(required_columns)

    batch1_shape_rows.append(
        {
            "source_id": source_id,
            "method": source_row["method"],
            "source_family": source_row["source_family"],
            "relative_path": source_row["relative_path"],
            "loaded_rows": int(len(source_df)),
            "loaded_columns": int(len(source_df.columns)),
            "inventory_rows": source_row["csv_row_count_inventory"],
            "inventory_columns": source_row["csv_column_count_inventory"],
            "actual_columns": actual_columns,
            "inventory_declared_columns": inventory_columns,
            "required_columns": required_columns,
            "missing_required_columns_actual": [
                column for column in required_columns
                if column not in actual_columns
            ],
            "extra_actual_columns_not_in_inventory": [
                column for column in actual_columns
                if column not in inventory_columns
            ],
            "inventory_columns_not_in_actual": [
                column for column in inventory_columns
                if column not in actual_columns
            ],
        }
    )

batch1_source_shapes_df = pd.DataFrame(batch1_shape_rows)

display(
    batch1_source_shapes_df[
        [
            "source_id",
            "method",
            "source_family",
            "loaded_rows",
            "loaded_columns",
            "inventory_rows",
            "inventory_columns",
            "missing_required_columns_actual",
            "extra_actual_columns_not_in_inventory",
            "inventory_columns_not_in_actual",
        ]
    ]
)

,source_id,method,source_family,loaded_rows,loaded_columns,inventory_rows,inventory_columns,missing_required_columns_actual,extra_actual_columns_not_in_inventory,inventory_columns_not_in_actual
0,opencv_restoration_manifest,opencv_telea,restoration_manifest,410,194,410.0,194.0,[],[],[]
1,lama_restoration_manifest,lama,restoration_manifest,410,218,410.0,218.0,[],[],[]
2,opencv_classical_metrics,opencv_telea,classical_metrics,2295,48,2295.0,48.0,[],[],[]
3,lama_classical_metrics,lama,classical_metrics,2260,51,2260.0,51.0,[],[],[]
4,opencv_lpips_metrics,opencv_telea,lpips_metrics,1175,47,1175.0,47.0,[],[],[]
5,lama_lpips_metrics,lama,lpips_metrics,1175,47,1175.0,47.0,[],[],[]
6,opencv_feature_metrics,opencv_telea,feature_metrics,1175,59,1175.0,59.0,[],[],[]
7,lama_feature_metrics,lama,feature_metrics,1175,59,1175.0,59.0,[],[],[]


In [14]:
batch1_check_rows = []

for _, row in batch1_source_shapes_df.iterrows():
    source_id = row["source_id"]

    loaded_rows = int(row["loaded_rows"])
    loaded_columns = int(row["loaded_columns"])

    inventory_rows = pd.to_numeric(
        pd.Series([row["inventory_rows"]]), errors="coerce"
    ).iloc[0]
    inventory_columns = pd.to_numeric(
        pd.Series([row["inventory_columns"]]), errors="coerce"
    ).iloc[0]

    missing_required_columns_actual = row["missing_required_columns_actual"]
    inventory_columns_not_in_actual = row["inventory_columns_not_in_actual"]

    batch1_check_rows.extend(
        [
            build_check(
                f"{source_id}_loaded_non_empty",
                loaded_rows,
                "> 0",
                loaded_rows > 0,
                f"{source_id} loaded with zero rows.",
            ),
            build_check(
                f"{source_id}_loaded_column_count_positive",
                loaded_columns,
                "> 0",
                loaded_columns > 0,
                f"{source_id} loaded with zero columns.",
            ),
            build_check(
                f"{source_id}_row_count_matches_inventory",
                loaded_rows,
                int(inventory_rows) if pd.notna(inventory_rows) else "inventory row count available",
                pd.notna(inventory_rows) and loaded_rows == int(inventory_rows),
                f"{source_id} loaded row count does not match inventory csv_row_count.",
            ),
            build_check(
                f"{source_id}_column_count_matches_inventory",
                loaded_columns,
                int(inventory_columns) if pd.notna(inventory_columns) else "inventory column count available",
                pd.notna(inventory_columns) and loaded_columns == int(inventory_columns),
                f"{source_id} loaded column count does not match inventory csv_column_count.",
            ),
            build_check(
                f"{source_id}_required_columns_present",
                missing_required_columns_actual,
                "no missing required columns",
                len(missing_required_columns_actual) == 0,
                f"{source_id} is missing required columns in the actual loaded CSV.",
            ),
            build_check(
                f"{source_id}_actual_columns_match_inventory_columns",
                inventory_columns_not_in_actual,
                "all inventory-declared columns present in actual loaded CSV",
                len(inventory_columns_not_in_actual) == 0,
                f"{source_id} actual columns differ from inventory-declared csv_columns.",
            ),
        ]
    )

batch1_schema_validation_df = pd.DataFrame(batch1_check_rows)

display(batch1_schema_validation_df)

if not batch1_schema_validation_df["passed"].astype(bool).all():
    failed_batch1_checks_df = batch1_schema_validation_df[
        ~batch1_schema_validation_df["passed"].astype(bool)
    ].copy()
    display(failed_batch1_checks_df)
    raise RuntimeError("Notebook 20 Batch 1 validation failed. Inspect loaded source schemas.")

,check_name,observed,expected,passed,failure_message
0,opencv_restoration_manifest_loaded_non_empty,410,> 0,True,
1,opencv_restoration_manifest_loaded_column_count_positive,194,> 0,True,
2,opencv_restoration_manifest_row_count_matches_inventory,410,410,True,
3,opencv_restoration_manifest_column_count_matches_inventory,194,194,True,
4,opencv_restoration_manifest_required_columns_present,[],no missing required columns,True,
5,opencv_restoration_manifest_actual_columns_match_inventory_columns,[],all inventory-declared columns present in actual loaded CSV,True,
6,lama_restoration_manifest_loaded_non_empty,410,> 0,True,
7,lama_restoration_manifest_loaded_column_count_positive,218,> 0,True,
8,lama_restoration_manifest_row_count_matches_inventory,410,410,True,
9,lama_restoration_manifest_column_count_matches_inventory,218,218,True,


In [15]:
batch1_source_shapes_export_df = batch1_source_shapes_df.copy()

for column in [
    "actual_columns",
    "inventory_declared_columns",
    "required_columns",
    "missing_required_columns_actual",
    "extra_actual_columns_not_in_inventory",
    "inventory_columns_not_in_actual",
]:
    batch1_source_shapes_export_df[column] = batch1_source_shapes_export_df[column].map(
        lambda value: json.dumps(value, ensure_ascii=False)
    )

batch1_schema_validation_export_df = batch1_schema_validation_df.copy()
for column in ["observed", "expected"]:
    batch1_schema_validation_export_df[column] = batch1_schema_validation_export_df[column].map(json_for_csv)

batch1_source_shapes_export_df.to_csv(BATCH1_SOURCE_TABLE_SHAPES_OUTPUT_PATH, index=False)
batch1_schema_validation_export_df.to_csv(BATCH1_SCHEMA_VALIDATION_OUTPUT_PATH, index=False)

stage_manifest = read_json(STAGE_MANIFEST_JSON_PATH)

stage_manifest.update(
    {
        "status": "batch_1_source_tables_loaded_complete",
        "updated_at_utc": datetime.now(timezone.utc).isoformat(),
    }
)

stage_manifest.setdefault("outputs", {})
stage_manifest["outputs"].update(
    {
        "batch1_source_table_shapes": project_relative_path(BATCH1_SOURCE_TABLE_SHAPES_OUTPUT_PATH),
        "batch1_schema_validation": project_relative_path(BATCH1_SCHEMA_VALIDATION_OUTPUT_PATH),
    }
)

stage_manifest.setdefault("row_counts", {})
stage_manifest["row_counts"].update(
    {
        "batch1_loaded_source_tables": int(len(SOURCE_TABLES)),
        "batch1_schema_validation_checks": int(len(batch1_schema_validation_df)),
    }
)

stage_manifest["batch1_source_shapes"] = {
    source_id: {
        "rows": int(source_df.shape[0]),
        "columns": int(source_df.shape[1]),
    }
    for source_id, source_df in SOURCE_TABLES.items()
}

stage_manifest["next_batch"] = (
    "Batch 2: normalize OpenCV and LaMa metric tables, preserve case_id granularity, "
    "and create metric-level paired comparisons."
)

save_json(STAGE_MANIFEST_JSON_PATH, stage_manifest)

print("Batch 1 complete.")
print("Saved:", project_relative_path(BATCH1_SOURCE_TABLE_SHAPES_OUTPUT_PATH))
print("Saved:", project_relative_path(BATCH1_SCHEMA_VALIDATION_OUTPUT_PATH))
print("Updated manifest:", project_relative_path(STAGE_MANIFEST_JSON_PATH))

Batch 1 complete.
Saved: outputs/20_opencv_lama_comparison_inventory_rebuild_v2/batch1_sources/batch1_source_table_shapes.csv
Saved: outputs/20_opencv_lama_comparison_inventory_rebuild_v2/batch1_sources/batch1_source_schema_validation.csv
Updated manifest: outputs/20_opencv_lama_comparison_inventory_rebuild_v2/stage_manifest.json


In [16]:
BATCH2_OUTPUT_DIR = OUTPUT_ROOT / "batch2_pairing"

for directory in [BATCH2_OUTPUT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

BATCH2_RESTORATION_PAIRS_OUTPUT_PATH = BATCH2_OUTPUT_DIR / "batch2_opencv_lama_restoration_pairs.csv"
BATCH2_METRIC_LONG_OUTPUT_PATH = BATCH2_OUTPUT_DIR / "batch2_metric_long.csv"
BATCH2_METRIC_PAIRS_OUTPUT_PATH = BATCH2_OUTPUT_DIR / "batch2_opencv_lama_metric_pairs.csv"
BATCH2_PAIRING_AUDIT_OUTPUT_PATH = BATCH2_OUTPUT_DIR / "batch2_pairing_audit.csv"
BATCH2_VALIDATION_OUTPUT_PATH = BATCH2_OUTPUT_DIR / "batch2_validation.csv"

print("BATCH2_OUTPUT_DIR:", project_relative_path(BATCH2_OUTPUT_DIR))

BATCH2_OUTPUT_DIR: outputs/20_opencv_lama_comparison_inventory_rebuild_v2/batch2_pairing


In [17]:
BATCH2_CASE_KEY_COLUMNS = [
    "case_id",
    "dataset_name",
    "painting_id",
    "mask_id",
    "mask_type",
]

BATCH2_METRIC_PAIR_KEY_COLUMNS = [
    *BATCH2_CASE_KEY_COLUMNS,
    "evaluation_region",
    "metric_family",
    "metric_name",
    "metric_value_type",
]

BATCH2_OPTIONAL_RESTORATION_CONTEXT_COLUMNS = [
    "damage_size_category",
    "damage_fraction",
    "mask_fraction",
    "mask_area_pixels",
    "clean_path",
    "damaged_path",
    "mask_path",
    "restored_path",
    "runtime_seconds",
    "output_written",
    "status",
    "issue",
    "iopaint_returncode",
]

BATCH2_METRIC_SPECS = [
    {
        "source_family": "classical_metrics",
        "opencv_source_id": "opencv_classical_metrics",
        "lama_source_id": "lama_classical_metrics",
        "metrics": [
            {"column": "restored_mse", "metric_name": "mse", "metric_value_type": "restored", "direction": "lower_is_better"},
            {"column": "mse_improvement", "metric_name": "mse", "metric_value_type": "improvement", "direction": "higher_is_better"},
            {"column": "restored_psnr", "metric_name": "psnr", "metric_value_type": "restored", "direction": "higher_is_better"},
            {"column": "psnr_improvement", "metric_name": "psnr", "metric_value_type": "improvement", "direction": "higher_is_better"},
            {"column": "restored_ssim", "metric_name": "ssim", "metric_value_type": "restored", "direction": "higher_is_better"},
            {"column": "ssim_improvement", "metric_name": "ssim", "metric_value_type": "improvement", "direction": "higher_is_better"},
        ],
    },
    {
        "source_family": "lpips_metrics",
        "opencv_source_id": "opencv_lpips_metrics",
        "lama_source_id": "lama_lpips_metrics",
        "metrics": [
            {"column": "restored_lpips", "metric_name": "lpips", "metric_value_type": "restored", "direction": "lower_is_better"},
            {"column": "lpips_improvement", "metric_name": "lpips", "metric_value_type": "improvement", "direction": "higher_is_better"},
        ],
    },
    {
        "source_family": "feature_metrics",
        "opencv_source_id": "opencv_feature_metrics",
        "lama_source_id": "lama_feature_metrics",
        "metrics": [
            {"column": "clip_restored_similarity", "metric_name": "clip", "metric_value_type": "restored", "direction": "higher_is_better"},
            {"column": "clip_similarity_improvement", "metric_name": "clip", "metric_value_type": "improvement", "direction": "higher_is_better"},
            {"column": "dinov2_restored_similarity", "metric_name": "dinov2", "metric_value_type": "restored", "direction": "higher_is_better"},
            {"column": "dinov2_similarity_improvement", "metric_name": "dinov2", "metric_value_type": "improvement", "direction": "higher_is_better"},
        ],
    },
]

BATCH2_NUMERIC_TIE_TOLERANCE = 1e-12

In [18]:
def batch2_text(value) -> str:
    if value is None:
        return ""
    if isinstance(value, float) and np.isnan(value):
        return ""
    text = str(value).strip()
    return "" if text.lower() in {"nan", "none", "null", "<na>"} else text


def batch2_normalize_key_columns(df: pd.DataFrame, key_columns: list[str]) -> pd.DataFrame:
    df = df.copy()

    missing_columns = [column for column in key_columns if column not in df.columns]
    if missing_columns:
        raise RuntimeError(f"Missing required key columns: {missing_columns}")

    for column in key_columns:
        df[column] = df[column].map(batch2_text)

    return df


def batch2_numeric_series(series: pd.Series) -> pd.Series:
    return pd.to_numeric(series, errors="coerce")


def batch2_source_relative_path(source_id: str) -> str:
    row = batch0_source_plan_df.loc[batch0_source_plan_df["source_id"].eq(source_id)]
    if row.empty:
        return ""
    return row.iloc[0]["relative_path"]


def batch2_available_columns(df: pd.DataFrame, columns: list[str]) -> list[str]:
    return [column for column in columns if column in df.columns]

In [19]:
def batch2_prefix_non_key_columns(
    df: pd.DataFrame,
    method_prefix: str,
    key_columns: list[str],
    preferred_context_columns: list[str],
) -> pd.DataFrame:
    df = batch2_normalize_key_columns(df, key_columns)

    selected_columns = list(dict.fromkeys([
        *key_columns,
        *batch2_available_columns(df, preferred_context_columns),
    ]))

    result_df = df[selected_columns].copy()

    rename_map = {
        column: f"{method_prefix}_{column}"
        for column in result_df.columns
        if column not in key_columns
    }

    result_df = result_df.rename(columns=rename_map)
    result_df[f"{method_prefix}_source_rows"] = 1

    return result_df


opencv_restoration_df = SOURCE_TABLES["opencv_restoration_manifest"].copy()
lama_restoration_df = SOURCE_TABLES["lama_restoration_manifest"].copy()

opencv_restoration_prefixed_df = batch2_prefix_non_key_columns(
    opencv_restoration_df,
    "opencv",
    BATCH2_CASE_KEY_COLUMNS,
    BATCH2_OPTIONAL_RESTORATION_CONTEXT_COLUMNS,
)

lama_restoration_prefixed_df = batch2_prefix_non_key_columns(
    lama_restoration_df,
    "lama",
    BATCH2_CASE_KEY_COLUMNS,
    BATCH2_OPTIONAL_RESTORATION_CONTEXT_COLUMNS,
)

batch2_restoration_pairs_df = opencv_restoration_prefixed_df.merge(
    lama_restoration_prefixed_df,
    on=BATCH2_CASE_KEY_COLUMNS,
    how="outer",
    validate="one_to_one",
)

batch2_restoration_pairs_df["restoration_pair_status"] = np.select(
    [
        batch2_restoration_pairs_df["opencv_source_rows"].notna()
        & batch2_restoration_pairs_df["lama_source_rows"].notna(),
        batch2_restoration_pairs_df["opencv_source_rows"].notna()
        & batch2_restoration_pairs_df["lama_source_rows"].isna(),
        batch2_restoration_pairs_df["opencv_source_rows"].isna()
        & batch2_restoration_pairs_df["lama_source_rows"].notna(),
    ],
    ["paired", "opencv_only", "lama_only"],
    default="unknown",
)

display(batch2_restoration_pairs_df.head())
display(batch2_restoration_pairs_df["restoration_pair_status"].value_counts(dropna=False).reset_index(name="rows"))

,case_id,dataset_name,painting_id,mask_id,mask_type,opencv_clean_path,opencv_damaged_path,opencv_mask_path,opencv_restored_path,opencv_runtime_seconds,opencv_output_written,opencv_status,opencv_issue,opencv_source_rows,lama_mask_area_pixels,lama_clean_path,lama_damaged_path,lama_mask_path,lama_restored_path,lama_runtime_seconds,lama_output_written,lama_status,lama_issue,lama_iopaint_returncode,lama_source_rows,restoration_pair_status
0,canonical__p001_loss_large,canonical,p001,p001_loss_large,loss_large,data/processed/clean/p001_clean.png,data/processed/masked/p001_loss_large_damaged.png,data/processed/masks/p001_loss_large_mask.png,data\processed\restored\opencv_telea\canonical\canonical__p001_loss_large_restored_opencv_telea.png,0.649697,True,ok,NaN,1,69018,data\processed\clean\p001_clean.png,data\processed\masked\p001_loss_large_damaged.png,data\processed\masks\p001_loss_large_mask.png,data/processed/restored/lama/canonical/canonical__p001_loss_large_restored_lama.png,1.460938,True,ok,NaN,0,1,paired
1,canonical__p001_loss_small,canonical,p001,p001_loss_small,loss_small,data/processed/clean/p001_clean.png,data/processed/masked/p001_loss_small_damaged.png,data/processed/masks/p001_loss_small_mask.png,data\processed\restored\opencv_telea\canonical\canonical__p001_loss_small_restored_opencv_telea.png,0.577950,True,ok,NaN,1,18239,data\processed\clean\p001_clean.png,data\processed\masked\p001_loss_small_damaged.png,data\processed\masks\p001_loss_small_mask.png,data/processed/restored/lama/canonical/canonical__p001_loss_small_restored_lama.png,1.460938,True,ok,NaN,0,1,paired
2,canonical__p001_mixed_damage,canonical,p001,p001_mixed_damage,mixed_damage,data/processed/clean/p001_clean.png,data/processed/masked/p001_mixed_damage_damaged.png,data/processed/masks/p001_mixed_damage_mask.png,data\processed\restored\opencv_telea\canonical\canonical__p001_mixed_damage_restored_opencv_telea.png,0.642271,True,ok,NaN,1,45733,data\processed\clean\p001_clean.png,data\processed\masked\p001_mixed_damage_damaged.png,data\processed\masks\p001_mixed_damage_mask.png,data/processed/restored/lama/canonical/canonical__p001_mixed_damage_restored_lama.png,1.460938,True,ok,NaN,0,1,paired
3,canonical__p001_scratch_thin,canonical,p001,p001_scratch_thin,scratch_thin,data/processed/clean/p001_clean.png,data/processed/masked/p001_scratch_thin_damaged.png,data/processed/masks/p001_scratch_thin_mask.png,data\processed\restored\opencv_telea\canonical\canonical__p001_scratch_thin_restored_opencv_telea.png,0.595698,True,ok,NaN,1,12058,data\processed\clean\p001_clean.png,data\processed\masked\p001_scratch_thin_damaged.png,data\processed\masks\p001_scratch_thin_mask.png,data/processed/restored/lama/canonical/canonical__p001_scratch_thin_restored_lama.png,1.460938,True,ok,NaN,0,1,paired
4,canonical__p001_zero_control,canonical,p001,p001_zero_control,zero_control,data/processed/clean/p001_clean.png,data/processed/masked/p001_zero_control_damaged.png,data/processed/masks/p001_zero_control_mask.png,data\processed\restored\opencv_telea\canonical\canonical__p001_zero_control_restored_opencv_telea.png,0.552721,True,ok,NaN,1,0,data\processed\clean\p001_clean.png,data\processed\masked\p001_zero_control_damaged.png,data\processed\masks\p001_zero_control_mask.png,data/processed/restored/lama/canonical/canonical__p001_zero_control_restored_lama.png,0.000000,True,ok,zero_control copied without model inference,0,1,paired


,restoration_pair_status,rows
0,paired,410


In [20]:
def batch2_metric_table_to_long(
    source_id: str,
    method: str,
    source_family: str,
    metric_specs: list[dict],
) -> pd.DataFrame:
    source_df = SOURCE_TABLES[source_id].copy()
    source_df = batch2_normalize_key_columns(source_df, [*BATCH2_CASE_KEY_COLUMNS, "evaluation_region"])

    long_frames = []

    for metric_spec in metric_specs:
        metric_column = metric_spec["column"]

        if metric_column not in source_df.columns:
            continue

        columns_to_keep = [
            *BATCH2_CASE_KEY_COLUMNS,
            "evaluation_region",
            metric_column,
        ]

        optional_columns = batch2_available_columns(
            source_df,
            ["status", "issue", "runtime_seconds", "restored_path"],
        )

        metric_long_df = source_df[[*columns_to_keep, *optional_columns]].copy()
        metric_long_df = metric_long_df.rename(columns={metric_column: "metric_value"})

        metric_long_df["metric_value"] = batch2_numeric_series(metric_long_df["metric_value"])
        metric_long_df["method"] = method
        metric_long_df["source_id"] = source_id
        metric_long_df["source_relative_path"] = batch2_source_relative_path(source_id)
        metric_long_df["metric_family"] = source_family
        metric_long_df["metric_column"] = metric_column
        metric_long_df["metric_name"] = metric_spec["metric_name"]
        metric_long_df["metric_value_type"] = metric_spec["metric_value_type"]
        metric_long_df["metric_direction"] = metric_spec["direction"]
        metric_long_df["metric_value_is_numeric"] = metric_long_df["metric_value"].notna()

        long_frames.append(metric_long_df)

    if not long_frames:
        return pd.DataFrame()

    return pd.concat(long_frames, ignore_index=True)

In [21]:
metric_long_frames = []

for family_spec in BATCH2_METRIC_SPECS:
    metric_long_frames.append(
        batch2_metric_table_to_long(
            source_id=family_spec["opencv_source_id"],
            method="opencv_telea",
            source_family=family_spec["source_family"],
            metric_specs=family_spec["metrics"],
        )
    )
    metric_long_frames.append(
        batch2_metric_table_to_long(
            source_id=family_spec["lama_source_id"],
            method="lama",
            source_family=family_spec["source_family"],
            metric_specs=family_spec["metrics"],
        )
    )

batch2_metric_long_df = pd.concat(
    [frame for frame in metric_long_frames if not frame.empty],
    ignore_index=True,
)

batch2_metric_long_df = batch2_metric_long_df[
    [
        *BATCH2_CASE_KEY_COLUMNS,
        "evaluation_region",
        "method",
        "source_id",
        "source_relative_path",
        "metric_family",
        "metric_column",
        "metric_name",
        "metric_value_type",
        "metric_direction",
        "metric_value",
        "metric_value_is_numeric",
        *batch2_available_columns(batch2_metric_long_df, ["status", "issue", "runtime_seconds", "restored_path"]),
    ]
].copy()

display(batch2_metric_long_df.head())
display(
    batch2_metric_long_df.groupby(
        ["method", "metric_family", "metric_name", "metric_value_type", "evaluation_region"],
        dropna=False,
    )
    .size()
    .reset_index(name="rows")
    .head(40)
)

,case_id,dataset_name,painting_id,mask_id,mask_type,evaluation_region,method,source_id,source_relative_path,metric_family,metric_column,metric_name,metric_value_type,metric_direction,metric_value,metric_value_is_numeric,status,issue,restored_path
0,canonical__p001_loss_large,canonical,p001,p001_loss_large,loss_large,boundary_region,opencv_telea,opencv_classical_metrics,data/processed/metrics/metrics_opencv_telea_classical.csv,classical_metrics,restored_mse,mse,restored,lower_is_better,6.173440,True,ok,NaN,NaN
1,canonical__p001_loss_large,canonical,p001,p001_loss_large,loss_large,content_region,opencv_telea,opencv_classical_metrics,data/processed/metrics/metrics_opencv_telea_classical.csv,classical_metrics,restored_mse,mse,restored,lower_is_better,7.165670,True,ok,NaN,NaN
2,canonical__p001_loss_large,canonical,p001,p001_loss_large,loss_large,full_image,opencv_telea,opencv_classical_metrics,data/processed/metrics/metrics_opencv_telea_classical.csv,classical_metrics,restored_mse,mse,restored,lower_is_better,6.185988,True,ok,NaN,NaN
3,canonical__p001_loss_large,canonical,p001,p001_loss_large,loss_large,mask_bbox_crop,opencv_telea,opencv_classical_metrics,data/processed/metrics/metrics_opencv_telea_classical.csv,classical_metrics,restored_mse,mse,restored,lower_is_better,29.282860,True,ok,NaN,NaN
4,canonical__p001_loss_large,canonical,p001,p001_loss_large,loss_large,masked_region,opencv_telea,opencv_classical_metrics,data/processed/metrics/metrics_opencv_telea_classical.csv,classical_metrics,restored_mse,mse,restored,lower_is_better,52.865112,True,ok,NaN,NaN


,method,metric_family,metric_name,metric_value_type,evaluation_region,rows
0,lama,classical_metrics,mse,improvement,boundary_region,360
1,lama,classical_metrics,mse,improvement,content_region,410
2,lama,classical_metrics,mse,improvement,full_image,410
3,lama,classical_metrics,mse,improvement,mask_bbox_crop,360
4,lama,classical_metrics,mse,improvement,masked_region,360
5,lama,classical_metrics,mse,improvement,outside_mask_region,360
6,lama,classical_metrics,mse,restored,boundary_region,360
7,lama,classical_metrics,mse,restored,content_region,410
8,lama,classical_metrics,mse,restored,full_image,410
9,lama,classical_metrics,mse,restored,mask_bbox_crop,360


In [22]:
def batch2_duplicate_key_audit(df: pd.DataFrame, key_columns: list[str], table_name: str) -> pd.DataFrame:
    duplicate_counts_df = (
        df.groupby(key_columns, dropna=False)
        .size()
        .reset_index(name="rows_per_key")
    )
    duplicate_counts_df = duplicate_counts_df[duplicate_counts_df["rows_per_key"] > 1].copy()
    duplicate_counts_df.insert(0, "table_name", table_name)
    return duplicate_counts_df


batch2_duplicate_audit_frames = [
    batch2_duplicate_key_audit(
        opencv_restoration_df,
        BATCH2_CASE_KEY_COLUMNS,
        "opencv_restoration_manifest",
    ),
    batch2_duplicate_key_audit(
        lama_restoration_df,
        BATCH2_CASE_KEY_COLUMNS,
        "lama_restoration_manifest",
    ),
    batch2_duplicate_key_audit(
        batch2_metric_long_df,
        [*BATCH2_METRIC_PAIR_KEY_COLUMNS, "method"],
        "batch2_metric_long",
    ),
]

batch2_duplicate_audit_df = pd.concat(batch2_duplicate_audit_frames, ignore_index=True)

display(batch2_duplicate_audit_df.head())

if not batch2_duplicate_audit_df.empty:
    raise RuntimeError(
        "Batch 2 found duplicate pairing keys. Inspect batch2_duplicate_audit_df before pairing."
    )

,table_name,case_id,dataset_name,painting_id,mask_id,mask_type,rows_per_key,evaluation_region,metric_family,metric_name,metric_value_type,method


In [23]:
opencv_metric_long_df = batch2_metric_long_df[
    batch2_metric_long_df["method"].eq("opencv_telea")
].copy()

lama_metric_long_df = batch2_metric_long_df[
    batch2_metric_long_df["method"].eq("lama")
].copy()

opencv_metric_pair_df = opencv_metric_long_df.rename(
    columns={
        "source_id": "opencv_source_id",
        "source_relative_path": "opencv_source_relative_path",
        "metric_column": "opencv_metric_column",
        "metric_value": "opencv_metric_value",
        "metric_value_is_numeric": "opencv_metric_value_is_numeric",
        "status": "opencv_status",
        "issue": "opencv_issue",
        "runtime_seconds": "opencv_runtime_seconds",
        "restored_path": "opencv_restored_path",
    }
).drop(columns=["method"], errors="ignore")

lama_metric_pair_df = lama_metric_long_df.rename(
    columns={
        "source_id": "lama_source_id",
        "source_relative_path": "lama_source_relative_path",
        "metric_column": "lama_metric_column",
        "metric_value": "lama_metric_value",
        "metric_value_is_numeric": "lama_metric_value_is_numeric",
        "status": "lama_status",
        "issue": "lama_issue",
        "runtime_seconds": "lama_runtime_seconds",
        "restored_path": "lama_restored_path",
    }
).drop(columns=["method"], errors="ignore")

batch2_metric_pairs_df = opencv_metric_pair_df.merge(
    lama_metric_pair_df,
    on=BATCH2_METRIC_PAIR_KEY_COLUMNS + ["metric_direction"],
    how="outer",
    validate="one_to_one",
)

batch2_metric_pairs_df["metric_pair_status"] = np.select(
    [
        batch2_metric_pairs_df["opencv_metric_value"].notna()
        & batch2_metric_pairs_df["lama_metric_value"].notna(),
        batch2_metric_pairs_df["opencv_metric_value"].notna()
        & batch2_metric_pairs_df["lama_metric_value"].isna(),
        batch2_metric_pairs_df["opencv_metric_value"].isna()
        & batch2_metric_pairs_df["lama_metric_value"].notna(),
    ],
    ["paired", "opencv_only", "lama_only"],
    default="unknown",
)

batch2_metric_pairs_df["lama_minus_opencv"] = (
    batch2_metric_pairs_df["lama_metric_value"] - batch2_metric_pairs_df["opencv_metric_value"]
)

batch2_metric_pairs_df["lama_advantage"] = np.where(
    batch2_metric_pairs_df["metric_direction"].eq("higher_is_better"),
    batch2_metric_pairs_df["lama_minus_opencv"],
    -batch2_metric_pairs_df["lama_minus_opencv"],
)

batch2_metric_pairs_df["absolute_metric_delta"] = batch2_metric_pairs_df["lama_minus_opencv"].abs()

In [24]:
def batch2_winner(row: pd.Series) -> str:
    if row["metric_pair_status"] != "paired":
        return row["metric_pair_status"]

    lama_advantage = row["lama_advantage"]

    if pd.isna(lama_advantage):
        return "paired_non_numeric"

    if abs(lama_advantage) <= BATCH2_NUMERIC_TIE_TOLERANCE:
        return "tie"

    if lama_advantage > 0:
        return "lama"

    return "opencv_telea"


batch2_metric_pairs_df["metric_winner"] = batch2_metric_pairs_df.apply(batch2_winner, axis=1)

display(batch2_metric_pairs_df.head())
display(batch2_metric_pairs_df["metric_pair_status"].value_counts(dropna=False).reset_index(name="rows"))
display(batch2_metric_pairs_df["metric_winner"].value_counts(dropna=False).reset_index(name="rows"))

,case_id,dataset_name,painting_id,mask_id,mask_type,evaluation_region,opencv_source_id,opencv_source_relative_path,metric_family,opencv_metric_column,metric_name,metric_value_type,metric_direction,opencv_metric_value,opencv_metric_value_is_numeric,opencv_status,opencv_issue,opencv_restored_path,lama_source_id,lama_source_relative_path,lama_metric_column,lama_metric_value,lama_metric_value_is_numeric,lama_status,lama_issue,lama_restored_path,metric_pair_status,lama_minus_opencv,lama_advantage,absolute_metric_delta,metric_winner
0,canonical__p001_loss_large,canonical,p001,p001_loss_large,loss_large,boundary_region,opencv_classical_metrics,data/processed/metrics/metrics_opencv_telea_classical.csv,classical_metrics,mse_improvement,mse,improvement,higher_is_better,53.382518,True,ok,NaN,NaN,lama_classical_metrics,outputs/metrics/classical_metrics_lama.csv,mse_improvement,25396.024811,True,ok,NaN,D:\Masters\FH\Thesis\painting-restoration-eval\data\processed\restored\lama\canonical\canonical__p001_loss_large_restored_lama.png,paired,25342.642293,25342.642293,25342.642293,lama
1,canonical__p001_loss_large,canonical,p001,p001_loss_large,loss_large,boundary_region,opencv_classical_metrics,data/processed/metrics/metrics_opencv_telea_classical.csv,classical_metrics,restored_mse,mse,restored,lower_is_better,6.173440,True,ok,NaN,NaN,lama_classical_metrics,outputs/metrics/classical_metrics_lama.csv,restored_mse,7.777923,True,ok,NaN,D:\Masters\FH\Thesis\painting-restoration-eval\data\processed\restored\lama\canonical\canonical__p001_loss_large_restored_lama.png,paired,1.604483,-1.604483,1.604483,opencv_telea
2,canonical__p001_loss_large,canonical,p001,p001_loss_large,loss_large,boundary_region,opencv_classical_metrics,data/processed/metrics/metrics_opencv_telea_classical.csv,classical_metrics,psnr_improvement,psnr,improvement,higher_is_better,9.843980,True,ok,NaN,NaN,lama_classical_metrics,outputs/metrics/classical_metrics_lama.csv,psnr_improvement,35.140351,True,ok,NaN,D:\Masters\FH\Thesis\painting-restoration-eval\data\processed\restored\lama\canonical\canonical__p001_loss_large_restored_lama.png,paired,25.296371,25.296371,25.296371,lama
3,canonical__p001_loss_large,canonical,p001,p001_loss_large,loss_large,boundary_region,opencv_classical_metrics,data/processed/metrics/metrics_opencv_telea_classical.csv,classical_metrics,restored_psnr,psnr,restored,higher_is_better,40.225531,True,ok,NaN,NaN,lama_classical_metrics,outputs/metrics/classical_metrics_lama.csv,restored_psnr,39.222167,True,ok,NaN,D:\Masters\FH\Thesis\painting-restoration-eval\data\processed\restored\lama\canonical\canonical__p001_loss_large_restored_lama.png,paired,-1.003364,-1.003364,1.003364,opencv_telea
4,canonical__p001_loss_large,canonical,p001,p001_loss_large,loss_large,boundary_region,opencv_classical_metrics,data/processed/metrics/metrics_opencv_telea_classical.csv,classical_metrics,ssim_improvement,ssim,improvement,higher_is_better,NaN,False,ok,NaN,NaN,lama_classical_metrics,outputs/metrics/classical_metrics_lama.csv,ssim_improvement,NaN,False,ok,NaN,D:\Masters\FH\Thesis\painting-restoration-eval\data\processed\restored\lama\canonical\canonical__p001_loss_large_restored_lama.png,unknown,NaN,NaN,NaN,unknown


,metric_pair_status,rows
0,paired,18380
1,unknown,2260
2,opencv_only,200
3,lama_only,70


,metric_winner,rows
0,lama,10011
1,opencv_telea,5729
2,unknown,2260
3,tie,2230
4,paired_non_numeric,410
5,opencv_only,200
6,lama_only,70


In [25]:
batch2_metric_pair_summary_df = (
    batch2_metric_pairs_df.groupby(
        [
            "metric_family",
            "metric_name",
            "metric_value_type",
            "metric_direction",
            "evaluation_region",
            "metric_pair_status",
            "metric_winner",
        ],
        dropna=False,
    )
    .size()
    .reset_index(name="rows")
)

batch2_metric_numeric_audit_df = (
    batch2_metric_long_df.groupby(
        ["method", "metric_family", "metric_name", "metric_value_type", "evaluation_region"],
        dropna=False,
    )
    .agg(
        rows=("metric_value", "size"),
        numeric_rows=("metric_value_is_numeric", "sum"),
        missing_or_non_numeric_rows=("metric_value_is_numeric", lambda value: int((~value.astype(bool)).sum())),
    )
    .reset_index()
)

batch2_restoration_pair_summary_df = (
    batch2_restoration_pairs_df.groupby("restoration_pair_status", dropna=False)
    .size()
    .reset_index(name="rows")
)

batch2_pairing_audit_df = pd.concat(
    [
        batch2_restoration_pair_summary_df.assign(audit_section="restoration_pair_status"),
        batch2_metric_pair_summary_df.assign(audit_section="metric_pair_status"),
        batch2_metric_numeric_audit_df.assign(audit_section="metric_numeric_parse_status"),
    ],
    ignore_index=True,
    sort=False,
)

display(batch2_pairing_audit_df.head(50))

,restoration_pair_status,rows,audit_section,metric_family,metric_name,metric_value_type,metric_direction,evaluation_region,metric_pair_status,metric_winner,method,numeric_rows,missing_or_non_numeric_rows
0,paired,410,restoration_pair_status,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,5,metric_pair_status,classical_metrics,mse,improvement,higher_is_better,boundary_region,lama_only,lama_only,NaN,NaN,NaN
2,NaN,336,metric_pair_status,classical_metrics,mse,improvement,higher_is_better,boundary_region,paired,lama,NaN,NaN,NaN
3,NaN,19,metric_pair_status,classical_metrics,mse,improvement,higher_is_better,boundary_region,paired,opencv_telea,NaN,NaN,NaN
4,NaN,310,metric_pair_status,classical_metrics,mse,improvement,higher_is_better,content_region,paired,lama,NaN,NaN,NaN
5,NaN,45,metric_pair_status,classical_metrics,mse,improvement,higher_is_better,content_region,paired,opencv_telea,NaN,NaN,NaN
6,NaN,55,metric_pair_status,classical_metrics,mse,improvement,higher_is_better,content_region,paired,tie,NaN,NaN,NaN
7,NaN,310,metric_pair_status,classical_metrics,mse,improvement,higher_is_better,full_image,paired,lama,NaN,NaN,NaN
8,NaN,45,metric_pair_status,classical_metrics,mse,improvement,higher_is_better,full_image,paired,opencv_telea,NaN,NaN,NaN
9,NaN,55,metric_pair_status,classical_metrics,mse,improvement,higher_is_better,full_image,paired,tie,NaN,NaN,NaN


In [26]:
batch2_check_rows = [
    build_check(
        "restoration_pairs_have_rows",
        len(batch2_restoration_pairs_df),
        "> 0",
        len(batch2_restoration_pairs_df) > 0,
        "No restoration pairs were produced.",
    ),
    build_check(
        "metric_long_has_rows",
        len(batch2_metric_long_df),
        "> 0",
        len(batch2_metric_long_df) > 0,
        "No long metric rows were produced.",
    ),
    build_check(
        "metric_pairs_have_rows",
        len(batch2_metric_pairs_df),
        "> 0",
        len(batch2_metric_pairs_df) > 0,
        "No metric pair rows were produced.",
    ),
    build_check(
        "no_duplicate_restoration_or_metric_pair_keys",
        len(batch2_duplicate_audit_df),
        "0 duplicate keys",
        len(batch2_duplicate_audit_df) == 0,
        "Duplicate pairing keys detected.",
    ),
    build_check(
        "metric_pairs_include_classical_lpips_clip_dinov2",
        sorted(batch2_metric_pairs_df["metric_name"].dropna().unique().tolist()),
        ["clip", "dinov2", "lpips", "mse", "psnr", "ssim"],
        set(["clip", "dinov2", "lpips", "mse", "psnr", "ssim"]).issubset(
            set(batch2_metric_pairs_df["metric_name"].dropna().unique())
        ),
        "Not all expected metric families were represented in the paired metric table.",
    ),
    build_check(
        "metric_pair_key_includes_case_id",
        BATCH2_METRIC_PAIR_KEY_COLUMNS,
        "case_id included in metric pairing key",
        "case_id" in BATCH2_METRIC_PAIR_KEY_COLUMNS,
        "Metric pairing key does not include case_id.",
    ),
]

batch2_validation_df = pd.DataFrame(batch2_check_rows)

display(batch2_validation_df)

if not batch2_validation_df["passed"].astype(bool).all():
    failed_batch2_checks_df = batch2_validation_df[
        ~batch2_validation_df["passed"].astype(bool)
    ].copy()
    display(failed_batch2_checks_df)
    raise RuntimeError("Notebook 20 Batch 2 validation failed. Inspect pairing outputs.")

,check_name,observed,expected,passed,failure_message
0,restoration_pairs_have_rows,410,> 0,True,
1,metric_long_has_rows,41430,> 0,True,
2,metric_pairs_have_rows,20910,> 0,True,
3,no_duplicate_restoration_or_metric_pair_keys,0,0 duplicate keys,True,
4,metric_pairs_include_classical_lpips_clip_dinov2,"[clip, dinov2, lpips, mse, psnr, ssim]","[clip, dinov2, lpips, mse, psnr, ssim]",True,
5,metric_pair_key_includes_case_id,"[case_id, dataset_name, painting_id, mask_id, mask_type, evaluation_region, metric_family, metric_name, metric_value_type]",case_id included in metric pairing key,True,


In [27]:
batch2_restoration_pairs_df.to_csv(BATCH2_RESTORATION_PAIRS_OUTPUT_PATH, index=False)
batch2_metric_long_df.to_csv(BATCH2_METRIC_LONG_OUTPUT_PATH, index=False)
batch2_metric_pairs_df.to_csv(BATCH2_METRIC_PAIRS_OUTPUT_PATH, index=False)
batch2_pairing_audit_df.to_csv(BATCH2_PAIRING_AUDIT_OUTPUT_PATH, index=False)

batch2_validation_export_df = batch2_validation_df.copy()
for column in ["observed", "expected"]:
    batch2_validation_export_df[column] = batch2_validation_export_df[column].map(json_for_csv)

batch2_validation_export_df.to_csv(BATCH2_VALIDATION_OUTPUT_PATH, index=False)

stage_manifest = read_json(STAGE_MANIFEST_JSON_PATH)

stage_manifest.update(
    {
        "status": "batch_2_opencv_lama_pairing_complete",
        "updated_at_utc": datetime.now(timezone.utc).isoformat(),
    }
)

stage_manifest.setdefault("outputs", {})
stage_manifest["outputs"].update(
    {
        "batch2_restoration_pairs": project_relative_path(BATCH2_RESTORATION_PAIRS_OUTPUT_PATH),
        "batch2_metric_long": project_relative_path(BATCH2_METRIC_LONG_OUTPUT_PATH),
        "batch2_metric_pairs": project_relative_path(BATCH2_METRIC_PAIRS_OUTPUT_PATH),
        "batch2_pairing_audit": project_relative_path(BATCH2_PAIRING_AUDIT_OUTPUT_PATH),
        "batch2_validation": project_relative_path(BATCH2_VALIDATION_OUTPUT_PATH),
    }
)

stage_manifest.setdefault("row_counts", {})
stage_manifest["row_counts"].update(
    {
        "batch2_restoration_pairs": int(len(batch2_restoration_pairs_df)),
        "batch2_metric_long_rows": int(len(batch2_metric_long_df)),
        "batch2_metric_pair_rows": int(len(batch2_metric_pairs_df)),
        "batch2_pairing_audit_rows": int(len(batch2_pairing_audit_df)),
        "batch2_validation_checks": int(len(batch2_validation_df)),
    }
)

stage_manifest["batch2_pairing_key"] = {
    "case_key_columns": BATCH2_CASE_KEY_COLUMNS,
    "metric_pair_key_columns": BATCH2_METRIC_PAIR_KEY_COLUMNS,
}

stage_manifest["next_batch"] = (
    "Batch 3: build unified paired-case table by joining restoration pairs with "
    "metric-pair evidence and per-case metric vote summaries."
)

save_json(STAGE_MANIFEST_JSON_PATH, stage_manifest)

print("Batch 2 complete.")
print("Saved:", project_relative_path(BATCH2_RESTORATION_PAIRS_OUTPUT_PATH))
print("Saved:", project_relative_path(BATCH2_METRIC_LONG_OUTPUT_PATH))
print("Saved:", project_relative_path(BATCH2_METRIC_PAIRS_OUTPUT_PATH))
print("Saved:", project_relative_path(BATCH2_PAIRING_AUDIT_OUTPUT_PATH))
print("Saved:", project_relative_path(BATCH2_VALIDATION_OUTPUT_PATH))
print("Updated manifest:", project_relative_path(STAGE_MANIFEST_JSON_PATH))

Batch 2 complete.
Saved: outputs/20_opencv_lama_comparison_inventory_rebuild_v2/batch2_pairing/batch2_opencv_lama_restoration_pairs.csv
Saved: outputs/20_opencv_lama_comparison_inventory_rebuild_v2/batch2_pairing/batch2_metric_long.csv
Saved: outputs/20_opencv_lama_comparison_inventory_rebuild_v2/batch2_pairing/batch2_opencv_lama_metric_pairs.csv
Saved: outputs/20_opencv_lama_comparison_inventory_rebuild_v2/batch2_pairing/batch2_pairing_audit.csv
Saved: outputs/20_opencv_lama_comparison_inventory_rebuild_v2/batch2_pairing/batch2_validation.csv
Updated manifest: outputs/20_opencv_lama_comparison_inventory_rebuild_v2/stage_manifest.json


In [28]:
BATCH3_OUTPUT_DIR = OUTPUT_ROOT / "batch3_unified_cases"
BATCH3_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BATCH3_UNIFIED_CASES_OUTPUT_PATH = BATCH3_OUTPUT_DIR / "batch3_unified_paired_cases.csv"
BATCH3_VALIDATION_OUTPUT_PATH = BATCH3_OUTPUT_DIR / "batch3_validation.csv"

BATCH3_MAIN_LOCAL_REGION_PRIORITY = [
    "mask_bbox_crop",
    "masked_region",
    "boundary_region",
    "content_region",
]

BATCH3_FULL_IMAGE_REGION_NAMES = {"full_image", "whole_image", "image"}

BATCH3_CASE_KEY_COLUMNS = BATCH2_CASE_KEY_COLUMNS
BATCH3_OPENCV_WIN_LABELS = {"opencv", "opencv_telea"}
BATCH3_LAMA_WIN_LABELS = {"lama"}
BATCH3_TIE_LABELS = {"tie"}

print("BATCH3_OUTPUT_DIR:", project_relative_path(BATCH3_OUTPUT_DIR))

BATCH3_OUTPUT_DIR: outputs/20_opencv_lama_comparison_inventory_rebuild_v2/batch3_unified_cases


In [29]:
def batch3_safe_slug(value) -> str:
    text = "" if pd.isna(value) else str(value).strip().lower()
    text = re.sub(r"[^a-z0-9]+", "_", text)
    return text.strip("_") or "unknown"


def batch3_first_nonblank(series: pd.Series, default: str = "") -> str:
    for value in series:
        if value is None:
            continue
        if isinstance(value, float) and np.isnan(value):
            continue
        text = str(value).strip()
        if text and text.lower() not in {"nan", "none", "null", "<na>"}:
            return text
    return default


def batch3_majority_winner(values: pd.Series) -> str:
    normalized = values.fillna("").astype(str).str.strip()
    lama_votes = int(normalized.isin(BATCH3_LAMA_WIN_LABELS).sum())
    opencv_votes = int(normalized.isin(BATCH3_OPENCV_WIN_LABELS).sum())
    tie_votes = int(normalized.isin(BATCH3_TIE_LABELS).sum())

    if lama_votes == 0 and opencv_votes == 0 and tie_votes == 0:
        return "no_votes"

    max_votes = max(lama_votes, opencv_votes, tie_votes)
    leaders = []
    if lama_votes == max_votes:
        leaders.append("lama")
    if opencv_votes == max_votes:
        leaders.append("opencv_telea")
    if tie_votes == max_votes:
        leaders.append("tie")

    return leaders[0] if len(leaders) == 1 else "mixed"


def batch3_join_unique_text(values: pd.Series) -> str:
    cleaned = []
    for value in values:
        text = "" if pd.isna(value) else str(value).strip()
        if text and text.lower() not in {"nan", "none", "null", "<na>"}:
            cleaned.append(text)
    return ", ".join(sorted(set(cleaned)))


def batch3_count_winner(values: pd.Series, labels: set[str]) -> int:
    return int(values.fillna("").astype(str).str.strip().isin(labels).sum())

In [30]:
available_regions = (
    batch2_metric_pairs_df["evaluation_region"]
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
    .tolist()
)

batch3_selected_main_local_regions = [
    region for region in BATCH3_MAIN_LOCAL_REGION_PRIORITY
    if region in available_regions
]

if not batch3_selected_main_local_regions:
    batch3_selected_main_local_regions = [
        region for region in available_regions
        if region not in BATCH3_FULL_IMAGE_REGION_NAMES
    ]

if not batch3_selected_main_local_regions:
    batch3_selected_main_local_regions = available_regions

batch3_metric_pairs_main_df = batch2_metric_pairs_df[
    batch2_metric_pairs_df["evaluation_region"].isin(batch3_selected_main_local_regions)
    & batch2_metric_pairs_df["metric_pair_status"].eq("paired")
].copy()

batch3_metric_pairs_main_df["metric_column_label"] = (
    batch3_metric_pairs_main_df["evaluation_region"].map(batch3_safe_slug)
    + "__"
    + batch3_metric_pairs_main_df["metric_name"].map(batch3_safe_slug)
    + "__"
    + batch3_metric_pairs_main_df["metric_value_type"].map(batch3_safe_slug)
)

print("Available regions:", available_regions)
print("Selected main local regions:", batch3_selected_main_local_regions)
print("Main-region paired metric rows:", len(batch3_metric_pairs_main_df))

Available regions: ['boundary_region', 'content_region', 'full_image', 'mask_bbox_crop', 'masked_region', 'outside_mask_region']
Selected main local regions: ['mask_bbox_crop', 'masked_region', 'boundary_region', 'content_region']
Main-region paired metric rows: 12020


In [31]:
batch3_metric_pairs_ranked_df = batch3_metric_pairs_main_df.copy()

batch3_metric_pairs_ranked_df["opencv_metric_rank"] = np.nan
batch3_metric_pairs_ranked_df["lama_metric_rank"] = np.nan

rank_group_columns = [
    "dataset_name",
    "mask_type",
    "evaluation_region",
    "metric_family",
    "metric_name",
    "metric_value_type",
]

for _, index_values in batch3_metric_pairs_ranked_df.groupby(rank_group_columns, dropna=False).groups.items():
    index_values = list(index_values)
    direction = batch3_metric_pairs_ranked_df.loc[index_values, "metric_direction"].dropna().astype(str)
    metric_direction = direction.iloc[0] if len(direction) else "higher_is_better"
    ascending = metric_direction == "lower_is_better"

    batch3_metric_pairs_ranked_df.loc[index_values, "opencv_metric_rank"] = (
        batch3_metric_pairs_ranked_df.loc[index_values, "opencv_metric_value"]
        .rank(method="min", ascending=ascending, na_option="bottom")
    )
    batch3_metric_pairs_ranked_df.loc[index_values, "lama_metric_rank"] = (
        batch3_metric_pairs_ranked_df.loc[index_values, "lama_metric_value"]
        .rank(method="min", ascending=ascending, na_option="bottom")
    )

batch3_metric_pairs_ranked_df["rank_change_lama_minus_opencv"] = (
    batch3_metric_pairs_ranked_df["lama_metric_rank"]
    - batch3_metric_pairs_ranked_df["opencv_metric_rank"]
)

batch3_metric_pairs_ranked_df["lama_rank_improved"] = (
    batch3_metric_pairs_ranked_df["rank_change_lama_minus_opencv"] < 0
)
batch3_metric_pairs_ranked_df["opencv_rank_improved"] = (
    batch3_metric_pairs_ranked_df["rank_change_lama_minus_opencv"] > 0
)
batch3_metric_pairs_ranked_df["rank_unchanged"] = (
    batch3_metric_pairs_ranked_df["rank_change_lama_minus_opencv"] == 0
)

display(batch3_metric_pairs_ranked_df.head())

,case_id,dataset_name,painting_id,mask_id,mask_type,evaluation_region,opencv_source_id,opencv_source_relative_path,metric_family,opencv_metric_column,metric_name,metric_value_type,metric_direction,opencv_metric_value,opencv_metric_value_is_numeric,opencv_status,opencv_issue,opencv_restored_path,lama_source_id,lama_source_relative_path,lama_metric_column,lama_metric_value,lama_metric_value_is_numeric,lama_status,lama_issue,lama_restored_path,metric_pair_status,lama_minus_opencv,lama_advantage,absolute_metric_delta,metric_winner,metric_column_label,opencv_metric_rank,lama_metric_rank,rank_change_lama_minus_opencv,lama_rank_improved,opencv_rank_improved,rank_unchanged
0,canonical__p001_loss_large,canonical,p001,p001_loss_large,loss_large,boundary_region,opencv_classical_metrics,data/processed/metrics/metrics_opencv_telea_classical.csv,classical_metrics,mse_improvement,mse,improvement,higher_is_better,53.382518,True,ok,NaN,NaN,lama_classical_metrics,outputs/metrics/classical_metrics_lama.csv,mse_improvement,25396.024811,True,ok,NaN,D:\Masters\FH\Thesis\painting-restoration-eval\data\processed\restored\lama\canonical\canonical__p001_loss_large_restored_lama.png,paired,25342.642293,25342.642293,25342.642293,lama,boundary_region__mse__improvement,1.0,1.0,0.0,False,False,True
1,canonical__p001_loss_large,canonical,p001,p001_loss_large,loss_large,boundary_region,opencv_classical_metrics,data/processed/metrics/metrics_opencv_telea_classical.csv,classical_metrics,restored_mse,mse,restored,lower_is_better,6.173440,True,ok,NaN,NaN,lama_classical_metrics,outputs/metrics/classical_metrics_lama.csv,restored_mse,7.777923,True,ok,NaN,D:\Masters\FH\Thesis\painting-restoration-eval\data\processed\restored\lama\canonical\canonical__p001_loss_large_restored_lama.png,paired,1.604483,-1.604483,1.604483,opencv_telea,boundary_region__mse__restored,3.0,3.0,0.0,False,False,True
2,canonical__p001_loss_large,canonical,p001,p001_loss_large,loss_large,boundary_region,opencv_classical_metrics,data/processed/metrics/metrics_opencv_telea_classical.csv,classical_metrics,psnr_improvement,psnr,improvement,higher_is_better,9.843980,True,ok,NaN,NaN,lama_classical_metrics,outputs/metrics/classical_metrics_lama.csv,psnr_improvement,35.140351,True,ok,NaN,D:\Masters\FH\Thesis\painting-restoration-eval\data\processed\restored\lama\canonical\canonical__p001_loss_large_restored_lama.png,paired,25.296371,25.296371,25.296371,lama,boundary_region__psnr__improvement,3.0,2.0,-1.0,True,False,False
3,canonical__p001_loss_large,canonical,p001,p001_loss_large,loss_large,boundary_region,opencv_classical_metrics,data/processed/metrics/metrics_opencv_telea_classical.csv,classical_metrics,restored_psnr,psnr,restored,higher_is_better,40.225531,True,ok,NaN,NaN,lama_classical_metrics,outputs/metrics/classical_metrics_lama.csv,restored_psnr,39.222167,True,ok,NaN,D:\Masters\FH\Thesis\painting-restoration-eval\data\processed\restored\lama\canonical\canonical__p001_loss_large_restored_lama.png,paired,-1.003364,-1.003364,1.003364,opencv_telea,boundary_region__psnr__restored,3.0,3.0,0.0,False,False,True
6,canonical__p001_loss_large,canonical,p001,p001_loss_large,loss_large,content_region,opencv_classical_metrics,data/processed/metrics/metrics_opencv_telea_classical.csv,classical_metrics,mse_improvement,mse,improvement,higher_is_better,9.221621,True,ok,NaN,NaN,lama_classical_metrics,outputs/metrics/classical_metrics_lama.csv,mse_improvement,6556.833107,True,ok,NaN,D:\Masters\FH\Thesis\painting-restoration-eval\data\processed\restored\lama\canonical\canonical__p001_loss_large_restored_lama.png,paired,6547.611486,6547.611486,6547.611486,lama,content_region__mse__improvement,2.0,2.0,0.0,False,False,True


In [32]:
def batch3_wide_first(
    df: pd.DataFrame,
    value_column: str,
    output_prefix: str,
) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame(columns=BATCH3_CASE_KEY_COLUMNS)

    wide_df = (
        df.groupby([*BATCH3_CASE_KEY_COLUMNS, "metric_column_label"], dropna=False)[value_column]
        .first()
        .unstack("metric_column_label")
        .reset_index()
    )

    rename_map = {
        column: f"{output_prefix}__{column}"
        for column in wide_df.columns
        if column not in BATCH3_CASE_KEY_COLUMNS
    }

    return wide_df.rename(columns=rename_map)


batch3_winner_wide_df = batch3_wide_first(
    batch3_metric_pairs_ranked_df,
    "metric_winner",
    "winner",
)

batch3_rank_change_wide_df = batch3_wide_first(
    batch3_metric_pairs_ranked_df,
    "rank_change_lama_minus_opencv",
    "rank_change_lama_minus_opencv",
)

batch3_lama_advantage_wide_df = batch3_wide_first(
    batch3_metric_pairs_ranked_df,
    "lama_advantage",
    "lama_advantage",
)

In [33]:
case_vote_rows = []

for key, group in batch3_metric_pairs_ranked_df.groupby(BATCH3_CASE_KEY_COLUMNS, dropna=False, sort=False):
    if not isinstance(key, tuple):
        key = (key,)

    record = dict(zip(BATCH3_CASE_KEY_COLUMNS, key))
    winners = group["metric_winner"]

    restored_group = group[group["metric_value_type"].eq("restored")]
    improvement_group = group[group["metric_value_type"].eq("improvement")]

    record["main_local_regions_used"] = batch3_join_unique_text(group["evaluation_region"])
    record["main_local_metric_pair_rows"] = int(len(group))

    record["lama_metric_votes"] = batch3_count_winner(winners, BATCH3_LAMA_WIN_LABELS)
    record["opencv_metric_votes"] = batch3_count_winner(winners, BATCH3_OPENCV_WIN_LABELS)
    record["tie_metric_votes"] = batch3_count_winner(winners, BATCH3_TIE_LABELS)
    record["metric_vote_margin_lama_minus_opencv"] = (
        record["lama_metric_votes"] - record["opencv_metric_votes"]
    )
    record["metric_majority_winner"] = batch3_majority_winner(winners)

    record["restored_metric_majority_winner"] = batch3_majority_winner(restored_group["metric_winner"])
    record["improvement_metric_majority_winner"] = batch3_majority_winner(improvement_group["metric_winner"])

    record["metric_families_used"] = batch3_join_unique_text(group["metric_family"])
    record["metric_names_used"] = batch3_join_unique_text(group["metric_name"])

    rank_change = pd.to_numeric(group["rank_change_lama_minus_opencv"], errors="coerce")
    record["rank_change_mean_lama_minus_opencv"] = rank_change.mean()
    record["rank_change_median_lama_minus_opencv"] = rank_change.median()
    record["rank_change_best_for_lama"] = rank_change.min()
    record["rank_change_worst_for_lama"] = rank_change.max()
    record["rank_lama_improved_metric_count"] = int((rank_change < 0).sum())
    record["rank_opencv_improved_metric_count"] = int((rank_change > 0).sum())
    record["rank_unchanged_metric_count"] = int((rank_change == 0).sum())

    family_winners = (
        group.groupby("metric_family", dropna=False)["metric_winner"]
        .apply(batch3_majority_winner)
        .tolist()
    )
    region_winners = (
        group.groupby("evaluation_region", dropna=False)["metric_winner"]
        .apply(batch3_majority_winner)
        .tolist()
    )

    record["flag_mixed_metric_winners"] = bool(
        record["lama_metric_votes"] > 0 and record["opencv_metric_votes"] > 0
    )
    record["flag_no_clear_metric_winner"] = record["metric_majority_winner"] in {"mixed", "tie", "no_votes"}
    record["flag_family_winner_disagreement"] = len(set(family_winners)) > 1
    record["flag_region_winner_disagreement"] = len(set(region_winners)) > 1

    record["flag_restored_vs_improvement_winner_disagreement"] = (
        record["restored_metric_majority_winner"] not in {"no_votes"}
        and record["improvement_metric_majority_winner"] not in {"no_votes"}
        and record["restored_metric_majority_winner"] != record["improvement_metric_majority_winner"]
    )

    record["flag_rank_and_vote_disagreement"] = bool(
        (
            record["metric_majority_winner"] == "lama"
            and pd.notna(record["rank_change_mean_lama_minus_opencv"])
            and record["rank_change_mean_lama_minus_opencv"] > 0
        )
        or (
            record["metric_majority_winner"] == "opencv_telea"
            and pd.notna(record["rank_change_mean_lama_minus_opencv"])
            and record["rank_change_mean_lama_minus_opencv"] < 0
        )
    )

    review_reasons = []
    for flag_column in [
        "flag_mixed_metric_winners",
        "flag_no_clear_metric_winner",
        "flag_family_winner_disagreement",
        "flag_region_winner_disagreement",
        "flag_restored_vs_improvement_winner_disagreement",
        "flag_rank_and_vote_disagreement",
    ]:
        if record[flag_column]:
            review_reasons.append(flag_column)

    record["needs_visual_review"] = bool(review_reasons)
    record["visual_review_reasons"] = ", ".join(review_reasons)

    case_vote_rows.append(record)

batch3_case_votes_df = pd.DataFrame(case_vote_rows)

display(batch3_case_votes_df.head())

,case_id,dataset_name,painting_id,mask_id,mask_type,main_local_regions_used,main_local_metric_pair_rows,lama_metric_votes,opencv_metric_votes,tie_metric_votes,metric_vote_margin_lama_minus_opencv,metric_majority_winner,restored_metric_majority_winner,improvement_metric_majority_winner,metric_families_used,metric_names_used,rank_change_mean_lama_minus_opencv,rank_change_median_lama_minus_opencv,rank_change_best_for_lama,rank_change_worst_for_lama,rank_lama_improved_metric_count,rank_opencv_improved_metric_count,rank_unchanged_metric_count,flag_mixed_metric_winners,flag_no_clear_metric_winner,flag_family_winner_disagreement,flag_region_winner_disagreement,flag_restored_vs_improvement_winner_disagreement,flag_rank_and_vote_disagreement,needs_visual_review,visual_review_reasons
0,canonical__p001_loss_large,canonical,p001,p001_loss_large,loss_large,"boundary_region, content_region, mask_bbox_crop, masked_region",32,20,12,0,8,lama,opencv_telea,lama,"classical_metrics, feature_metrics, lpips_metrics","clip, dinov2, lpips, mse, psnr, ssim",4.18750,3.0,-25.0,24.0,5,19,8,True,False,True,True,True,True,True,"flag_mixed_metric_winners, flag_family_winner_disagreement, flag_region_winner_disagreement, flag_restored_vs_improvement_winner_disagreement, flag_rank_and_vote_disagreement"
1,canonical__p001_loss_small,canonical,p001,p001_loss_small,loss_small,"boundary_region, content_region, mask_bbox_crop, masked_region",32,20,12,0,8,lama,opencv_telea,lama,"classical_metrics, feature_metrics, lpips_metrics","clip, dinov2, lpips, mse, psnr, ssim",2.15625,2.0,-3.0,16.0,5,22,5,True,False,True,True,True,True,True,"flag_mixed_metric_winners, flag_family_winner_disagreement, flag_region_winner_disagreement, flag_restored_vs_improvement_winner_disagreement, flag_rank_and_vote_disagreement"
2,canonical__p001_mixed_damage,canonical,p001,p001_mixed_damage,mixed_damage,"boundary_region, content_region, mask_bbox_crop, masked_region",32,24,8,0,16,lama,mixed,lama,"classical_metrics, feature_metrics, lpips_metrics","clip, dinov2, lpips, mse, psnr, ssim",6.50000,2.0,-5.0,40.0,6,21,5,True,False,False,True,True,True,True,"flag_mixed_metric_winners, flag_region_winner_disagreement, flag_restored_vs_improvement_winner_disagreement, flag_rank_and_vote_disagreement"
3,canonical__p001_scratch_thin,canonical,p001,p001_scratch_thin,scratch_thin,"boundary_region, content_region, mask_bbox_crop, masked_region",32,8,24,0,-16,opencv_telea,opencv_telea,mixed,"classical_metrics, feature_metrics, lpips_metrics","clip, dinov2, lpips, mse, psnr, ssim",3.06250,1.0,-1.0,23.0,1,18,13,True,False,False,True,True,False,True,"flag_mixed_metric_winners, flag_region_winner_disagreement, flag_restored_vs_improvement_winner_disagreement"
4,canonical__p001_zero_control,canonical,p001,p001_zero_control,zero_control,content_region,12,0,0,11,0,tie,tie,tie,"classical_metrics, feature_metrics, lpips_metrics","clip, dinov2, lpips, mse, psnr, ssim",0.00000,0.0,0.0,0.0,0,0,12,False,True,False,False,False,False,True,flag_no_clear_metric_winner


In [34]:
batch3_unified_cases_df = batch2_restoration_pairs_df.copy()

for column in BATCH3_CASE_KEY_COLUMNS:
    batch3_unified_cases_df[column] = batch3_unified_cases_df[column].fillna("").astype(str).str.strip()

batch3_unified_cases_df = batch3_unified_cases_df.merge(
    batch3_case_votes_df,
    on=BATCH3_CASE_KEY_COLUMNS,
    how="outer",
)

for wide_df in [
    batch3_winner_wide_df,
    batch3_rank_change_wide_df,
    batch3_lama_advantage_wide_df,
]:
    batch3_unified_cases_df = batch3_unified_cases_df.merge(
        wide_df,
        on=BATCH3_CASE_KEY_COLUMNS,
        how="left",
    )

numeric_zero_columns = [
    "main_local_metric_pair_rows",
    "lama_metric_votes",
    "opencv_metric_votes",
    "tie_metric_votes",
    "metric_vote_margin_lama_minus_opencv",
    "rank_lama_improved_metric_count",
    "rank_opencv_improved_metric_count",
    "rank_unchanged_metric_count",
]

for column in numeric_zero_columns:
    batch3_unified_cases_df[column] = pd.to_numeric(
        batch3_unified_cases_df.get(column, 0),
        errors="coerce",
    ).fillna(0).astype(int)

for column in [
    "metric_majority_winner",
    "restored_metric_majority_winner",
    "improvement_metric_majority_winner",
]:
    batch3_unified_cases_df[column] = (
        batch3_unified_cases_df.get(column, "no_votes")
        .fillna("no_votes")
        .astype(str)
    )

for column in [
    "flag_mixed_metric_winners",
    "flag_no_clear_metric_winner",
    "flag_family_winner_disagreement",
    "flag_region_winner_disagreement",
    "flag_restored_vs_improvement_winner_disagreement",
    "flag_rank_and_vote_disagreement",
    "needs_visual_review",
]:
    batch3_unified_cases_df[column] = (
        batch3_unified_cases_df.get(column, False)
        .fillna(False)
        .astype(bool)
    )

batch3_unified_cases_df["has_main_local_metric_pairs"] = (
    batch3_unified_cases_df["main_local_metric_pair_rows"] > 0
)

batch3_unified_cases_df["case_presence_status"] = np.select(
    [
        batch3_unified_cases_df["restoration_pair_status"].eq("paired")
        & batch3_unified_cases_df["has_main_local_metric_pairs"],
        batch3_unified_cases_df["restoration_pair_status"].eq("paired")
        & ~batch3_unified_cases_df["has_main_local_metric_pairs"],
        batch3_unified_cases_df["restoration_pair_status"].isna()
        & batch3_unified_cases_df["has_main_local_metric_pairs"],
    ],
    [
        "paired_restoration_with_main_local_metrics",
        "paired_restoration_missing_main_local_metrics",
        "metrics_only_no_restoration_pair_row",
    ],
    default=batch3_unified_cases_df["restoration_pair_status"].fillna("unknown"),
)

batch3_unified_cases_df["pair_case_key"] = batch3_unified_cases_df[BATCH3_CASE_KEY_COLUMNS].apply(
    lambda row: "|".join(f"{column}={str(row[column]).strip()}" for column in BATCH3_CASE_KEY_COLUMNS),
    axis=1,
)

batch3_unified_cases_df = batch3_unified_cases_df.sort_values(BATCH3_CASE_KEY_COLUMNS).reset_index(drop=True)

display(batch3_unified_cases_df.head())
display(batch3_unified_cases_df["case_presence_status"].value_counts(dropna=False).reset_index(name="rows"))

,case_id,dataset_name,painting_id,mask_id,mask_type,opencv_clean_path,opencv_damaged_path,opencv_mask_path,opencv_restored_path,opencv_runtime_seconds,opencv_output_written,opencv_status,opencv_issue,opencv_source_rows,lama_mask_area_pixels,lama_clean_path,lama_damaged_path,lama_mask_path,lama_restored_path,lama_runtime_seconds,lama_output_written,lama_status,lama_issue,lama_iopaint_returncode,lama_source_rows,restoration_pair_status,main_local_regions_used,main_local_metric_pair_rows,lama_metric_votes,opencv_metric_votes,tie_metric_votes,metric_vote_margin_lama_minus_opencv,metric_majority_winner,restored_metric_majority_winner,improvement_metric_majority_winner,metric_families_used,metric_names_used,rank_change_mean_lama_minus_opencv,rank_change_median_lama_minus_opencv,rank_change_best_for_lama,rank_change_worst_for_lama,rank_lama_improved_metric_count,rank_opencv_improved_metric_count,rank_unchanged_metric_count,flag_mixed_metric_winners,flag_no_clear_metric_winner,flag_family_winner_disagreement,flag_region_winner_disagreement,flag_restored_vs_improvement_winner_disagreement,flag_rank_and_vote_disagreement,needs_visual_review,visual_review_reasons,winner__boundary_region__mse__improvement,winner__boundary_region__mse__restored,winner__boundary_region__psnr__improvement,winner__boundary_region__psnr__restored,winner__content_region__clip__improvement,winner__content_region__clip__restored,winner__content_region__dinov2__improvement,winner__content_region__dinov2__restored,winner__content_region__lpips__improvement,winner__content_region__lpips__restored,winner__content_region__mse__improvement,winner__content_region__mse__restored,winner__content_region__psnr__improvement,winner__content_region__psnr__restored,winner__content_region__ssim__improvement,winner__content_region__ssim__restored,winner__mask_bbox_crop__clip__improvement,winner__mask_bbox_crop__clip__restored,winner__mask_bbox_crop__dinov2__improvement,winner__mask_bbox_crop__dinov2__restored,winner__mask_bbox_crop__lpips__improvement,winner__mask_bbox_crop__lpips__restored,winner__mask_bbox_crop__mse__improvement,winner__mask_bbox_crop__mse__restored,winner__mask_bbox_crop__psnr__improvement,winner__mask_bbox_crop__psnr__restored,winner__mask_bbox_crop__ssim__improvement,winner__mask_bbox_crop__ssim__restored,winner__masked_region__mse__improvement,winner__masked_region__mse__restored,winner__masked_region__psnr__improvement,winner__masked_region__psnr__restored,rank_change_lama_minus_opencv__boundary_region__mse__improvement,rank_change_lama_minus_opencv__boundary_region__mse__restored,rank_change_lama_minus_opencv__boundary_region__psnr__improvement,rank_change_lama_minus_opencv__boundary_region__psnr__restored,rank_change_lama_minus_opencv__content_region__clip__improvement,rank_change_lama_minus_opencv__content_region__clip__restored,rank_change_lama_minus_opencv__content_region__dinov2__improvement,rank_change_lama_minus_opencv__content_region__dinov2__restored,rank_change_lama_minus_opencv__content_region__lpips__improvement,rank_change_lama_minus_opencv__content_region__lpips__restored,rank_change_lama_minus_opencv__content_region__mse__improvement,rank_change_lama_minus_opencv__content_region__mse__restored,rank_change_lama_minus_opencv__content_region__psnr__improvement,rank_change_lama_minus_opencv__content_region__psnr__restored,rank_change_lama_minus_opencv__content_region__ssim__improvement,rank_change_lama_minus_opencv__content_region__ssim__restored,rank_change_lama_minus_opencv__mask_bbox_crop__clip__improvement,rank_change_lama_minus_opencv__mask_bbox_crop__clip__restored,rank_change_lama_minus_opencv__mask_bbox_crop__dinov2__improvement,rank_change_lama_minus_opencv__mask_bbox_crop__dinov2__restored,rank_change_lama_minus_opencv__mask_bbox_crop__lpips__improvement,rank_change_lama_minus_opencv__mask_bbox_crop__lpips__restored,rank_change_lama_minus_opencv__mask_bbox_crop__mse__improvement,rank_change_lama_minus_opencv__mask_bbox_crop__mse__rest

,case_presence_status,rows
0,paired_restoration_with_main_local_metrics,410


In [35]:
winner_columns = [
    column for column in batch3_unified_cases_df.columns
    if column.startswith("winner__")
]

rank_change_columns = [
    column for column in batch3_unified_cases_df.columns
    if column.startswith("rank_change_lama_minus_opencv__")
]

duplicate_case_key_count = int(batch3_unified_cases_df["pair_case_key"].duplicated().sum())

batch3_check_rows = [
    build_check(
        "batch3_selected_main_local_regions_non_empty",
        batch3_selected_main_local_regions,
        "at least one selected region",
        len(batch3_selected_main_local_regions) > 0,
        "No main local evaluation regions were available.",
    ),
    build_check(
        "batch3_unified_cases_have_rows",
        len(batch3_unified_cases_df),
        "> 0",
        len(batch3_unified_cases_df) > 0,
        "Unified paired-case table has no rows.",
    ),
    build_check(
        "batch3_pair_case_key_unique",
        duplicate_case_key_count,
        0,
        duplicate_case_key_count == 0,
        "Unified paired-case table has duplicate pair_case_key values.",
    ),
    build_check(
        "batch3_winner_columns_created",
        winner_columns,
        "one or more winner columns",
        len(winner_columns) > 0,
        "No wide winner columns were created.",
    ),
    build_check(
        "batch3_rank_change_columns_created",
        rank_change_columns,
        "one or more rank-change columns",
        len(rank_change_columns) > 0,
        "No wide rank-change columns were created.",
    ),
    build_check(
        "batch3_vote_counts_consistent",
        int(
            (
                batch3_unified_cases_df["lama_metric_votes"]
                + batch3_unified_cases_df["opencv_metric_votes"]
                + batch3_unified_cases_df["tie_metric_votes"]
                <= batch3_unified_cases_df["main_local_metric_pair_rows"]
            ).sum()
        ),
        len(batch3_unified_cases_df),
        bool(
            (
                batch3_unified_cases_df["lama_metric_votes"]
                + batch3_unified_cases_df["opencv_metric_votes"]
                + batch3_unified_cases_df["tie_metric_votes"]
                <= batch3_unified_cases_df["main_local_metric_pair_rows"]
            ).all()
        ),
        "Vote counts exceed main local metric pair row counts.",
    ),
]

batch3_validation_df = pd.DataFrame(batch3_check_rows)

display(batch3_validation_df)

if not batch3_validation_df["passed"].astype(bool).all():
    failed_batch3_checks_df = batch3_validation_df[
        ~batch3_validation_df["passed"].astype(bool)
    ].copy()
    display(failed_batch3_checks_df)
    raise RuntimeError("Notebook 20 Batch 3 validation failed. Inspect unified case table.")

,check_name,observed,expected,passed,failure_message
0,batch3_selected_main_local_regions_non_empty,"[mask_bbox_crop, masked_region, boundary_region, content_region]",at least one selected region,True,
1,batch3_unified_cases_have_rows,410,> 0,True,
2,batch3_pair_case_key_unique,0,0,True,
3,batch3_winner_columns_created,"[winner__boundary_region__mse__improvement, winner__boundary_region__mse__restored, winner__boundary_region__psnr__improvement, winner__boundary_region__psnr__restored, winner_...",one or more winner columns,True,
4,batch3_rank_change_columns_created,"[rank_change_lama_minus_opencv__boundary_region__mse__improvement, rank_change_lama_minus_opencv__boundary_region__mse__restored, rank_change_lama_minus_opencv__boundary_region...",one or more rank-change columns,True,
5,batch3_vote_counts_consistent,410,410,True,


In [36]:
batch3_unified_cases_df.to_csv(BATCH3_UNIFIED_CASES_OUTPUT_PATH, index=False)

batch3_validation_export_df = batch3_validation_df.copy()
for column in ["observed", "expected"]:
    batch3_validation_export_df[column] = batch3_validation_export_df[column].map(json_for_csv)

batch3_validation_export_df.to_csv(BATCH3_VALIDATION_OUTPUT_PATH, index=False)

stage_manifest = read_json(STAGE_MANIFEST_JSON_PATH)

stage_manifest.update(
    {
        "status": "batch_3_unified_paired_cases_complete",
        "updated_at_utc": datetime.now(timezone.utc).isoformat(),
    }
)

stage_manifest.setdefault("outputs", {})
stage_manifest["outputs"].update(
    {
        "batch3_unified_cases": project_relative_path(BATCH3_UNIFIED_CASES_OUTPUT_PATH),
        "batch3_validation": project_relative_path(BATCH3_VALIDATION_OUTPUT_PATH),
    }
)

stage_manifest.setdefault("row_counts", {})
stage_manifest["row_counts"].update(
    {
        "batch3_unified_case_rows": int(len(batch3_unified_cases_df)),
        "batch3_cases_with_main_local_metric_pairs": int(
            batch3_unified_cases_df["has_main_local_metric_pairs"].sum()
        ),
        "batch3_winner_columns": int(len(winner_columns)),
        "batch3_rank_change_columns": int(len(rank_change_columns)),
        "batch3_validation_checks": int(len(batch3_validation_df)),
    }
)

stage_manifest["batch3_main_local_regions"] = batch3_selected_main_local_regions
stage_manifest["batch3_rank_change_interpretation"] = (
    "rank_change_lama_minus_opencv < 0 means LaMa ranks better; "
    "> 0 means OpenCV Telea ranks better."
)

stage_manifest["next_batch"] = (
    "Batch 4: build compact grouped summaries and conservative review flags from "
    "the unified paired-case table."
)

save_json(STAGE_MANIFEST_JSON_PATH, stage_manifest)

print("Batch 3 complete.")
print("Saved:", project_relative_path(BATCH3_UNIFIED_CASES_OUTPUT_PATH))
print("Saved:", project_relative_path(BATCH3_VALIDATION_OUTPUT_PATH))
print("Updated manifest:", project_relative_path(STAGE_MANIFEST_JSON_PATH))

Batch 3 complete.
Saved: outputs/20_opencv_lama_comparison_inventory_rebuild_v2/batch3_unified_cases/batch3_unified_paired_cases.csv
Saved: outputs/20_opencv_lama_comparison_inventory_rebuild_v2/batch3_unified_cases/batch3_validation.csv
Updated manifest: outputs/20_opencv_lama_comparison_inventory_rebuild_v2/stage_manifest.json


In [37]:
import re

BATCH4_OUTPUT_DIR = OUTPUT_ROOT / "batch4_compact_summaries"
BATCH4_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BATCH4_COMPACT_SUMMARIES_OUTPUT_PATH = BATCH4_OUTPUT_DIR / "batch4_compact_group_summaries.csv"
BATCH4_RUNTIME_SUMMARY_OUTPUT_PATH = BATCH4_OUTPUT_DIR / "batch4_runtime_summary.csv"
BATCH4_FAILURE_PATTERNS_OUTPUT_PATH = BATCH4_OUTPUT_DIR / "batch4_deterministic_failure_patterns.csv"
BATCH4_VALIDATION_OUTPUT_PATH = BATCH4_OUTPUT_DIR / "batch4_validation.csv"

BATCH4_CASE_KEY_COLUMNS = BATCH3_CASE_KEY_COLUMNS

BATCH4_SUMMARY_DIMENSION_COLUMNS = [
    "summary_mask_type",
    "summary_category",
    "summary_style_or_period",
    "summary_dataset_name",
    "summary_damage_family",
]

print("BATCH4_OUTPUT_DIR:", project_relative_path(BATCH4_OUTPUT_DIR))

BATCH4_OUTPUT_DIR: outputs/20_opencv_lama_comparison_inventory_rebuild_v2/batch4_compact_summaries


In [38]:
def batch4_text(value) -> str:
    if value is None:
        return ""
    if isinstance(value, float) and np.isnan(value):
        return ""
    text = str(value).strip()
    return "" if text.lower() in {"nan", "none", "null", "<na>"} else text


def batch4_slug(value) -> str:
    text = batch4_text(value).lower()
    text = re.sub(r"[^a-z0-9]+", "_", text)
    return text.strip("_") or "unknown"


def batch4_series(df: pd.DataFrame, column: str, default="") -> pd.Series:
    if column in df.columns:
        return df[column]
    return pd.Series([default] * len(df), index=df.index)


def batch4_first_nonblank(series: pd.Series, default: str = "") -> str:
    for value in series:
        text = batch4_text(value)
        if text:
            return text
    return default


def batch4_coalesce_columns(df: pd.DataFrame, columns: list[str], default: str) -> pd.Series:
    result = pd.Series([default] * len(df), index=df.index, dtype="object")
    for column in columns:
        if column not in df.columns:
            continue
        candidate = df[column].map(batch4_text)
        result = result.mask(result.eq(default) & candidate.ne(""), candidate)
    return result.fillna(default).astype(str)


def batch4_boolish(value):
    text = batch4_text(value).lower()
    if text in {"true", "1", "yes", "y", "written", "success", "ok"}:
        return True
    if text in {"false", "0", "no", "n", "failed", "missing"}:
        return False
    return np.nan


def batch4_top_counts(series: pd.Series, limit: int = 4) -> str:
    cleaned = series.map(batch4_text)
    cleaned = cleaned[cleaned.ne("")]
    if cleaned.empty:
        return ""
    counts = cleaned.value_counts(dropna=False).head(limit)
    return "; ".join(f"{label}={int(count)}" for label, count in counts.items())

In [39]:
def batch4_metadata_lookup_from_sources() -> pd.DataFrame:
    if "SOURCE_TABLES" not in globals():
        return pd.DataFrame(columns=BATCH4_CASE_KEY_COLUMNS)

    category_candidates = ["category", "painting_category", "object_category", "genre"]
    style_candidates = ["style_or_period", "style", "period", "art_period", "movement"]
    damage_candidates = ["damage_family", "dataset_damage_family", "damage_size_category"]

    frames = []

    for source_id, source_df in SOURCE_TABLES.items():
        if not isinstance(source_df, pd.DataFrame):
            continue
        if any(column not in source_df.columns for column in BATCH4_CASE_KEY_COLUMNS):
            continue

        tmp = source_df[BATCH4_CASE_KEY_COLUMNS].copy()
        for column in BATCH4_CASE_KEY_COLUMNS:
            tmp[column] = tmp[column].map(batch4_text)

        tmp["metadata_category"] = batch4_coalesce_columns(source_df, category_candidates, "")
        tmp["metadata_style_or_period"] = batch4_coalesce_columns(source_df, style_candidates, "")
        tmp["metadata_damage_family"] = batch4_coalesce_columns(source_df, damage_candidates, "")
        tmp["metadata_source_id"] = "__".join(source_id) if isinstance(source_id, tuple) else str(source_id)
        frames.append(tmp)

    if not frames:
        return pd.DataFrame(columns=BATCH4_CASE_KEY_COLUMNS)

    metadata_df = pd.concat(frames, ignore_index=True)
    metadata_df = (
        metadata_df.groupby(BATCH4_CASE_KEY_COLUMNS, dropna=False)
        .agg(
            metadata_category=("metadata_category", batch4_first_nonblank),
            metadata_style_or_period=("metadata_style_or_period", batch4_first_nonblank),
            metadata_damage_family=("metadata_damage_family", batch4_first_nonblank),
            metadata_sources=("metadata_source_id", batch4_top_counts),
        )
        .reset_index()
    )

    return metadata_df


batch4_source_df = batch3_unified_cases_df.copy()

for column in BATCH4_CASE_KEY_COLUMNS:
    batch4_source_df[column] = batch4_source_df[column].map(batch4_text)

batch4_metadata_lookup_df = batch4_metadata_lookup_from_sources()

if not batch4_metadata_lookup_df.empty:
    batch4_source_df = batch4_source_df.merge(
        batch4_metadata_lookup_df,
        on=BATCH4_CASE_KEY_COLUMNS,
        how="left",
    )

display(batch4_metadata_lookup_df.head())

,case_id,dataset_name,painting_id,mask_id,mask_type,metadata_category,metadata_style_or_period,metadata_damage_family,metadata_sources
0,canonical__p001_loss_large,canonical,p001,p001_loss_large,loss_large,portrait_figure,Baroque,,opencv_classical_metrics=6; lama_classical_metrics=6; opencv_lpips_metrics=3; lama_lpips_metrics=3
1,canonical__p001_loss_small,canonical,p001,p001_loss_small,loss_small,portrait_figure,Baroque,,opencv_classical_metrics=6; lama_classical_metrics=6; opencv_lpips_metrics=3; lama_lpips_metrics=3
2,canonical__p001_mixed_damage,canonical,p001,p001_mixed_damage,mixed_damage,portrait_figure,Baroque,,opencv_classical_metrics=6; lama_classical_metrics=6; opencv_lpips_metrics=3; lama_lpips_metrics=3
3,canonical__p001_scratch_thin,canonical,p001,p001_scratch_thin,scratch_thin,portrait_figure,Baroque,,opencv_classical_metrics=6; lama_classical_metrics=6; opencv_lpips_metrics=3; lama_lpips_metrics=3
4,canonical__p001_zero_control,canonical,p001,p001_zero_control,zero_control,portrait_figure,Baroque,,opencv_classical_metrics=3; lama_classical_metrics=2; opencv_lpips_metrics=2; lama_lpips_metrics=2


In [40]:
def batch4_damage_family_from_row(row: pd.Series) -> str:
    for column in [
        "metadata_damage_family",
        "damage_family",
        "dataset_damage_family",
        "opencv_damage_size_category",
        "lama_damage_size_category",
        "damage_size_category",
    ]:
        value = batch4_text(row.get(column))
        if value:
            return value

    haystack = " ".join(
        batch4_text(row.get(column)).lower()
        for column in ["case_id", "dataset_name", "mask_id", "mask_type"]
    )

    if "zero" in haystack and "control" in haystack:
        return "zero_control"
    if "damage_size" in haystack or "damage-size" in haystack:
        return "damage_size"
    if "mask_robust" in haystack or "mask-robust" in haystack:
        return "mask_robustness"
    if "synthetic" in haystack or "degradation" in haystack:
        return "synthetic_degradation"
    if "canonical" in haystack:
        return "canonical"

    return "unknown_damage_family"


batch4_analysis_df = batch4_source_df.copy()

batch4_analysis_df["summary_mask_type"] = batch4_coalesce_columns(
    batch4_analysis_df,
    ["mask_type", "opencv_mask_type", "lama_mask_type"],
    "unknown_mask",
)

batch4_analysis_df["summary_category"] = batch4_coalesce_columns(
    batch4_analysis_df,
    ["metadata_category", "category", "opencv_category", "lama_category", "painting_category", "genre"],
    "unknown_category",
)

batch4_analysis_df["summary_style_or_period"] = batch4_coalesce_columns(
    batch4_analysis_df,
    ["metadata_style_or_period", "style_or_period", "style", "period", "opencv_style", "lama_style"],
    "unknown_style_or_period",
)

batch4_analysis_df["summary_dataset_name"] = batch4_coalesce_columns(
    batch4_analysis_df,
    ["dataset_name", "opencv_dataset_name", "lama_dataset_name"],
    "unknown_dataset",
)

batch4_analysis_df["summary_damage_family"] = batch4_analysis_df.apply(batch4_damage_family_from_row, axis=1)

for prefix in ["opencv", "lama"]:
    runtime_candidates = [
        column for column in batch4_analysis_df.columns
        if column.startswith(prefix) and "runtime" in column and "seconds" in column
    ]
    runtime_value = pd.Series([np.nan] * len(batch4_analysis_df), index=batch4_analysis_df.index)
    for column in runtime_candidates:
        runtime_value = runtime_value.fillna(pd.to_numeric(batch4_analysis_df[column], errors="coerce"))
    batch4_analysis_df[f"{prefix}_runtime_seconds_batch4"] = runtime_value

display(batch4_analysis_df[BATCH4_CASE_KEY_COLUMNS + BATCH4_SUMMARY_DIMENSION_COLUMNS].head())

,case_id,dataset_name,painting_id,mask_id,mask_type,summary_mask_type,summary_category,summary_style_or_period,summary_dataset_name,summary_damage_family
0,canonical__p001_loss_large,canonical,p001,p001_loss_large,loss_large,loss_large,portrait_figure,Baroque,canonical,canonical
1,canonical__p001_loss_small,canonical,p001,p001_loss_small,loss_small,loss_small,portrait_figure,Baroque,canonical,canonical
2,canonical__p001_mixed_damage,canonical,p001,p001_mixed_damage,mixed_damage,mixed_damage,portrait_figure,Baroque,canonical,canonical
3,canonical__p001_scratch_thin,canonical,p001,p001_scratch_thin,scratch_thin,scratch_thin,portrait_figure,Baroque,canonical,canonical
4,canonical__p001_zero_control,canonical,p001,p001_zero_control,zero_control,zero_control,portrait_figure,Baroque,canonical,zero_control


In [41]:
BATCH4_SUCCESS_STATUS_VALUES = {
    "",
    "success",
    "complete",
    "completed",
    "ok",
    "passed",
    "written",
    "restored",
    "done",
    "true",
    "1",
}


def batch4_model_failure_pattern(row: pd.Series, prefix: str) -> str:
    source_rows = row.get(f"{prefix}_source_rows")
    has_source_row = pd.notna(source_rows)

    if not has_source_row:
        return "no_manifest_row"

    status = batch4_text(row.get(f"{prefix}_status")).lower()
    issue = batch4_text(row.get(f"{prefix}_issue"))
    returncode = batch4_text(row.get(f"{prefix}_iopaint_returncode"))

    output_written_column = f"{prefix}_output_written"
    restored_path_column = f"{prefix}_restored_path"

    if output_written_column in row.index:
        output_written = batch4_boolish(row.get(output_written_column))
        if output_written is False:
            return "output_not_written"

    if returncode and returncode not in {"0", "0.0"}:
        return f"nonzero_returncode:{batch4_slug(returncode)}"

    if status not in BATCH4_SUCCESS_STATUS_VALUES:
        return f"status:{batch4_slug(status)}"

    if issue:
        return f"issue:{batch4_slug(issue)[:80]}"

    if restored_path_column in row.index and not batch4_text(row.get(restored_path_column)):
        return "restored_path_missing_or_blank"

    return "success_or_not_flagged"


for prefix in ["opencv", "lama"]:
    batch4_analysis_df[f"{prefix}_deterministic_failure_pattern"] = batch4_analysis_df.apply(
        lambda row: batch4_model_failure_pattern(row, prefix),
        axis=1,
    )
    batch4_analysis_df[f"{prefix}_deterministic_failure_flag"] = ~batch4_analysis_df[
        f"{prefix}_deterministic_failure_pattern"
    ].eq("success_or_not_flagged")

display(
    batch4_analysis_df[
        [
            *BATCH4_CASE_KEY_COLUMNS,
            "opencv_deterministic_failure_pattern",
            "lama_deterministic_failure_pattern",
        ]
    ].head()
)

,case_id,dataset_name,painting_id,mask_id,mask_type,opencv_deterministic_failure_pattern,lama_deterministic_failure_pattern
0,canonical__p001_loss_large,canonical,p001,p001_loss_large,loss_large,success_or_not_flagged,success_or_not_flagged
1,canonical__p001_loss_small,canonical,p001,p001_loss_small,loss_small,success_or_not_flagged,success_or_not_flagged
2,canonical__p001_mixed_damage,canonical,p001,p001_mixed_damage,mixed_damage,success_or_not_flagged,success_or_not_flagged
3,canonical__p001_scratch_thin,canonical,p001,p001_scratch_thin,scratch_thin,success_or_not_flagged,success_or_not_flagged
4,canonical__p001_zero_control,canonical,p001,p001_zero_control,zero_control,success_or_not_flagged,issue:zero_control_copied_without_model_inference


In [42]:
def batch4_make_group_summary(summary_section: str, group_columns: list[str]) -> pd.DataFrame:
    rows = []

    grouped = [((), batch4_analysis_df)] if not group_columns else batch4_analysis_df.groupby(
        group_columns,
        dropna=False,
        sort=True,
    )

    for key, group in grouped:
        if not isinstance(key, tuple):
            key = (key,)

        record = {
            "summary_section": summary_section,
            "grouping_columns": ", ".join(group_columns) if group_columns else "all_cases",
        }

        for column in BATCH4_SUMMARY_DIMENSION_COLUMNS:
            record[column] = "all"

        for column, value in zip(group_columns, key):
            record[column] = batch4_text(value) or "unknown"

        majority = group["metric_majority_winner"].fillna("no_votes").astype(str)

        record.update(
            {
                "case_count": int(len(group)),
                "paired_restoration_cases": int(group["restoration_pair_status"].eq("paired").sum()),
                "cases_with_main_local_metrics": int(group["has_main_local_metric_pairs"].fillna(False).astype(bool).sum()),
                "lama_majority_cases": int(majority.eq("lama").sum()),
                "opencv_majority_cases": int(majority.eq("opencv_telea").sum()),
                "tie_majority_cases": int(majority.eq("tie").sum()),
                "mixed_or_no_vote_cases": int(majority.isin(["mixed", "no_votes"]).sum()),
                "needs_visual_review_cases": int(group["needs_visual_review"].fillna(False).astype(bool).sum()),
                "mixed_metric_winner_cases": int(group["flag_mixed_metric_winners"].fillna(False).astype(bool).sum()),
                "family_disagreement_cases": int(group["flag_family_winner_disagreement"].fillna(False).astype(bool).sum()),
                "region_disagreement_cases": int(group["flag_region_winner_disagreement"].fillna(False).astype(bool).sum()),
                "mean_vote_margin_lama_minus_opencv": pd.to_numeric(group["metric_vote_margin_lama_minus_opencv"], errors="coerce").mean(),
                "median_rank_change_lama_minus_opencv": pd.to_numeric(group["rank_change_median_lama_minus_opencv"], errors="coerce").median(),
                "opencv_runtime_median_seconds": group["opencv_runtime_seconds_batch4"].median(),
                "lama_runtime_median_seconds": group["lama_runtime_seconds_batch4"].median(),
                "opencv_runtime_p95_seconds": group["opencv_runtime_seconds_batch4"].quantile(0.95),
                "lama_runtime_p95_seconds": group["lama_runtime_seconds_batch4"].quantile(0.95),
                "opencv_deterministic_failure_cases": int(group["opencv_deterministic_failure_flag"].sum()),
                "lama_deterministic_failure_cases": int(group["lama_deterministic_failure_flag"].sum()),
                "opencv_top_failure_patterns": batch4_top_counts(group["opencv_deterministic_failure_pattern"]),
                "lama_top_failure_patterns": batch4_top_counts(group["lama_deterministic_failure_pattern"]),
            }
        )

        rows.append(record)

    return pd.DataFrame(rows)


BATCH4_GROUP_DEFINITIONS = [
    ("overall", []),
    ("by_mask_type", ["summary_mask_type"]),
    ("by_category", ["summary_category"]),
    ("by_style_or_period", ["summary_style_or_period"]),
    ("by_dataset_name", ["summary_dataset_name"]),
    ("by_damage_family", ["summary_damage_family"]),
    ("by_dataset_damage_family", ["summary_dataset_name", "summary_damage_family"]),
    ("by_mask_damage_family", ["summary_mask_type", "summary_damage_family"]),
    ("by_category_style_or_period", ["summary_category", "summary_style_or_period"]),
]

batch4_compact_summaries_df = pd.concat(
    [
        batch4_make_group_summary(summary_section, group_columns)
        for summary_section, group_columns in BATCH4_GROUP_DEFINITIONS
    ],
    ignore_index=True,
)

display(batch4_compact_summaries_df.head(30))

,summary_section,grouping_columns,summary_mask_type,summary_category,summary_style_or_period,summary_dataset_name,summary_damage_family,case_count,paired_restoration_cases,cases_with_main_local_metrics,lama_majority_cases,opencv_majority_cases,tie_majority_cases,mixed_or_no_vote_cases,needs_visual_review_cases,mixed_metric_winner_cases,family_disagreement_cases,region_disagreement_cases,mean_vote_margin_lama_minus_opencv,median_rank_change_lama_minus_opencv,opencv_runtime_median_seconds,lama_runtime_median_seconds,opencv_runtime_p95_seconds,lama_runtime_p95_seconds,opencv_deterministic_failure_cases,lama_deterministic_failure_cases,opencv_top_failure_patterns,lama_top_failure_patterns
0,overall,all_cases,all,all,all,all,all,410,410,410,297,46,55,12,410,354,217,345,7.014634,0.00,0.445032,1.460938,0.648071,1.460938,0,50,success_or_not_flagged=410,success_or_not_flagged=360; issue:zero_control_copied_without_model_inference=50
1,by_mask_type,summary_mask_type,loss_large,all,all,all,all,50,50,50,50,0,0,0,50,50,33,50,10.800000,0.25,0.482988,1.460938,0.651685,1.460938,0,0,success_or_not_flagged=50,success_or_not_flagged=50
2,by_mask_type,summary_mask_type,loss_small,all,all,all,all,50,50,50,49,0,0,1,50,49,20,48,12.960000,0.25,0.406896,1.460938,0.547525,1.460938,0,0,success_or_not_flagged=50,success_or_not_flagged=50
3,by_mask_type,summary_mask_type,mixed_damage,all,all,all,all,50,50,50,50,0,0,0,50,50,23,50,12.560000,0.00,0.473159,1.460938,0.605182,1.460938,0,0,success_or_not_flagged=50,success_or_not_flagged=50
4,by_mask_type,summary_mask_type,scratch_thin,all,all,all,all,50,50,50,22,23,0,5,50,50,35,47,-0.560000,0.00,0.410826,1.460938,0.549320,1.460938,0,0,success_or_not_flagged=50,success_or_not_flagged=50
5,by_mask_type,summary_mask_type,unknown_mask,all,all,all,all,160,160,160,126,23,5,6,160,155,106,150,6.800000,0.00,0.456829,1.460938,0.681479,1.460938,0,0,success_or_not_flagged=160,success_or_not_flagged=160
6,by_mask_type,summary_mask_type,zero_control,all,all,all,all,50,50,50,0,0,50,0,50,0,0,0,0.000000,0.00,0.388795,0.000000,0.518188,0.000000,0,50,success_or_not_flagged=50,issue:zero_control_copied_without_model_inference=50
7,by_category,summary_category,all,abstraction_surrealism,all,all,all,82,82,82,54,11,12,5,82,69,42,65,7.146341,1.00,0.431353,1.460938,0.607048,1.460938,0,10,success_or_not_flagged=82,success_or_not_flagged=72; issue:zero_control_copied_without_model_inference=10
8,by_category,summary_category,all,architecture_structured,all,all,all,82,82,82,55,14,12,1,82,70,41,70,5.512195,0.00,0.413408,1.460938,0.552357,1.460938,0,10,success_or_not_flagged=82,success_or_not_flagged=72; issue:zero_control_copied_without_model_inference=10
9,by_category,summary_category,all,high_texture_brushwork,all,all,all,82,82,82,65,5,11,1,82,71,47,70,7.512195,-1.25,0.453095,1.460938,0.622505,1.460938,0,10,success_or_not_flagged=82,success_or_not_flagged=72; issue:zero_control_copied_without_model_inference=10


In [43]:
runtime_rows = []

for prefix, model_name in [("opencv", "opencv_telea"), ("lama", "lama")]:
    tmp = batch4_analysis_df[
        BATCH4_CASE_KEY_COLUMNS
        + BATCH4_SUMMARY_DIMENSION_COLUMNS
        + [f"{prefix}_runtime_seconds_batch4", f"{prefix}_deterministic_failure_pattern"]
    ].copy()

    tmp["model_name"] = model_name
    tmp["runtime_seconds"] = tmp[f"{prefix}_runtime_seconds_batch4"]
    tmp["deterministic_failure_pattern"] = tmp[f"{prefix}_deterministic_failure_pattern"]
    runtime_rows.append(tmp)

batch4_runtime_long_df = pd.concat(runtime_rows, ignore_index=True)

batch4_runtime_summary_df = (
    batch4_runtime_long_df.groupby(
        ["model_name", "summary_mask_type", "summary_damage_family"],
        dropna=False,
    )
    .agg(
        case_count=("runtime_seconds", "size"),
        runtime_available_cases=("runtime_seconds", lambda value: int(value.notna().sum())),
        runtime_missing_cases=("runtime_seconds", lambda value: int(value.isna().sum())),
        runtime_mean_seconds=("runtime_seconds", "mean"),
        runtime_median_seconds=("runtime_seconds", "median"),
        runtime_p95_seconds=("runtime_seconds", lambda value: value.quantile(0.95)),
        runtime_max_seconds=("runtime_seconds", "max"),
        deterministic_failure_cases=(
            "deterministic_failure_pattern",
            lambda value: int((value != "success_or_not_flagged").sum()),
        ),
        top_failure_patterns=("deterministic_failure_pattern", batch4_top_counts),
    )
    .reset_index()
)

display(batch4_runtime_summary_df.head(30))

,model_name,summary_mask_type,summary_damage_family,case_count,runtime_available_cases,runtime_missing_cases,runtime_mean_seconds,runtime_median_seconds,runtime_p95_seconds,runtime_max_seconds,deterministic_failure_cases,top_failure_patterns
0,lama,loss_large,canonical,50,50,0,1.460938,1.460938,1.460938,1.460938,0,success_or_not_flagged=50
1,lama,loss_small,canonical,50,50,0,1.460938,1.460938,1.460938,1.460938,0,success_or_not_flagged=50
2,lama,mixed_damage,canonical,50,50,0,1.460938,1.460938,1.460938,1.460938,0,success_or_not_flagged=50
3,lama,scratch_thin,canonical,50,50,0,1.460938,1.460938,1.460938,1.460938,0,success_or_not_flagged=50
4,lama,unknown_mask,damage_size,35,35,0,1.460938,1.460938,1.460938,1.460938,0,success_or_not_flagged=35
5,lama,unknown_mask,mask_robustness,75,75,0,1.460938,1.460938,1.460938,1.460938,0,success_or_not_flagged=75
6,lama,unknown_mask,synthetic_degradation,50,50,0,1.460938,1.460938,1.460938,1.460938,0,success_or_not_flagged=50
7,lama,zero_control,zero_control,50,50,0,0.000000,0.000000,0.000000,0.000000,50,issue:zero_control_copied_without_model_inference=50
8,opencv_telea,loss_large,canonical,50,50,0,0.492091,0.482988,0.651685,0.676054,0,success_or_not_flagged=50
9,opencv_telea,loss_small,canonical,50,50,0,0.421457,0.406896,0.547525,0.593053,0,success_or_not_flagged=50


In [44]:
failure_rows = []

for prefix, model_name in [("opencv", "opencv_telea"), ("lama", "lama")]:
    tmp = batch4_analysis_df[
        BATCH4_CASE_KEY_COLUMNS
        + BATCH4_SUMMARY_DIMENSION_COLUMNS
        + [
            f"{prefix}_deterministic_failure_pattern",
            f"{prefix}_runtime_seconds_batch4",
        ]
    ].copy()

    tmp["model_name"] = model_name
    tmp["deterministic_failure_pattern"] = tmp[f"{prefix}_deterministic_failure_pattern"]
    tmp["runtime_seconds"] = tmp[f"{prefix}_runtime_seconds_batch4"]
    tmp["is_deterministic_failure"] = ~tmp["deterministic_failure_pattern"].eq("success_or_not_flagged")

    failure_rows.append(tmp)

batch4_failure_case_long_df = pd.concat(failure_rows, ignore_index=True)

batch4_failure_patterns_df = (
    batch4_failure_case_long_df.groupby(
        [
            "model_name",
            "is_deterministic_failure",
            "deterministic_failure_pattern",
            "summary_dataset_name",
            "summary_mask_type",
            "summary_category",
            "summary_style_or_period",
            "summary_damage_family",
        ],
        dropna=False,
    )
    .agg(
        case_count=("case_id", "size"),
        runtime_available_cases=("runtime_seconds", lambda value: int(value.notna().sum())),
        runtime_median_seconds=("runtime_seconds", "median"),
        case_examples=("case_id", lambda value: ", ".join(value.dropna().astype(str).head(5))),
    )
    .reset_index()
    .sort_values(
        ["is_deterministic_failure", "case_count", "model_name"],
        ascending=[False, False, True],
    )
)

display(batch4_failure_patterns_df.head(40))

,model_name,is_deterministic_failure,deterministic_failure_pattern,summary_dataset_name,summary_mask_type,summary_category,summary_style_or_period,summary_damage_family,case_count,runtime_available_cases,runtime_median_seconds,case_examples
91,lama,True,issue:zero_control_copied_without_model_inference,canonical,zero_control,abstraction_surrealism,unknown_style_or_period,zero_control,10,10,0.000000,"canonical__p031_zero_control, canonical__p032_zero_control, canonical__p033_zero_control, canonical__p034_zero_control, canonical__p035_zero_control"
92,lama,True,issue:zero_control_copied_without_model_inference,canonical,zero_control,architecture_structured,unknown_style_or_period,zero_control,10,10,0.000000,"canonical__p021_zero_control, canonical__p022_zero_control, canonical__p023_zero_control, canonical__p024_zero_control, canonical__p025_zero_control"
99,lama,True,issue:zero_control_copied_without_model_inference,canonical,zero_control,landscape_natural,unknown_style_or_period,zero_control,10,10,0.000000,"canonical__p011_zero_control, canonical__p012_zero_control, canonical__p013_zero_control, canonical__p014_zero_control, canonical__p015_zero_control"
97,lama,True,issue:zero_control_copied_without_model_inference,canonical,zero_control,high_texture_brushwork,Post-Impressionism,zero_control,3,3,0.000000,"canonical__p041_zero_control, canonical__p043_zero_control, canonical__p048_zero_control"
93,lama,True,issue:zero_control_copied_without_model_inference,canonical,zero_control,high_texture_brushwork,Impressionism,zero_control,2,2,0.000000,"canonical__p044_zero_control, canonical__p047_zero_control"
98,lama,True,issue:zero_control_copied_without_model_inference,canonical,zero_control,high_texture_brushwork,unknown_style_or_period,zero_control,2,2,0.000000,"canonical__p042_zero_control, canonical__p050_zero_control"
94,lama,True,issue:zero_control_copied_without_model_inference,canonical,zero_control,high_texture_brushwork,Impressionism / modern French painting,zero_control,1,1,0.000000,canonical__p046_zero_control
95,lama,True,issue:zero_control_copied_without_model_inference,canonical,zero_control,high_texture_brushwork,Impressionist outdoor figure painting,zero_control,1,1,0.000000,canonical__p045_zero_control
96,lama,True,issue:zero_control_copied_without_model_inference,canonical,zero_control,high_texture_brushwork,Modern French painting,zero_control,1,1,0.000000,canonical__p049_zero_control
100,lama,True,issue:zero_control_copied_without_model_inference,canonical,zero_control,portrait_figure,18th century portraiture,zero_control,1,1,0.000000,canonical__p005_zero_control


In [45]:
expected_summary_sections = [name for name, _ in BATCH4_GROUP_DEFINITIONS]
observed_summary_sections = sorted(batch4_compact_summaries_df["summary_section"].dropna().unique().tolist())

batch4_check_rows = [
    build_check(
        "batch4_analysis_has_rows",
        len(batch4_analysis_df),
        "> 0",
        len(batch4_analysis_df) > 0,
        "Batch 4 analysis table has no rows.",
    ),
    build_check(
        "batch4_all_summary_sections_created",
        observed_summary_sections,
        expected_summary_sections,
        set(expected_summary_sections).issubset(set(observed_summary_sections)),
        "One or more expected compact summary sections are missing.",
    ),
    build_check(
        "batch4_compact_summaries_have_rows",
        len(batch4_compact_summaries_df),
        "> 0",
        len(batch4_compact_summaries_df) > 0,
        "Compact summary CSV would be empty.",
    ),
    build_check(
        "batch4_runtime_summary_has_both_models",
        sorted(batch4_runtime_summary_df["model_name"].dropna().unique().tolist()),
        ["lama", "opencv_telea"],
        {"lama", "opencv_telea"}.issubset(set(batch4_runtime_summary_df["model_name"].dropna().unique())),
        "Runtime summary does not contain both models.",
    ),
    build_check(
        "batch4_failure_patterns_have_both_models",
        sorted(batch4_failure_patterns_df["model_name"].dropna().unique().tolist()),
        ["lama", "opencv_telea"],
        {"lama", "opencv_telea"}.issubset(set(batch4_failure_patterns_df["model_name"].dropna().unique())),
        "Failure-pattern summary does not contain both models.",
    ),
    build_check(
        "batch4_category_column_created",
        batch4_analysis_df["summary_category"].dropna().nunique(),
        ">= 1",
        batch4_analysis_df["summary_category"].dropna().nunique() >= 1,
        "Category summary column was not created.",
    ),
    build_check(
        "batch4_style_or_period_column_created",
        batch4_analysis_df["summary_style_or_period"].dropna().nunique(),
        ">= 1",
        batch4_analysis_df["summary_style_or_period"].dropna().nunique() >= 1,
        "Style/period summary column was not created.",
    ),
]

batch4_validation_df = pd.DataFrame(batch4_check_rows)

display(batch4_validation_df)

if not batch4_validation_df["passed"].astype(bool).all():
    failed_batch4_checks_df = batch4_validation_df[
        ~batch4_validation_df["passed"].astype(bool)
    ].copy()
    display(failed_batch4_checks_df)
    raise RuntimeError("Notebook 20 Batch 4 validation failed. Inspect compact summaries.")

,check_name,observed,expected,passed,failure_message
0,batch4_analysis_has_rows,410,> 0,True,
1,batch4_all_summary_sections_created,"[by_category, by_category_style_or_period, by_damage_family, by_dataset_damage_family, by_dataset_name, by_mask_damage_family, by_mask_type, by_style_or_period, overall]","[overall, by_mask_type, by_category, by_style_or_period, by_dataset_name, by_damage_family, by_dataset_damage_family, by_mask_damage_family, by_category_style_or_period]",True,
2,batch4_compact_summaries_have_rows,67,> 0,True,
3,batch4_runtime_summary_has_both_models,"[lama, opencv_telea]","[lama, opencv_telea]",True,
4,batch4_failure_patterns_have_both_models,"[lama, opencv_telea]","[lama, opencv_telea]",True,
5,batch4_category_column_created,5,>= 1,True,
6,batch4_style_or_period_column_created,14,>= 1,True,


In [46]:
batch4_compact_summaries_df.to_csv(BATCH4_COMPACT_SUMMARIES_OUTPUT_PATH, index=False)
batch4_runtime_summary_df.to_csv(BATCH4_RUNTIME_SUMMARY_OUTPUT_PATH, index=False)
batch4_failure_patterns_df.to_csv(BATCH4_FAILURE_PATTERNS_OUTPUT_PATH, index=False)

batch4_validation_export_df = batch4_validation_df.copy()
for column in ["observed", "expected"]:
    batch4_validation_export_df[column] = batch4_validation_export_df[column].map(json_for_csv)

batch4_validation_export_df.to_csv(BATCH4_VALIDATION_OUTPUT_PATH, index=False)

stage_manifest = read_json(STAGE_MANIFEST_JSON_PATH)

stage_manifest.update(
    {
        "status": "batch_4_compact_summaries_complete",
        "updated_at_utc": datetime.now(timezone.utc).isoformat(),
    }
)

stage_manifest.setdefault("outputs", {})
stage_manifest["outputs"].update(
    {
        "batch4_compact_summaries": project_relative_path(BATCH4_COMPACT_SUMMARIES_OUTPUT_PATH),
        "batch4_runtime_summary": project_relative_path(BATCH4_RUNTIME_SUMMARY_OUTPUT_PATH),
        "batch4_failure_patterns": project_relative_path(BATCH4_FAILURE_PATTERNS_OUTPUT_PATH),
        "batch4_validation": project_relative_path(BATCH4_VALIDATION_OUTPUT_PATH),
    }
)

stage_manifest.setdefault("row_counts", {})
stage_manifest["row_counts"].update(
    {
        "batch4_analysis_rows": int(len(batch4_analysis_df)),
        "batch4_compact_summary_rows": int(len(batch4_compact_summaries_df)),
        "batch4_runtime_summary_rows": int(len(batch4_runtime_summary_df)),
        "batch4_failure_pattern_rows": int(len(batch4_failure_patterns_df)),
        "batch4_validation_checks": int(len(batch4_validation_df)),
    }
)

stage_manifest["batch4_summary_sections"] = expected_summary_sections
stage_manifest["next_batch"] = (
    "Batch 5: select visual review cases, audit image paths, create comparison panels, "
    "build figure manifest, and generate the HTML comparison report."
)

save_json(STAGE_MANIFEST_JSON_PATH, stage_manifest)

print("Batch 4 complete.")
print("Saved:", project_relative_path(BATCH4_COMPACT_SUMMARIES_OUTPUT_PATH))
print("Saved:", project_relative_path(BATCH4_RUNTIME_SUMMARY_OUTPUT_PATH))
print("Saved:", project_relative_path(BATCH4_FAILURE_PATTERNS_OUTPUT_PATH))
print("Saved:", project_relative_path(BATCH4_VALIDATION_OUTPUT_PATH))
print("Updated manifest:", project_relative_path(STAGE_MANIFEST_JSON_PATH))

Batch 4 complete.
Saved: outputs/20_opencv_lama_comparison_inventory_rebuild_v2/batch4_compact_summaries/batch4_compact_group_summaries.csv
Saved: outputs/20_opencv_lama_comparison_inventory_rebuild_v2/batch4_compact_summaries/batch4_runtime_summary.csv
Saved: outputs/20_opencv_lama_comparison_inventory_rebuild_v2/batch4_compact_summaries/batch4_deterministic_failure_patterns.csv
Saved: outputs/20_opencv_lama_comparison_inventory_rebuild_v2/batch4_compact_summaries/batch4_validation.csv
Updated manifest: outputs/20_opencv_lama_comparison_inventory_rebuild_v2/stage_manifest.json


In [48]:
import base64
import html
import mimetypes
import re
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    from PIL import Image, ImageDraw, ImageFont, ImageOps
    PIL_AVAILABLE = True
except ImportError:
    PIL_AVAILABLE = False

BATCH5_BASE_OUTPUT_DIR = globals().get("OUTPUT_ROOT", globals().get("NOTEBOOK_OUTPUT_DIR"))
if BATCH5_BASE_OUTPUT_DIR is None:
    BATCH5_BASE_OUTPUT_DIR = PROJECT_ROOT / "outputs" / "20_opencv_lama_comparison_v2"
BATCH5_BASE_OUTPUT_DIR = Path(BATCH5_BASE_OUTPUT_DIR)

BATCH5_OUTPUT_DIR = BATCH5_BASE_OUTPUT_DIR / "batch5_visual_report"
BATCH5_TABLES_DIR = BATCH5_OUTPUT_DIR / "tables"
BATCH5_FIGURES_DIR = BATCH5_OUTPUT_DIR / "figures"
BATCH5_PANELS_DIR = BATCH5_FIGURES_DIR / "selected_case_panels"
BATCH5_PLOTS_DIR = BATCH5_FIGURES_DIR / "plots"
BATCH5_REPORTS_DIR = BATCH5_OUTPUT_DIR / "reports"
BATCH5_VALIDATION_DIR = BATCH5_OUTPUT_DIR / "validation"

for directory in [BATCH5_TABLES_DIR, BATCH5_PANELS_DIR, BATCH5_PLOTS_DIR, BATCH5_REPORTS_DIR, BATCH5_VALIDATION_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

BATCH5_SELECTED_CASES_OUTPUT_PATH = BATCH5_TABLES_DIR / "opencv_lama_batch5_selected_visual_cases.csv"
BATCH5_IMAGE_PATH_AUDIT_OUTPUT_PATH = BATCH5_TABLES_DIR / "opencv_lama_batch5_image_path_audit.csv"
BATCH5_FIGURE_MANIFEST_OUTPUT_PATH = BATCH5_TABLES_DIR / "opencv_lama_batch5_figure_manifest.csv"
BATCH5_HTML_REPORT_OUTPUT_PATH = BATCH5_REPORTS_DIR / "opencv_lama_batch5_comparison_report.html"
BATCH5_VALIDATION_OUTPUT_PATH = BATCH5_VALIDATION_DIR / "opencv_lama_comparison_batch5_validation.csv"

BATCH5_MAX_SELECTED_CASES = 24
BATCH5_PANEL_TILE_SIZE = (260, 220)
BATCH5_CASE_KEY_COLUMNS = globals().get("BATCH4_CASE_KEY_COLUMNS", globals().get("BATCH3_CASE_KEY_COLUMNS", ["case_id"]))

print("Batch 5 tables:", project_relative_path(BATCH5_TABLES_DIR))
print("Batch 5 figures:", project_relative_path(BATCH5_FIGURES_DIR))
print("Batch 5 report:", project_relative_path(BATCH5_HTML_REPORT_OUTPUT_PATH))

Batch 5 tables: outputs/20_opencv_lama_comparison_inventory_rebuild_v2/batch5_visual_report/tables
Batch 5 figures: outputs/20_opencv_lama_comparison_inventory_rebuild_v2/batch5_visual_report/figures
Batch 5 report: outputs/20_opencv_lama_comparison_inventory_rebuild_v2/batch5_visual_report/reports/opencv_lama_batch5_comparison_report.html


In [49]:
def batch5_text(value) -> str:
    if value is None:
        return ""
    if isinstance(value, float) and np.isnan(value):
        return ""
    text_value = str(value).strip()
    return "" if text_value.lower() in {"nan", "none", "null", "<na>"} else text_value


def batch5_slug(value) -> str:
    text_value = batch5_text(value).lower()
    text_value = re.sub(r"[^a-z0-9]+", "_", text_value)
    return text_value.strip("_") or "unknown"


def batch5_resolve_path(value) -> Path | None:
    text_value = batch5_text(value)
    if not text_value:
        return None
    candidate = Path(text_value)
    if candidate.is_absolute():
        return candidate
    return PROJECT_ROOT / candidate


def batch5_project_path(value) -> str:
    path = batch5_resolve_path(value)
    if path is None:
        return ""
    try:
        return project_relative_path(path)
    except Exception:
        return str(path)


def batch5_first_existing_column(df: pd.DataFrame, candidates: list[str]) -> str | None:
    for column in candidates:
        if column in df.columns:
            return column
    return None


def batch5_first_nonblank_from_row(row: pd.Series, candidates: list[str]) -> str:
    for column in candidates:
        if column in row.index:
            value = batch5_text(row.get(column))
            if value:
                return value
    return ""


def batch5_numeric(df: pd.DataFrame, column: str) -> pd.Series:
    if column not in df.columns:
        return pd.Series([np.nan] * len(df), index=df.index)
    return pd.to_numeric(df[column], errors="coerce")


def batch5_bool_series(df: pd.DataFrame, column: str, default: bool = False) -> pd.Series:
    if column not in df.columns:
        return pd.Series([default] * len(df), index=df.index, dtype=bool)
    return df[column].fillna(default).astype(bool)


def batch5_case_key_from_row(row: pd.Series, key_columns: list[str]) -> str:
    present_columns = [column for column in key_columns if column in row.index]
    if not present_columns:
        present_columns = ["case_id"] if "case_id" in row.index else []
    if not present_columns:
        return batch5_text(row.name)
    return "|".join(f"{column}={batch5_text(row.get(column))}" for column in present_columns)


def batch5_file_to_data_uri(path: Path) -> str:
    mime_type = mimetypes.guess_type(str(path))[0] or "image/png"
    data = base64.b64encode(path.read_bytes()).decode("ascii")
    return f"data:{mime_type};base64,{data}"


def batch5_dataframe_to_html(df: pd.DataFrame, max_rows: int = 30) -> str:
    if df is None or df.empty:
        return "<p>No rows available.</p>"
    display_df = df.head(max_rows).copy()
    return display_df.to_html(index=False, escape=True, border=0, classes="data-table")


In [50]:
if "batch4_analysis_df" in globals() and isinstance(batch4_analysis_df, pd.DataFrame):
    batch5_cases_source_df = batch4_analysis_df.copy()
elif "batch3_unified_cases_df" in globals() and isinstance(batch3_unified_cases_df, pd.DataFrame):
    batch5_cases_source_df = batch3_unified_cases_df.copy()
elif "BATCH3_UNIFIED_CASES_OUTPUT_PATH" in globals() and Path(BATCH3_UNIFIED_CASES_OUTPUT_PATH).is_file():
    batch5_cases_source_df = pd.read_csv(BATCH3_UNIFIED_CASES_OUTPUT_PATH, low_memory=False)
else:
    raise RuntimeError("Batch 5 needs batch4_analysis_df, batch3_unified_cases_df, or the Batch 3 unified cases CSV.")

if "batch4_compact_summaries_df" not in globals():
    batch4_compact_summaries_df = pd.DataFrame()
if "batch4_runtime_summary_df" not in globals():
    batch4_runtime_summary_df = pd.DataFrame()
if "batch4_failure_patterns_df" not in globals():
    batch4_failure_patterns_df = pd.DataFrame()

batch5_cases_df = batch5_cases_source_df.copy()
for column in BATCH5_CASE_KEY_COLUMNS:
    if column in batch5_cases_df.columns:
        batch5_cases_df[column] = batch5_cases_df[column].map(batch5_text)

batch5_present_case_key_columns = [column for column in BATCH5_CASE_KEY_COLUMNS if column in batch5_cases_df.columns]
if not batch5_present_case_key_columns and "case_id" in batch5_cases_df.columns:
    batch5_present_case_key_columns = ["case_id"]

batch5_cases_df["batch5_visual_case_key"] = batch5_cases_df.apply(
    lambda row: batch5_case_key_from_row(row, batch5_present_case_key_columns),
    axis=1,
)

print("Batch 5 input cases:", len(batch5_cases_df))
print("Batch 5 input columns:", len(batch5_cases_df.columns))


Batch 5 input cases: 410
Batch 5 input columns: 167


In [51]:
BATCH5_IMAGE_ROLE_CANDIDATES = {
    "clean": [
        "clean_path", "clean_image_path", "ground_truth_path", "target_path", "original_path",
        "source_clean_path", "opencv_clean_path", "lama_clean_path", "clean_file",
        "clean_image_file", "reference_path", "reference_image_path",
    ],
    "damaged": [
        "damaged_path", "damaged_image_path", "input_path", "source_damaged_path",
        "opencv_damaged_path", "lama_damaged_path", "damaged_file", "damaged_image_file",
        "input_image_path",
    ],
    "mask": [
        "mask_path", "mask_image_path", "binary_mask_path", "source_mask_path",
        "opencv_mask_path", "lama_mask_path", "mask_file", "mask_image_file",
        "binary_mask_file",
    ],
    "opencv_restored": [
        "opencv_restored_path", "opencv_output_path", "opencv_restoration_path",
        "restored_path_opencv", "restored_path_opencv_telea", "opencv_restored_image_path",
        "opencv_telea_restored_path", "opencv_telea_restored_image_path",
        "restored_image_path_opencv", "restored_image_path_opencv_telea",
    ],
    "lama_restored": [
        "lama_restored_path", "lama_output_path", "lama_restoration_path", "restored_path_lama",
        "lama_restored_image_path", "restored_image_path_lama",
    ],
}

# Pull in path hints from SOURCE_TABLES when Batch 3 did not carry clean/damaged/mask paths forward.
def batch5_source_path_lookup() -> pd.DataFrame:
    if "SOURCE_TABLES" not in globals():
        return pd.DataFrame(columns=BATCH5_CASE_KEY_COLUMNS)

    lookup_frames = []
    source_path_candidates = sorted(set(sum(BATCH5_IMAGE_ROLE_CANDIDATES.values(), [])))
    for source_id, source_df in SOURCE_TABLES.items():
        if not isinstance(source_df, pd.DataFrame):
            continue
        if any(column not in source_df.columns for column in BATCH5_CASE_KEY_COLUMNS):
            continue
        available_path_columns = [column for column in source_path_candidates if column in source_df.columns]
        if not available_path_columns:
            continue
        tmp = source_df[BATCH5_CASE_KEY_COLUMNS + available_path_columns].copy()
        for column in BATCH5_CASE_KEY_COLUMNS + available_path_columns:
            tmp[column] = tmp[column].map(batch5_text)
        lookup_frames.append(tmp)

    if not lookup_frames:
        return pd.DataFrame(columns=BATCH5_CASE_KEY_COLUMNS)

    merged_lookup_df = pd.concat(lookup_frames, ignore_index=True)
    agg_spec = {column: (column, lambda values: next((batch5_text(v) for v in values if batch5_text(v)), "")) for column in merged_lookup_df.columns if column not in BATCH5_CASE_KEY_COLUMNS}
    return merged_lookup_df.groupby(BATCH5_CASE_KEY_COLUMNS, dropna=False).agg(**agg_spec).reset_index()

batch5_source_paths_df = batch5_source_path_lookup()
if not batch5_source_paths_df.empty:
    batch5_cases_df = batch5_cases_df.merge(batch5_source_paths_df, on=BATCH5_CASE_KEY_COLUMNS, how="left", suffixes=("", "_source_lookup"))

for role, candidates in BATCH5_IMAGE_ROLE_CANDIDATES.items():
    expanded_candidates = candidates + [f"{column}_source_lookup" for column in candidates]
    batch5_cases_df[f"{role}_path_batch5"] = batch5_cases_df.apply(
        lambda row: batch5_first_nonblank_from_row(row, expanded_candidates),
        axis=1,
    )
    batch5_cases_df[f"{role}_path_resolved_batch5"] = batch5_cases_df[f"{role}_path_batch5"].map(batch5_project_path)
    batch5_cases_df[f"{role}_path_exists_batch5"] = batch5_cases_df[f"{role}_path_batch5"].map(
        lambda value: bool((path := batch5_resolve_path(value)) and path.is_file())
    )

image_audit_rows = []
for role in BATCH5_IMAGE_ROLE_CANDIDATES:
    for _, row in batch5_cases_df.iterrows():
        image_audit_rows.append(
            {
                "case_id": row.get("case_id", ""),
                "image_role": role,
                "path_raw": row.get(f"{role}_path_batch5", ""),
                "path_resolved": row.get(f"{role}_path_resolved_batch5", ""),
                "path_exists": bool(row.get(f"{role}_path_exists_batch5", False)),
            }
        )

batch5_image_path_audit_df = pd.DataFrame(image_audit_rows)
batch5_image_path_audit_df.to_csv(BATCH5_IMAGE_PATH_AUDIT_OUTPUT_PATH, index=False)

display(batch5_image_path_audit_df.groupby("image_role")["path_exists"].agg(["sum", "count"]).reset_index())


,image_role,sum,count
0,clean,410,410
1,damaged,410,410
2,lama_restored,410,410
3,mask,410,410
4,opencv_restored,410,410


In [52]:
batch5_rank_score = batch5_numeric(batch5_cases_df, "rank_change_median_lama_minus_opencv")
batch5_vote_margin = batch5_numeric(batch5_cases_df, "metric_vote_margin_lama_minus_opencv")
batch5_metric_rows = batch5_numeric(batch5_cases_df, "main_local_metric_pair_rows")

batch5_cases_df["batch5_selection_score"] = 0.0
batch5_cases_df["batch5_selection_score"] += batch5_bool_series(batch5_cases_df, "needs_visual_review").astype(int) * 6
batch5_cases_df["batch5_selection_score"] += batch5_bool_series(batch5_cases_df, "flag_mixed_metric_winners").astype(int) * 4
batch5_cases_df["batch5_selection_score"] += batch5_bool_series(batch5_cases_df, "flag_family_winner_disagreement").astype(int) * 3
batch5_cases_df["batch5_selection_score"] += batch5_bool_series(batch5_cases_df, "flag_region_winner_disagreement").astype(int) * 3
batch5_cases_df["batch5_selection_score"] += batch5_bool_series(batch5_cases_df, "opencv_deterministic_failure_flag").astype(int) * 5
batch5_cases_df["batch5_selection_score"] += batch5_bool_series(batch5_cases_df, "lama_deterministic_failure_flag").astype(int) * 5
batch5_cases_df["batch5_selection_score"] += batch5_rank_score.abs().fillna(0).clip(upper=10)
batch5_cases_df["batch5_selection_score"] += batch5_vote_margin.abs().fillna(0).clip(upper=10) / 2
batch5_cases_df["batch5_selection_score"] += batch5_metric_rows.fillna(0).clip(upper=20) / 20

batch5_reason_columns = [
    "visual_review_reasons",
    "opencv_deterministic_failure_pattern",
    "lama_deterministic_failure_pattern",
    "case_presence_status",
]

def batch5_selection_reason(row: pd.Series) -> str:
    reasons = []
    for column in batch5_reason_columns:
        value = batch5_text(row.get(column))
        if value and value not in {"success_or_not_flagged", "paired_restoration_with_main_local_metrics"}:
            reasons.append(f"{column}={value}")
    if row.get("batch5_high_lama_case", False):
        reasons.append("strong_lama_metric_advantage")
    if row.get("batch5_high_opencv_case", False):
        reasons.append("strong_opencv_metric_advantage")
    if row.get("batch5_coverage_case", False):
        reasons.append("coverage_case")
    return "; ".join(reasons[:6]) or "high_priority_metric_review_case"

batch5_cases_df["batch5_high_lama_case"] = batch5_vote_margin.rank(method="first", ascending=False) <= 6
batch5_cases_df["batch5_high_opencv_case"] = batch5_vote_margin.rank(method="first", ascending=True) <= 6
batch5_cases_df["batch5_coverage_case"] = False

coverage_columns = [
    column for column in ["summary_mask_type", "summary_category", "summary_style_or_period", "summary_dataset_name", "summary_damage_family", "mask_type", "dataset_name"]
    if column in batch5_cases_df.columns
]
for column in coverage_columns:
    top_per_group = batch5_cases_df.sort_values("batch5_selection_score", ascending=False).groupby(column, dropna=False).head(1).index
    batch5_cases_df.loc[top_per_group, "batch5_coverage_case"] = True

priority_mask = (
    batch5_bool_series(batch5_cases_df, "needs_visual_review")
    | batch5_bool_series(batch5_cases_df, "batch5_high_lama_case")
    | batch5_bool_series(batch5_cases_df, "batch5_high_opencv_case")
    | batch5_bool_series(batch5_cases_df, "batch5_coverage_case")
)

batch5_selected_cases_df = batch5_cases_df.loc[priority_mask].copy()
if batch5_selected_cases_df.empty:
    batch5_selected_cases_df = batch5_cases_df.copy()

batch5_selected_cases_df["selection_reason"] = batch5_selected_cases_df.apply(batch5_selection_reason, axis=1)
batch5_selected_cases_df = (
    batch5_selected_cases_df.sort_values(["batch5_selection_score", "case_id"], ascending=[False, True])
    .drop_duplicates(subset=["batch5_visual_case_key"])
    .head(BATCH5_MAX_SELECTED_CASES)
    .reset_index(drop=True)
)

batch5_selected_cases_df["batch5_selected_order"] = np.arange(1, len(batch5_selected_cases_df) + 1)
batch5_selected_cases_df.to_csv(BATCH5_SELECTED_CASES_OUTPUT_PATH, index=False)

print("Selected visual cases:", len(batch5_selected_cases_df))
display(batch5_selected_cases_df[[column for column in ["batch5_selected_order", "case_id", "selection_reason", "metric_majority_winner", "batch5_selection_score"] if column in batch5_selected_cases_df.columns]].head(30))


Selected visual cases: 24


,batch5_selected_order,case_id,selection_reason,metric_majority_winner,batch5_selection_score
0,1,canonical__p005_loss_large,"visual_review_reasons=flag_mixed_metric_winners, flag_family_winner_disagreement, flag_region_winner_disagreement, flag_restored_vs_improvement_winner_disagreement, flag_rank_a...",lama,31.0
1,2,canonical__p007_loss_large,"visual_review_reasons=flag_mixed_metric_winners, flag_family_winner_disagreement, flag_region_winner_disagreement, flag_restored_vs_improvement_winner_disagreement, flag_rank_a...",lama,31.0
2,3,canonical__p010_loss_large,"visual_review_reasons=flag_mixed_metric_winners, flag_family_winner_disagreement, flag_region_winner_disagreement, flag_restored_vs_improvement_winner_disagreement; coverage_case",lama,31.0
3,4,canonical__p012_loss_large,"visual_review_reasons=flag_mixed_metric_winners, flag_family_winner_disagreement, flag_region_winner_disagreement, flag_restored_vs_improvement_winner_disagreement",lama,31.0
4,5,canonical__p020_mixed_damage,"visual_review_reasons=flag_mixed_metric_winners, flag_family_winner_disagreement, flag_region_winner_disagreement, flag_restored_vs_improvement_winner_disagreement, flag_rank_a...",lama,31.0
5,6,canonical__p037_scratch_thin,"visual_review_reasons=flag_mixed_metric_winners, flag_family_winner_disagreement, flag_region_winner_disagreement, flag_restored_vs_improvement_winner_disagreement; coverage_case",opencv_telea,31.0
6,7,canonical__p041_loss_large,"visual_review_reasons=flag_mixed_metric_winners, flag_family_winner_disagreement, flag_region_winner_disagreement, flag_restored_vs_improvement_winner_disagreement; coverage_case",lama,31.0
7,8,canonical__p050_loss_large,"visual_review_reasons=flag_mixed_metric_winners, flag_family_winner_disagreement, flag_region_winner_disagreement, flag_restored_vs_improvement_winner_disagreement; coverage_case",lama,31.0
8,9,mask_robustness__p026__loss_large__variant_01,"visual_review_reasons=flag_mixed_metric_winners, flag_family_winner_disagreement, flag_region_winner_disagreement, flag_restored_vs_improvement_winner_disagreement",lama,31.0
9,10,mask_robustness__p026__loss_large__variant_05,"visual_review_reasons=flag_mixed_metric_winners, flag_family_winner_disagreement, flag_region_winner_disagreement, flag_restored_vs_improvement_winner_disagreement; coverage_case",lama,31.0


In [53]:
figure_manifest_records = []


def batch5_add_figure_record(figure_type: str, figure_path: Path, title: str, related_case_id: str = "", notes: str = "") -> None:
    figure_manifest_records.append(
        {
            "figure_id": batch5_slug(figure_path.stem),
            "figure_type": figure_type,
            "title": title,
            "related_case_id": related_case_id,
            "relative_path": project_relative_path(figure_path),
            "exists": bool(figure_path.is_file()),
            "size_bytes": int(figure_path.stat().st_size) if figure_path.is_file() else 0,
            "notes": notes,
        }
    )


def batch5_save_bar_plot(df: pd.DataFrame, group_column: str, value_column: str, output_name: str, title: str, ylabel: str) -> Path | None:
    if group_column not in df.columns or value_column not in df.columns:
        return None
    plot_df = df[[group_column, value_column]].copy()
    plot_df[value_column] = pd.to_numeric(plot_df[value_column], errors="coerce")
    plot_df = plot_df.dropna(subset=[value_column])
    if plot_df.empty:
        return None
    grouped = plot_df.groupby(group_column, dropna=False)[value_column].mean().sort_values(ascending=False).head(16)
    if grouped.empty:
        return None

    fig, ax = plt.subplots(figsize=(11, max(4, 0.42 * len(grouped))))
    grouped.sort_values().plot(kind="barh", ax=ax, color="#2563eb")
    ax.axvline(0, color="#111827", linewidth=0.8)
    ax.set_title(title)
    ax.set_xlabel(ylabel)
    ax.set_ylabel(group_column)
    fig.tight_layout()
    output_path = BATCH5_PLOTS_DIR / output_name
    fig.savefig(output_path, dpi=160, bbox_inches="tight")
    plt.close(fig)
    batch5_add_figure_record("summary_plot", output_path, title)
    return output_path

batch5_save_bar_plot(batch5_cases_df, "metric_majority_winner", "metric_vote_margin_lama_minus_opencv", "batch5_vote_margin_by_majority_winner.png", "Mean Vote Margin By Majority Winner", "Mean LaMa minus OpenCV vote margin")
batch5_save_bar_plot(batch5_cases_df, "summary_mask_type", "metric_vote_margin_lama_minus_opencv", "batch5_vote_margin_by_mask_type.png", "Mean Vote Margin By Mask Type", "Mean LaMa minus OpenCV vote margin")
batch5_save_bar_plot(batch5_cases_df, "summary_damage_family", "metric_vote_margin_lama_minus_opencv", "batch5_vote_margin_by_damage_family.png", "Mean Vote Margin By Damage Family", "Mean LaMa minus OpenCV vote margin")
batch5_save_bar_plot(batch5_cases_df, "summary_category", "metric_vote_margin_lama_minus_opencv", "batch5_vote_margin_by_category.png", "Mean Vote Margin By Category", "Mean LaMa minus OpenCV vote margin")
batch5_save_bar_plot(batch5_cases_df, "summary_style_or_period", "metric_vote_margin_lama_minus_opencv", "batch5_vote_margin_by_style_or_period.png", "Mean Vote Margin By Style/Period", "Mean LaMa minus OpenCV vote margin")

print("Generated summary plots:", len(figure_manifest_records))


Generated summary plots: 5


In [54]:
if not PIL_AVAILABLE:
    raise RuntimeError("Batch 5 requires Pillow to generate selected-case comparison panels.")

BATCH5_PANEL_ROLES = [
    ("clean", "Clean"),
    ("damaged", "Damaged"),
    ("mask", "Mask"),
    ("opencv_restored", "OpenCV Telea"),
    ("lama_restored", "LaMa"),
]

batch5_panel_font = ImageFont.load_default()


def batch5_make_placeholder(tile_size: tuple[int, int], label: str) -> Image.Image:
    image = Image.new("RGB", tile_size, color=(245, 245, 245))
    draw = ImageDraw.Draw(image)
    draw.rectangle([0, 0, tile_size[0] - 1, tile_size[1] - 1], outline=(190, 190, 190), width=2)
    draw.text((12, tile_size[1] // 2 - 8), f"Missing: {label}", fill=(130, 45, 20), font=batch5_panel_font)
    return image


def batch5_load_panel_tile(path_value: str, label: str) -> tuple[Image.Image, bool, str]:
    path = batch5_resolve_path(path_value)
    if path is None or not path.is_file():
        return batch5_make_placeholder(BATCH5_PANEL_TILE_SIZE, label), False, "" if path is None else project_relative_path(path)
    image = Image.open(path).convert("RGB")
    image = ImageOps.contain(image, BATCH5_PANEL_TILE_SIZE, method=Image.Resampling.LANCZOS)
    canvas = Image.new("RGB", BATCH5_PANEL_TILE_SIZE, color=(255, 255, 255))
    offset = ((BATCH5_PANEL_TILE_SIZE[0] - image.width) // 2, (BATCH5_PANEL_TILE_SIZE[1] - image.height) // 2)
    canvas.paste(image, offset)
    return canvas, True, project_relative_path(path)


def batch5_create_case_panel(row: pd.Series) -> dict:
    case_id = batch5_text(row.get("case_id")) or f"case_{int(row.get('batch5_selected_order', 0)):03d}"
    safe_case_id = batch5_slug(case_id)
    output_path = BATCH5_PANELS_DIR / f"batch5_panel_{int(row.get('batch5_selected_order', 0)):02d}_{safe_case_id}.png"

    margin = 14
    header_height = 86
    label_height = 28
    tile_width, tile_height = BATCH5_PANEL_TILE_SIZE
    panel_width = margin + len(BATCH5_PANEL_ROLES) * (tile_width + margin)
    panel_height = header_height + label_height + tile_height + margin

    panel = Image.new("RGB", (panel_width, panel_height), color=(255, 255, 255))
    draw = ImageDraw.Draw(panel)

    title = f"{case_id} | winner={batch5_text(row.get('metric_majority_winner')) or 'unknown'} | margin={batch5_text(row.get('metric_vote_margin_lama_minus_opencv'))}"
    subtitle = batch5_text(row.get("selection_reason"))[:220]
    draw.text((margin, 12), title, fill=(17, 24, 39), font=batch5_panel_font)
    draw.text((margin, 36), subtitle, fill=(75, 85, 99), font=batch5_panel_font)
    draw.text((margin, 60), f"mask={batch5_text(row.get('summary_mask_type') or row.get('mask_type'))} | damage={batch5_text(row.get('summary_damage_family'))}", fill=(75, 85, 99), font=batch5_panel_font)

    role_records = []
    x = margin
    y_label = header_height
    y_tile = header_height + label_height
    for role, label in BATCH5_PANEL_ROLES:
        tile, exists, resolved_path = batch5_load_panel_tile(row.get(f"{role}_path_batch5", ""), label)
        draw.text((x, y_label + 6), label, fill=(17, 24, 39), font=batch5_panel_font)
        panel.paste(tile, (x, y_tile))
        role_records.append({"role": role, "exists": exists, "resolved_path": resolved_path})
        x += tile_width + margin

    panel.save(output_path)
    batch5_add_figure_record("selected_case_panel", output_path, f"Selected case panel: {case_id}", case_id, subtitle)

    return {
        "batch5_visual_case_key": row.get("batch5_visual_case_key", case_id),
        "case_id": case_id,
        "panel_path": project_relative_path(output_path),
        "panel_exists": output_path.is_file(),
        "available_panel_images": int(sum(record["exists"] for record in role_records)),
        "missing_panel_roles": ", ".join(record["role"] for record in role_records if not record["exists"]),
    }

batch5_panel_manifest_df = pd.DataFrame([batch5_create_case_panel(row) for _, row in batch5_selected_cases_df.iterrows()])
batch5_selected_cases_df = batch5_selected_cases_df.merge(batch5_panel_manifest_df, on="batch5_visual_case_key", how="left", suffixes=("", "_panel"))
batch5_selected_cases_df.to_csv(BATCH5_SELECTED_CASES_OUTPUT_PATH, index=False)

print("Generated selected-case panels:", int(batch5_panel_manifest_df["panel_exists"].sum()))
display(batch5_panel_manifest_df.head(30))


Generated selected-case panels: 24


,batch5_visual_case_key,case_id,panel_path,panel_exists,available_panel_images,missing_panel_roles
0,case_id=canonical__p005_loss_large|dataset_name=canonical|painting_id=p005|mask_id=p005_loss_large|mask_type=loss_large,canonical__p005_loss_large,outputs/20_opencv_lama_comparison_inventory_rebuild_v2/batch5_visual_report/figures/selected_case_panels/batch5_panel_01_canonical_p005_loss_large.png,True,5,
1,case_id=canonical__p007_loss_large|dataset_name=canonical|painting_id=p007|mask_id=p007_loss_large|mask_type=loss_large,canonical__p007_loss_large,outputs/20_opencv_lama_comparison_inventory_rebuild_v2/batch5_visual_report/figures/selected_case_panels/batch5_panel_02_canonical_p007_loss_large.png,True,5,
2,case_id=canonical__p010_loss_large|dataset_name=canonical|painting_id=p010|mask_id=p010_loss_large|mask_type=loss_large,canonical__p010_loss_large,outputs/20_opencv_lama_comparison_inventory_rebuild_v2/batch5_visual_report/figures/selected_case_panels/batch5_panel_03_canonical_p010_loss_large.png,True,5,
3,case_id=canonical__p012_loss_large|dataset_name=canonical|painting_id=p012|mask_id=p012_loss_large|mask_type=loss_large,canonical__p012_loss_large,outputs/20_opencv_lama_comparison_inventory_rebuild_v2/batch5_visual_report/figures/selected_case_panels/batch5_panel_04_canonical_p012_loss_large.png,True,5,
4,case_id=canonical__p020_mixed_damage|dataset_name=canonical|painting_id=p020|mask_id=p020_mixed_damage|mask_type=mixed_damage,canonical__p020_mixed_damage,outputs/20_opencv_lama_comparison_inventory_rebuild_v2/batch5_visual_report/figures/selected_case_panels/batch5_panel_05_canonical_p020_mixed_damage.png,True,5,
5,case_id=canonical__p037_scratch_thin|dataset_name=canonical|painting_id=p037|mask_id=p037_scratch_thin|mask_type=scratch_thin,canonical__p037_scratch_thin,outputs/20_opencv_lama_comparison_inventory_rebuild_v2/batch5_visual_report/figures/selected_case_panels/batch5_panel_06_canonical_p037_scratch_thin.png,True,5,
6,case_id=canonical__p041_loss_large|dataset_name=canonical|painting_id=p041|mask_id=p041_loss_large|mask_type=loss_large,canonical__p041_loss_large,outputs/20_opencv_lama_comparison_inventory_rebuild_v2/batch5_visual_report/figures/selected_case_panels/batch5_panel_07_canonical_p041_loss_large.png,True,5,
7,case_id=canonical__p050_loss_large|dataset_name=canonical|painting_id=p050|mask_id=p050_loss_large|mask_type=loss_large,canonical__p050_loss_large,outputs/20_opencv_lama_comparison_inventory_rebuild_v2/batch5_visual_report/figures/selected_case_panels/batch5_panel_08_canonical_p050_loss_large.png,True,5,
8,case_id=mask_robustness__p026__loss_large__variant_01|dataset_name=mask_robustness|painting_id=p026|mask_id=|mask_type=,mask_robustness__p026__loss_large__variant_01,outputs/20_opencv_lama_comparison_inventory_rebuild_v2/batch5_visual_report/figures/selected_case_panels/batch5_panel_09_mask_robustness_p026_loss_large_variant_01.png,True,5,
9,case_id=mask_robustness__p026__loss_large__variant_05|dataset_name=mask_robustness|painting_id=p026|mask_id=|mask_type=,mask_robustness__p026__loss_large__variant_05,outputs/20_opencv_lama_comparison_inventory_rebuild_v2/batch5_visual_report/figures/selected_case_panels/batch5_panel_10_mask_robustness_p026_loss_large_variant_05.png,True,5,


In [55]:
batch5_figure_manifest_df = pd.DataFrame(figure_manifest_records).drop_duplicates(subset=["relative_path"]).reset_index(drop=True)
batch5_figure_manifest_df.to_csv(BATCH5_FIGURE_MANIFEST_OUTPUT_PATH, index=False)

status_cards_df = pd.DataFrame(
    [
        {"item": "Input cases", "value": int(len(batch5_cases_df))},
        {"item": "Selected visual cases", "value": int(len(batch5_selected_cases_df))},
        {"item": "Cases needing visual review", "value": int(batch5_bool_series(batch5_cases_df, "needs_visual_review").sum())},
        {"item": "Generated figures", "value": int(len(batch5_figure_manifest_df))},
        {"item": "Generated selected-case panels", "value": int((batch5_figure_manifest_df["figure_type"] == "selected_case_panel").sum()) if not batch5_figure_manifest_df.empty else 0},
        {"item": "Image audit rows", "value": int(len(batch5_image_path_audit_df))},
    ]
)

finding_rows = []
if "metric_majority_winner" in batch5_cases_df.columns:
    for winner, count in batch5_cases_df["metric_majority_winner"].fillna("unknown").value_counts().items():
        finding_rows.append({"finding": f"Case majority winner count: {winner}", "value": int(count)})
if "selection_reason" in batch5_selected_cases_df.columns:
    finding_rows.append({"finding": "Selected cases are prioritized for disagreement, failures, metric extremes, and metadata coverage.", "value": int(len(batch5_selected_cases_df))})

finding_notes_df = pd.DataFrame(finding_rows)

def batch5_image_tag(path_value: str, alt: str) -> str:
    path = batch5_resolve_path(path_value)
    if path is None or not path.is_file():
        return f"<div class='missing'>Missing image: {html.escape(alt)}</div>"
    return f"<img src='{batch5_file_to_data_uri(path)}' alt='{html.escape(alt)}'>"

case_sections = []
for _, row in batch5_selected_cases_df.iterrows():
    panel_html = batch5_image_tag(row.get("panel_path", ""), f"Panel {row.get('case_id', '')}")
    case_sections.append(
        f"""
        <section class='case-card'>
          <h3>{html.escape(batch5_text(row.get('batch5_selected_order')))}. {html.escape(batch5_text(row.get('case_id')))}</h3>
          <p><strong>Selection reason:</strong> {html.escape(batch5_text(row.get('selection_reason')))}</p>
          <p><strong>Majority winner:</strong> {html.escape(batch5_text(row.get('metric_majority_winner')))} | <strong>Vote margin:</strong> {html.escape(batch5_text(row.get('metric_vote_margin_lama_minus_opencv')))} | <strong>Rank change:</strong> {html.escape(batch5_text(row.get('rank_change_median_lama_minus_opencv')))}</p>
          {panel_html}
        </section>
        """
    )

plot_sections = []
for _, figure_row in batch5_figure_manifest_df[batch5_figure_manifest_df["figure_type"].eq("summary_plot")].iterrows():
    plot_sections.append(
        f"<figure>{batch5_image_tag(figure_row['relative_path'], figure_row['title'])}<figcaption>{html.escape(figure_row['title'])}</figcaption></figure>"
    )

html_report = f"""<!doctype html>
<html lang='en'>
<head>
<meta charset='utf-8'>
<title>OpenCV Telea vs LaMa Batch 5 Comparison Report</title>
<style>
body {{ font-family: Arial, sans-serif; margin: 32px; color: #111827; background: #f8fafc; line-height: 1.5; }}
h1, h2, h3 {{ color: #111827; }}
.summary, .case-card, figure {{ background: #fff; border: 1px solid #d1d5db; border-radius: 8px; padding: 18px; margin: 18px 0; }}
.data-table {{ border-collapse: collapse; width: 100%; font-size: 12px; }}
.data-table th, .data-table td {{ border: 1px solid #d1d5db; padding: 6px 8px; text-align: left; vertical-align: top; }}
.data-table th {{ background: #e5e7eb; }}
img {{ max-width: 100%; height: auto; border: 1px solid #e5e7eb; background: #fff; }}
.missing {{ padding: 20px; background: #fff7ed; border: 1px solid #fed7aa; color: #9a3412; }}
.note {{ color: #4b5563; }}
</style>
</head>
<body>
<h1>OpenCV Telea vs LaMa Batch 5 Comparison Report</h1>
<p class='note'>Generated at {html.escape(datetime.now(timezone.utc).isoformat())}. This report is a qualitative review layer over the complete metric tables.</p>
<section class='summary'>
<h2>Status</h2>
{batch5_dataframe_to_html(status_cards_df)}
</section>
<section class='summary'>
<h2>Short Findings</h2>
{batch5_dataframe_to_html(finding_notes_df)}
</section>
<section class='summary'>
<h2>Compact Group Summary Preview</h2>
{batch5_dataframe_to_html(batch4_compact_summaries_df, max_rows=20)}
</section>
<section class='summary'>
<h2>Summary Plots</h2>
{''.join(plot_sections) if plot_sections else '<p>No summary plots were generated.</p>'}
</section>
<section class='summary'>
<h2>Selected Visual Cases</h2>
{''.join(case_sections)}
</section>
</body>
</html>
"""

BATCH5_HTML_REPORT_OUTPUT_PATH.write_text(html_report, encoding="utf-8")

print("Saved selected visual cases:", project_relative_path(BATCH5_SELECTED_CASES_OUTPUT_PATH))
print("Saved image path audit:", project_relative_path(BATCH5_IMAGE_PATH_AUDIT_OUTPUT_PATH))
print("Saved figure manifest:", project_relative_path(BATCH5_FIGURE_MANIFEST_OUTPUT_PATH))
print("Saved HTML report:", project_relative_path(BATCH5_HTML_REPORT_OUTPUT_PATH))


Saved selected visual cases: outputs/20_opencv_lama_comparison_inventory_rebuild_v2/batch5_visual_report/tables/opencv_lama_batch5_selected_visual_cases.csv
Saved image path audit: outputs/20_opencv_lama_comparison_inventory_rebuild_v2/batch5_visual_report/tables/opencv_lama_batch5_image_path_audit.csv
Saved figure manifest: outputs/20_opencv_lama_comparison_inventory_rebuild_v2/batch5_visual_report/tables/opencv_lama_batch5_figure_manifest.csv
Saved HTML report: outputs/20_opencv_lama_comparison_inventory_rebuild_v2/batch5_visual_report/reports/opencv_lama_batch5_comparison_report.html


In [56]:
batch5_output_paths = {
    "selected_cases": BATCH5_SELECTED_CASES_OUTPUT_PATH,
    "image_path_audit": BATCH5_IMAGE_PATH_AUDIT_OUTPUT_PATH,
    "figure_manifest": BATCH5_FIGURE_MANIFEST_OUTPUT_PATH,
    "html_report": BATCH5_HTML_REPORT_OUTPUT_PATH,
}

batch5_figure_paths_exist = (
    bool(batch5_figure_manifest_df["exists"].astype(bool).all())
    if not batch5_figure_manifest_df.empty and "exists" in batch5_figure_manifest_df.columns
    else False
)

batch5_selected_cases_unique = batch5_selected_cases_df["batch5_visual_case_key"].astype(str).duplicated().sum() == 0

batch5_check_rows = [
    build_check(
        "batch5_input_cases_have_rows",
        len(batch5_cases_df),
        "> 0",
        len(batch5_cases_df) > 0,
        "Batch 5 input case table is empty.",
    ),
    build_check(
        "batch5_selected_cases_have_rows",
        len(batch5_selected_cases_df),
        "> 0",
        len(batch5_selected_cases_df) > 0,
        "No visual cases were selected.",
    ),
    build_check(
        "batch5_selected_cases_unique",
        bool(batch5_selected_cases_unique),
        True,
        bool(batch5_selected_cases_unique),
        "Selected visual cases contain duplicate case_id values.",
    ),
    build_check(
        "batch5_image_path_audit_written",
        project_relative_path(BATCH5_IMAGE_PATH_AUDIT_OUTPUT_PATH),
        "existing non-empty CSV",
        BATCH5_IMAGE_PATH_AUDIT_OUTPUT_PATH.is_file() and BATCH5_IMAGE_PATH_AUDIT_OUTPUT_PATH.stat().st_size > 0,
        "Image path audit CSV missing or empty.",
    ),
    build_check(
        "batch5_panels_generated_for_selected_cases",
        int(batch5_panel_manifest_df["panel_exists"].sum()) if not batch5_panel_manifest_df.empty else 0,
        int(len(batch5_selected_cases_df)),
        not batch5_panel_manifest_df.empty and int(batch5_panel_manifest_df["panel_exists"].sum()) == int(len(batch5_selected_cases_df)),
        "Not every selected case received a generated panel.",
    ),
    build_check(
        "batch5_figure_manifest_written",
        project_relative_path(BATCH5_FIGURE_MANIFEST_OUTPUT_PATH),
        "existing non-empty CSV",
        BATCH5_FIGURE_MANIFEST_OUTPUT_PATH.is_file() and BATCH5_FIGURE_MANIFEST_OUTPUT_PATH.stat().st_size > 0,
        "Figure manifest CSV missing or empty.",
    ),
    build_check(
        "batch5_manifest_figures_exist",
        batch5_figure_paths_exist,
        True,
        batch5_figure_paths_exist,
        "One or more figure manifest paths does not exist.",
    ),
    build_check(
        "batch5_html_report_written",
        project_relative_path(BATCH5_HTML_REPORT_OUTPUT_PATH),
        "existing non-empty HTML",
        BATCH5_HTML_REPORT_OUTPUT_PATH.is_file() and BATCH5_HTML_REPORT_OUTPUT_PATH.stat().st_size > 0,
        "HTML report missing or empty.",
    ),
    build_check(
        "batch5_required_outputs_exist",
        {name: path.is_file() for name, path in batch5_output_paths.items()},
        "all True",
        all(path.is_file() for path in batch5_output_paths.values()),
        "One or more Batch 5 outputs is missing.",
    ),
]

batch5_validation_df = pd.DataFrame(batch5_check_rows)

batch5_validation_export_df = batch5_validation_df.copy()
for column in ["observed", "expected"]:
    batch5_validation_export_df[column] = batch5_validation_export_df[column].map(json_for_csv)
batch5_validation_export_df.to_csv(BATCH5_VALIDATION_OUTPUT_PATH, index=False)

display(batch5_validation_df)

if not batch5_validation_df["passed"].astype(bool).all():
    failed_batch5_checks_df = batch5_validation_df[~batch5_validation_df["passed"].astype(bool)].copy()
    display(failed_batch5_checks_df)
    raise RuntimeError("Notebook 20 Batch 5 validation failed. Inspect selected cases, panels, and report outputs.")


,check_name,observed,expected,passed,failure_message
0,batch5_input_cases_have_rows,410,> 0,True,
1,batch5_selected_cases_have_rows,24,> 0,True,
2,batch5_selected_cases_unique,True,True,True,
3,batch5_image_path_audit_written,outputs/20_opencv_lama_comparison_inventory_rebuild_v2/batch5_visual_report/tables/opencv_lama_batch5_image_path_audit.csv,existing non-empty CSV,True,
4,batch5_panels_generated_for_selected_cases,24,24,True,
5,batch5_figure_manifest_written,outputs/20_opencv_lama_comparison_inventory_rebuild_v2/batch5_visual_report/tables/opencv_lama_batch5_figure_manifest.csv,existing non-empty CSV,True,
6,batch5_manifest_figures_exist,True,True,True,
7,batch5_html_report_written,outputs/20_opencv_lama_comparison_inventory_rebuild_v2/batch5_visual_report/reports/opencv_lama_batch5_comparison_report.html,existing non-empty HTML,True,
8,batch5_required_outputs_exist,"{'selected_cases': True, 'image_path_audit': True, 'figure_manifest': True, 'html_report': True}",all True,True,


In [57]:
stage_manifest = read_json(STAGE_MANIFEST_JSON_PATH) if STAGE_MANIFEST_JSON_PATH.is_file() else {}

stage_manifest.update(
    {
        "status": "batch_5_visual_report_complete",
        "updated_at_utc": datetime.now(timezone.utc).isoformat(),
    }
)

stage_manifest.setdefault("outputs", {})
stage_manifest["outputs"].update(
    {
        "batch5_selected_visual_cases": project_relative_path(BATCH5_SELECTED_CASES_OUTPUT_PATH),
        "batch5_image_path_audit": project_relative_path(BATCH5_IMAGE_PATH_AUDIT_OUTPUT_PATH),
        "batch5_figure_manifest": project_relative_path(BATCH5_FIGURE_MANIFEST_OUTPUT_PATH),
        "batch5_html_report": project_relative_path(BATCH5_HTML_REPORT_OUTPUT_PATH),
        "batch5_validation": project_relative_path(BATCH5_VALIDATION_OUTPUT_PATH),
    }
)

stage_manifest.setdefault("row_counts", {})
stage_manifest["row_counts"].update(
    {
        "batch5_input_case_rows": int(len(batch5_cases_df)),
        "batch5_selected_visual_case_rows": int(len(batch5_selected_cases_df)),
        "batch5_image_path_audit_rows": int(len(batch5_image_path_audit_df)),
        "batch5_figure_manifest_rows": int(len(batch5_figure_manifest_df)),
        "batch5_validation_checks": int(len(batch5_validation_df)),
    }
)

stage_manifest["batch5_selection_policy"] = (
    "Select visual cases using visual-review flags, deterministic failure flags, metric extremes, "
    "and metadata coverage across mask/category/style/dataset/damage dimensions."
)
stage_manifest["next_batch"] = (
    "Batch 6: final artifact validation, manifest/handoff update, and downstream thesis-report handoff."
)

save_json(STAGE_MANIFEST_JSON_PATH, stage_manifest)

print("Batch 5 complete.")
print("Saved:", project_relative_path(BATCH5_SELECTED_CASES_OUTPUT_PATH))
print("Saved:", project_relative_path(BATCH5_IMAGE_PATH_AUDIT_OUTPUT_PATH))
print("Saved:", project_relative_path(BATCH5_FIGURE_MANIFEST_OUTPUT_PATH))
print("Saved:", project_relative_path(BATCH5_HTML_REPORT_OUTPUT_PATH))
print("Saved:", project_relative_path(BATCH5_VALIDATION_OUTPUT_PATH))
print("Updated manifest:", project_relative_path(STAGE_MANIFEST_JSON_PATH))


Batch 5 complete.
Saved: outputs/20_opencv_lama_comparison_inventory_rebuild_v2/batch5_visual_report/tables/opencv_lama_batch5_selected_visual_cases.csv
Saved: outputs/20_opencv_lama_comparison_inventory_rebuild_v2/batch5_visual_report/tables/opencv_lama_batch5_image_path_audit.csv
Saved: outputs/20_opencv_lama_comparison_inventory_rebuild_v2/batch5_visual_report/tables/opencv_lama_batch5_figure_manifest.csv
Saved: outputs/20_opencv_lama_comparison_inventory_rebuild_v2/batch5_visual_report/reports/opencv_lama_batch5_comparison_report.html
Saved: outputs/20_opencv_lama_comparison_inventory_rebuild_v2/batch5_visual_report/validation/opencv_lama_comparison_batch5_validation.csv
Updated manifest: outputs/20_opencv_lama_comparison_inventory_rebuild_v2/stage_manifest.json


In [58]:
import json
import re
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

BATCH6_BASE_OUTPUT_DIR = Path(
    globals().get(
        "OUTPUT_ROOT",
        globals().get(
            "NOTEBOOK_OUTPUT_DIR",
            PROJECT_ROOT / "outputs" / "20_opencv_lama_comparison",
        ),
    )
)

BATCH6_OUTPUT_DIR = BATCH6_BASE_OUTPUT_DIR / "batch6_final_handoff"
BATCH6_TABLES_DIR = BATCH6_OUTPUT_DIR / "tables"
BATCH6_VALIDATION_DIR = BATCH6_OUTPUT_DIR / "validation"
BATCH6_MANIFESTS_DIR = BATCH6_OUTPUT_DIR / "manifests"

for directory in [BATCH6_TABLES_DIR, BATCH6_VALIDATION_DIR, BATCH6_MANIFESTS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

BATCH6_ARTIFACT_INDEX_OUTPUT_PATH = BATCH6_TABLES_DIR / "opencv_lama_batch6_artifact_index.csv"
BATCH6_FINAL_VALIDATION_OUTPUT_PATH = BATCH6_VALIDATION_DIR / "opencv_lama_batch6_final_validation.csv"
BATCH6_HANDOFF_MANIFEST_OUTPUT_PATH = BATCH6_MANIFESTS_DIR / "opencv_lama_batch6_handoff_manifest.json"

if "STAGE_MANIFEST_JSON_PATH" not in globals():
    STAGE_MANIFEST_JSON_PATH = globals().get(
        "MANIFESTS_DIR",
        BATCH6_BASE_OUTPUT_DIR / "manifests",
    ) / "20_opencv_lama_comparison_manifest.json"

STAGE_MANIFEST_JSON_PATH = Path(STAGE_MANIFEST_JSON_PATH)

print("Batch 6 output dir:", project_relative_path(BATCH6_OUTPUT_DIR))
print("Stage manifest:", project_relative_path(STAGE_MANIFEST_JSON_PATH))

Batch 6 output dir: outputs/20_opencv_lama_comparison_inventory_rebuild_v2/batch6_final_handoff
Stage manifest: outputs/20_opencv_lama_comparison_inventory_rebuild_v2/stage_manifest.json


In [59]:
def batch6_text(value) -> str:
    if value is None:
        return ""
    if isinstance(value, float) and np.isnan(value):
        return ""
    text = str(value).strip()
    return "" if text.lower() in {"nan", "none", "null", "<na>"} else text


def batch6_slug(value) -> str:
    text = batch6_text(value).lower()
    text = re.sub(r"[^a-z0-9]+", "_", text)
    return text.strip("_") or "unknown"


def batch6_resolve_path(value) -> Path | None:
    text = batch6_text(value)
    if not text:
        return None
    if text.startswith("data:"):
        return None
    path = Path(text)
    if path.is_absolute():
        return path
    return PROJECT_ROOT / path


def batch6_infer_batch(label: str) -> str:
    match = re.search(r"batch[_-]?(\d+)", label.lower())
    return f"batch_{match.group(1)}" if match else "stage"


def batch6_infer_artifact_type(label: str, path: Path | None) -> str:
    text = label.lower()
    suffix = path.suffix.lower() if path is not None else ""

    if "validation" in text:
        return "validation"
    if "manifest" in text:
        return "manifest"
    if "report" in text or suffix in {".html", ".htm"}:
        return "report"
    if "figure" in text or suffix in {".png", ".jpg", ".jpeg", ".webp"}:
        return "figure"
    if suffix == ".csv":
        return "table"
    if suffix == ".json":
        return "manifest"
    return "artifact"


def batch6_file_sha256(path: Path | None) -> str:
    if path is None or not path.is_file():
        return ""
    try:
        return sha256_file(path)
    except Exception:
        return ""


def batch6_artifact_record(
    artifact_key: str,
    path_value,
    *,
    batch: str | None = None,
    artifact_type: str | None = None,
    required: bool = True,
    source: str = "",
    notes: str = "",
) -> dict:
    path = batch6_resolve_path(path_value)
    exists = bool(path is not None and path.is_file())
    size_bytes = int(path.stat().st_size) if exists else 0

    return {
        "artifact_key": artifact_key,
        "batch": batch or batch6_infer_batch(artifact_key),
        "artifact_type": artifact_type or batch6_infer_artifact_type(artifact_key, path),
        "path": project_relative_path(path) if path is not None else "",
        "required": bool(required),
        "exists": bool(exists),
        "size_bytes": size_bytes,
        "sha256": batch6_file_sha256(path),
        "source": source,
        "notes": notes,
    }


def batch6_load_stage_manifest() -> dict:
    if STAGE_MANIFEST_JSON_PATH.is_file():
        return read_json(STAGE_MANIFEST_JSON_PATH)
    return {}


stage_manifest = batch6_load_stage_manifest()
print("Stage manifest status:", stage_manifest.get("status", "missing"))

Stage manifest status: batch_5_visual_report_complete


In [65]:
def batch6_collect_artifact_records(include_batch6_outputs: bool = True) -> list[dict]:
    records = []

    stage_manifest_current = batch6_load_stage_manifest()

    if STAGE_MANIFEST_JSON_PATH:
        records.append(
            batch6_artifact_record(
                "stage_manifest",
                STAGE_MANIFEST_JSON_PATH,
                batch="stage",
                artifact_type="manifest",
                required=True,
                source="stage_manifest_path",
            )
        )

    for output_key, output_path in stage_manifest_current.get("outputs", {}).items():
        records.append(
            batch6_artifact_record(
                output_key,
                output_path,
                required=True,
                source="stage_manifest.outputs",
            )
        )

    for global_name, global_value in sorted(globals().items()):
        if not (
            global_name.startswith("BATCH")
            and global_name.endswith("OUTPUT_PATH")
            and isinstance(global_value, (str, Path))
        ):
            continue

        if global_name.startswith("BATCH6") and not include_batch6_outputs:
            continue

        records.append(
            batch6_artifact_record(
                global_name.lower(),
                global_value,
                required=True,
                source="notebook_global_output_path",
            )
        )

    figure_manifest_df = None
    if "batch5_figure_manifest_df" in globals() and isinstance(batch5_figure_manifest_df, pd.DataFrame):
        figure_manifest_df = batch5_figure_manifest_df.copy()
    elif "BATCH5_FIGURE_MANIFEST_OUTPUT_PATH" in globals() and Path(BATCH5_FIGURE_MANIFEST_OUTPUT_PATH).is_file():
        figure_manifest_df = pd.read_csv(BATCH5_FIGURE_MANIFEST_OUTPUT_PATH)

    if figure_manifest_df is not None and not figure_manifest_df.empty and "relative_path" in figure_manifest_df.columns:
        for _, row in figure_manifest_df.iterrows():
            figure_key = batch6_text(row.get("figure_id")) or batch6_slug(row.get("relative_path"))
            records.append(
                batch6_artifact_record(
                    f"batch5_figure__{figure_key}",
                    row.get("relative_path"),
                    batch="batch_5",
                    artifact_type="figure",
                    required=True,
                    source="batch5_figure_manifest",
                    notes=batch6_text(row.get("title")),
                )
            )

    deduped = {}
    for record in records:
        dedupe_key = record["path"] or record["artifact_key"]
        if dedupe_key not in deduped:
            deduped[dedupe_key] = record
        else:
            deduped[dedupe_key]["source"] = (
                deduped[dedupe_key]["source"] + "; " + record["source"]
            ).strip("; ")

    return list(deduped.values())


batch6_artifact_index_df = pd.DataFrame(batch6_collect_artifact_records(include_batch6_outputs=False))
batch6_artifact_index_df = batch6_artifact_index_df.sort_values(
    ["batch", "artifact_type", "artifact_key", "path"]
).reset_index(drop=True)

display(batch6_artifact_index_df.head(50))
print("Indexed artifacts:", len(batch6_artifact_index_df))

,artifact_key,batch,artifact_type,path,required,exists,size_bytes,sha256,source,notes
0,batch0_inventory_snapshot,batch_0,table,outputs/20_opencv_lama_comparison_inventory_rebuild_v2/batch0_inventory/batch0_inventory_snapshot.csv,True,True,1679042,,stage_manifest.outputs; notebook_global_output_path,
1,batch0_source_plan,batch_0,table,outputs/20_opencv_lama_comparison_inventory_rebuild_v2/batch0_inventory/batch0_inventory_source_plan.csv,True,True,4572,,stage_manifest.outputs; notebook_global_output_path,
2,batch0_validation,batch_0,validation,outputs/20_opencv_lama_comparison_inventory_rebuild_v2/batch0_inventory/batch0_validation.csv,True,True,730,,stage_manifest.outputs; notebook_global_output_path,
3,batch1_source_table_shapes,batch_1,table,outputs/20_opencv_lama_comparison_inventory_rebuild_v2/batch1_sources/batch1_source_table_shapes.csv,True,True,35856,,stage_manifest.outputs; notebook_global_output_path,
4,batch1_schema_validation,batch_1,validation,outputs/20_opencv_lama_comparison_inventory_rebuild_v2/batch1_sources/batch1_source_schema_validation.csv,True,True,3865,,stage_manifest.outputs; notebook_global_output_path,
5,batch2_metric_long,batch_2,table,outputs/20_opencv_lama_comparison_inventory_rebuild_v2/batch2_pairing/batch2_metric_long.csv,True,True,12880214,,stage_manifest.outputs; notebook_global_output_path,
6,batch2_metric_pairs,batch_2,table,outputs/20_opencv_lama_comparison_inventory_rebuild_v2/batch2_pairing/batch2_opencv_lama_metric_pairs.csv,True,True,10989787,,stage_manifest.outputs; notebook_global_output_path,
7,batch2_pairing_audit,batch_2,table,outputs/20_opencv_lama_comparison_inventory_rebuild_v2/batch2_pairing/batch2_pairing_audit.csv,True,True,26622,,stage_manifest.outputs; notebook_global_output_path,
8,batch2_restoration_pairs,batch_2,table,outputs/20_opencv_lama_comparison_inventory_rebuild_v2/batch2_pairing/batch2_opencv_lama_restoration_pairs.csv,True,True,278673,,stage_manifest.outputs; notebook_global_output_path,
9,batch2_validation,batch_2,validation,outputs/20_opencv_lama_comparison_inventory_rebuild_v2/batch2_pairing/batch2_validation.csv,True,True,671,,stage_manifest.outputs; notebook_global_output_path,


Indexed artifacts: 54


In [66]:
def batch6_df_has_rows(name: str) -> bool:
    value = globals().get(name)
    return isinstance(value, pd.DataFrame) and len(value) > 0


def batch6_path_exists_from_global(name: str) -> bool:
    if name not in globals():
        return False
    path = batch6_resolve_path(globals()[name])
    return bool(path is not None and path.is_file() and path.stat().st_size > 0)


required_artifacts_df = batch6_artifact_index_df[batch6_artifact_index_df["required"].astype(bool)].copy()
missing_required_df = required_artifacts_df[~required_artifacts_df["exists"].astype(bool)].copy()
empty_required_df = required_artifacts_df[
    required_artifacts_df["exists"].astype(bool)
    & (pd.to_numeric(required_artifacts_df["size_bytes"], errors="coerce").fillna(0) <= 0)
].copy()

batch6_stage_outputs = stage_manifest.get("outputs", {}) if isinstance(stage_manifest, dict) else {}

batch6_check_rows = [
    build_check(
        "batch6_stage_manifest_readable",
        project_relative_path(STAGE_MANIFEST_JSON_PATH),
        "existing readable JSON",
        STAGE_MANIFEST_JSON_PATH.is_file() and isinstance(stage_manifest, dict),
        "Stage manifest is missing or unreadable.",
    ),
    build_check(
        "batch6_stage_manifest_has_outputs",
        len(batch6_stage_outputs),
        "> 0",
        len(batch6_stage_outputs) > 0,
        "Stage manifest has no outputs section.",
    ),
    build_check(
        "batch6_artifact_index_has_rows",
        len(batch6_artifact_index_df),
        "> 0",
        len(batch6_artifact_index_df) > 0,
        "Artifact index has no rows.",
    ),
    build_check(
        "batch6_required_artifacts_exist",
        int(len(missing_required_df)),
        0,
        len(missing_required_df) == 0,
        "One or more required artifacts are missing.",
    ),
    build_check(
        "batch6_required_artifacts_non_empty",
        int(len(empty_required_df)),
        0,
        len(empty_required_df) == 0,
        "One or more required artifacts are empty.",
    ),
    build_check(
        "batch6_batch3_unified_cases_available",
        {
            "dataframe_rows": int(len(globals().get("batch3_unified_cases_df", []))) if batch6_df_has_rows("batch3_unified_cases_df") else 0,
            "csv_exists": batch6_path_exists_from_global("BATCH3_UNIFIED_CASES_OUTPUT_PATH"),
        },
        "dataframe rows > 0 or CSV exists",
        batch6_df_has_rows("batch3_unified_cases_df") or batch6_path_exists_from_global("BATCH3_UNIFIED_CASES_OUTPUT_PATH"),
        "Batch 3 unified cases are unavailable.",
    ),
    build_check(
        "batch6_batch4_compact_summaries_available",
        {
            "dataframe_rows": int(len(globals().get("batch4_compact_summaries_df", []))) if batch6_df_has_rows("batch4_compact_summaries_df") else 0,
            "csv_exists": batch6_path_exists_from_global("BATCH4_COMPACT_SUMMARIES_OUTPUT_PATH"),
        },
        "dataframe rows > 0 or CSV exists",
        batch6_df_has_rows("batch4_compact_summaries_df") or batch6_path_exists_from_global("BATCH4_COMPACT_SUMMARIES_OUTPUT_PATH"),
        "Batch 4 compact summaries are unavailable.",
    ),
    build_check(
        "batch6_batch5_selected_cases_available",
        {
            "dataframe_rows": int(len(globals().get("batch5_selected_cases_df", []))) if batch6_df_has_rows("batch5_selected_cases_df") else 0,
            "csv_exists": batch6_path_exists_from_global("BATCH5_SELECTED_CASES_OUTPUT_PATH"),
        },
        "dataframe rows > 0 or CSV exists",
        batch6_df_has_rows("batch5_selected_cases_df") or batch6_path_exists_from_global("BATCH5_SELECTED_CASES_OUTPUT_PATH"),
        "Batch 5 selected visual cases are unavailable.",
    ),
    build_check(
        "batch6_batch5_html_report_available",
        project_relative_path(BATCH5_HTML_REPORT_OUTPUT_PATH) if "BATCH5_HTML_REPORT_OUTPUT_PATH" in globals() else "",
        "existing non-empty HTML",
        batch6_path_exists_from_global("BATCH5_HTML_REPORT_OUTPUT_PATH"),
        "Batch 5 HTML comparison report is missing or empty.",
    ),
]

if "batch5_figure_manifest_df" in globals() and isinstance(batch5_figure_manifest_df, pd.DataFrame):
    batch6_figure_manifest_exists_check = bool(
        len(batch5_figure_manifest_df) > 0
        and "exists" in batch5_figure_manifest_df.columns
        and batch5_figure_manifest_df["exists"].astype(bool).all()
    )
else:
    batch6_figure_manifest_exists_check = batch6_path_exists_from_global("BATCH5_FIGURE_MANIFEST_OUTPUT_PATH")

batch6_check_rows.append(
    build_check(
        "batch6_batch5_figure_manifest_valid",
        batch6_figure_manifest_exists_check,
        True,
        batch6_figure_manifest_exists_check,
        "Batch 5 figure manifest is unavailable or contains missing figures.",
    )
)

batch6_validation_df = pd.DataFrame(batch6_check_rows)
display(batch6_validation_df)

,check_name,observed,expected,passed,failure_message
0,batch6_stage_manifest_readable,outputs/20_opencv_lama_comparison_inventory_rebuild_v2/stage_manifest.json,existing readable JSON,True,
1,batch6_stage_manifest_has_outputs,24,> 0,True,
2,batch6_artifact_index_has_rows,54,> 0,True,
3,batch6_required_artifacts_exist,0,0,True,
4,batch6_required_artifacts_non_empty,0,0,True,
5,batch6_batch3_unified_cases_available,"{'dataframe_rows': 410, 'csv_exists': True}",dataframe rows > 0 or CSV exists,True,
6,batch6_batch4_compact_summaries_available,"{'dataframe_rows': 67, 'csv_exists': True}",dataframe rows > 0 or CSV exists,True,
7,batch6_batch5_selected_cases_available,"{'dataframe_rows': 24, 'csv_exists': True}",dataframe rows > 0 or CSV exists,True,
8,batch6_batch5_html_report_available,outputs/20_opencv_lama_comparison_inventory_rebuild_v2/batch5_visual_report/reports/opencv_lama_batch5_comparison_report.html,existing non-empty HTML,True,
9,batch6_batch5_figure_manifest_valid,True,True,True,


In [67]:
batch6_validation_passed = bool(batch6_validation_df["passed"].astype(bool).all())

batch6_validation_export_df = batch6_validation_df.copy()
for column in ["observed", "expected"]:
    batch6_validation_export_df[column] = batch6_validation_export_df[column].map(json_for_csv)

batch6_validation_export_df.to_csv(BATCH6_FINAL_VALIDATION_OUTPUT_PATH, index=False)

stage_manifest = batch6_load_stage_manifest()
stage_manifest.setdefault("outputs", {})
stage_manifest.setdefault("row_counts", {})

stage_manifest["outputs"].update(
    {
        "batch6_artifact_index": project_relative_path(BATCH6_ARTIFACT_INDEX_OUTPUT_PATH),
        "batch6_final_validation": project_relative_path(BATCH6_FINAL_VALIDATION_OUTPUT_PATH),
        "batch6_handoff_manifest": project_relative_path(BATCH6_HANDOFF_MANIFEST_OUTPUT_PATH),
    }
)

stage_manifest["row_counts"].update(
    {
        "batch6_artifact_index_rows": int(len(batch6_artifact_index_df)),
        "batch6_validation_checks": int(len(batch6_validation_df)),
        "batch6_validation_failed_checks": int((~batch6_validation_df["passed"].astype(bool)).sum()),
    }
)

stage_manifest.update(
    {
        "status": (
            "notebook_20_opencv_lama_comparison_complete"
            if batch6_validation_passed
            else "batch_6_final_validation_failed"
        ),
        "updated_at_utc": datetime.now(timezone.utc).isoformat(),
        "final_validation_passed": batch6_validation_passed,
        "next_batch": "Notebook 20 complete. Use the handoff manifest for thesis reporting and downstream synthesis.",
    }
)

save_json(STAGE_MANIFEST_JSON_PATH, stage_manifest)

batch6_handoff_manifest = {
    "notebook_id": globals().get("NOTEBOOK_ID", "20_opencv_lama_comparison"),
    "notebook_name": globals().get("NOTEBOOK_NAME", "20_opencv_lama_comparison.ipynb"),
    "status": stage_manifest["status"],
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "project_root": str(PROJECT_ROOT),
    "stage_manifest": project_relative_path(STAGE_MANIFEST_JSON_PATH),
    "artifact_index": project_relative_path(BATCH6_ARTIFACT_INDEX_OUTPUT_PATH),
    "final_validation": project_relative_path(BATCH6_FINAL_VALIDATION_OUTPUT_PATH),
    "primary_outputs": {
        "unified_cases": stage_manifest["outputs"].get("batch3_unified_cases", ""),
        "compact_summaries": stage_manifest["outputs"].get("batch4_compact_summaries", ""),
        "runtime_summary": stage_manifest["outputs"].get("batch4_runtime_summary", ""),
        "failure_patterns": stage_manifest["outputs"].get("batch4_failure_patterns", ""),
        "selected_visual_cases": stage_manifest["outputs"].get("batch5_selected_visual_cases", ""),
        "figure_manifest": stage_manifest["outputs"].get("batch5_figure_manifest", ""),
        "html_report": stage_manifest["outputs"].get("batch5_html_report", ""),
    },
    "row_counts": stage_manifest.get("row_counts", {}),
    "validation_summary": {
        "passed": batch6_validation_passed,
        "checks": int(len(batch6_validation_df)),
        "failed_checks": batch6_validation_df.loc[
            ~batch6_validation_df["passed"].astype(bool),
            "check_name",
        ].astype(str).tolist(),
    },
    "handoff_notes": [
        "Use the unified cases CSV for case-level OpenCV Telea vs LaMa comparisons.",
        "Use compact summaries for thesis-level group statements by mask, dataset, category, style/period, and damage family.",
        "Use deterministic failure patterns only as implementation/audit evidence, not as subjective quality scores.",
        "Use the HTML report and selected panels as qualitative examples, not as a replacement for complete metric tables.",
    ],
}

save_json(BATCH6_HANDOFF_MANIFEST_OUTPUT_PATH, batch6_handoff_manifest)

batch6_artifact_index_df = pd.DataFrame(batch6_collect_artifact_records(include_batch6_outputs=True))
batch6_artifact_index_df = batch6_artifact_index_df.sort_values(
    ["batch", "artifact_type", "artifact_key", "path"]
).reset_index(drop=True)
batch6_artifact_index_df.to_csv(BATCH6_ARTIFACT_INDEX_OUTPUT_PATH, index=False)

print("Batch 6 final validation passed:", batch6_validation_passed)
print("Saved:", project_relative_path(BATCH6_ARTIFACT_INDEX_OUTPUT_PATH))
print("Saved:", project_relative_path(BATCH6_FINAL_VALIDATION_OUTPUT_PATH))
print("Saved:", project_relative_path(BATCH6_HANDOFF_MANIFEST_OUTPUT_PATH))
print("Updated:", project_relative_path(STAGE_MANIFEST_JSON_PATH))

Batch 6 final validation passed: True
Saved: outputs/20_opencv_lama_comparison_inventory_rebuild_v2/batch6_final_handoff/tables/opencv_lama_batch6_artifact_index.csv
Saved: outputs/20_opencv_lama_comparison_inventory_rebuild_v2/batch6_final_handoff/validation/opencv_lama_batch6_final_validation.csv
Saved: outputs/20_opencv_lama_comparison_inventory_rebuild_v2/batch6_final_handoff/manifests/opencv_lama_batch6_handoff_manifest.json
Updated: outputs/20_opencv_lama_comparison_inventory_rebuild_v2/stage_manifest.json


In [68]:
batch6_post_output_paths = {
    "artifact_index": BATCH6_ARTIFACT_INDEX_OUTPUT_PATH,
    "final_validation": BATCH6_FINAL_VALIDATION_OUTPUT_PATH,
    "handoff_manifest": BATCH6_HANDOFF_MANIFEST_OUTPUT_PATH,
    "stage_manifest": STAGE_MANIFEST_JSON_PATH,
}

batch6_post_checks_df = pd.DataFrame(
    [
        build_check(
            f"batch6_{name}_written",
            project_relative_path(path),
            "existing non-empty file",
            path.is_file() and path.stat().st_size > 0,
            f"Batch 6 {name} was not written.",
        )
        for name, path in batch6_post_output_paths.items()
    ]
)

display(batch6_post_checks_df)

batch6_all_checks_passed = (
    bool(batch6_validation_df["passed"].astype(bool).all())
    and bool(batch6_post_checks_df["passed"].astype(bool).all())
)

if not batch6_all_checks_passed:
    failed_final_checks_df = pd.concat(
        [
            batch6_validation_df[~batch6_validation_df["passed"].astype(bool)],
            batch6_post_checks_df[~batch6_post_checks_df["passed"].astype(bool)],
        ],
        ignore_index=True,
    )
    display(failed_final_checks_df)
    raise RuntimeError("Notebook 20 Batch 6 final validation failed. Inspect missing or empty artifacts.")

print("Batch 6 complete. Notebook 20 handoff is ready.")

,check_name,observed,expected,passed,failure_message
0,batch6_artifact_index_written,outputs/20_opencv_lama_comparison_inventory_rebuild_v2/batch6_final_handoff/tables/opencv_lama_batch6_artifact_index.csv,existing non-empty file,True,
1,batch6_final_validation_written,outputs/20_opencv_lama_comparison_inventory_rebuild_v2/batch6_final_handoff/validation/opencv_lama_batch6_final_validation.csv,existing non-empty file,True,
2,batch6_handoff_manifest_written,outputs/20_opencv_lama_comparison_inventory_rebuild_v2/batch6_final_handoff/manifests/opencv_lama_batch6_handoff_manifest.json,existing non-empty file,True,
3,batch6_stage_manifest_written,outputs/20_opencv_lama_comparison_inventory_rebuild_v2/stage_manifest.json,existing non-empty file,True,


Batch 6 complete. Notebook 20 handoff is ready.
